# ================================================================
# WEEK 5 — CANDIDATE PROFILE PROCESSING AND BASELINE RECOMMENDATION
# ================================================================

"""
Main focus:
- Candidate profile preparation
- Candidate data-quality assessment
- Privacy and anonymization
- Candidate feature extraction
- Candidate skill normalization
- Candidate-to-CIP education mapping
- Transparent baseline recommendation model
- Initial skill-gap analysis
- Labelled evaluation profiles

Analytics type:
- Diagnostic analytics
- Initial predictive / recommendation analytics

Candidate source:
- Public sample CVs / resumes
- 65 DOCX files

Recommendation baseline:
- Transparent occupation-specific weighted skill overlap
"""

In [332]:
# ================================================================
# CELL 2 — IMPORT LIBRARIES
# ================================================================

import pandas as pd
import numpy as np

import os
import glob
import re
from pathlib import Path

from docx import Document

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [333]:
# ================================================================
# CELL 3 — DEFINE PROJECT PATHS
# ================================================================

PROJECT_PATH = r"C:\Users\Admin\Capstone_Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "Data"
)

CV_PATH = os.path.join(
    DATA_PATH,
    "CV"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "Outputs"
)

print("=" * 70)
print("WEEK 5 — PROJECT PATHS")
print("=" * 70)

print("\nProject path:")
print(PROJECT_PATH)

print("\nCV path:")
print(CV_PATH)

print("\nOutput path:")
print(OUTPUT_PATH)

print("\nCV folder exists:", os.path.exists(CV_PATH))
print("Output folder exists:", os.path.exists(OUTPUT_PATH))

WEEK 5 — PROJECT PATHS

Project path:
C:\Users\Admin\Capstone_Project

CV path:
C:\Users\Admin\Capstone_Project\Data\CV

Output path:
C:\Users\Admin\Capstone_Project\Outputs

CV folder exists: True
Output folder exists: True


In [334]:
# ================================================================
# CELL 4 — LOAD WEEK 4 OUTPUTS
# ================================================================

onet_skills = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_skills_profile.csv"
    )
)

onet_cip_profile = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "onet_cip_integrated_occupation_profile.csv"
    )
)

validated_mapping = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "validated_occupation_onet_mapping.csv"
    )
)

week4_summary = pd.read_csv(
    os.path.join(
        TABLES_PATH,
        "week4_final_summary.csv"
    )
)

print("=" * 70)
print("WEEK 5 — WEEK 4 OUTPUTS LOADED")
print("=" * 70)

print("\nO*NET skills profile:")
print(onet_skills.shape)

print("\nO*NET + CIP integrated profile:")
print(onet_cip_profile.shape)

print("\nValidated occupation mapping:")
print(validated_mapping.shape)

print("\nWeek 4 summary:")
print(week4_summary.shape)

WEEK 5 — WEEK 4 OUTPUTS LOADED

O*NET skills profile:
(200, 7)

O*NET + CIP integrated profile:
(20, 12)

Validated occupation mapping:
(20, 16)

Week 4 summary:
(10, 2)


In [335]:
# ================================================================
# CELL 5 — INSPECT WEEK 4 DATA STRUCTURES
# ================================================================

print("=" * 70)
print("O*NET SKILLS PROFILE — COLUMNS")
print("=" * 70)

print(onet_skills.columns.tolist())

print("\nFirst 5 rows:")
display(onet_skills.head())


print("\n" + "=" * 70)
print("O*NET + CIP PROFILE — COLUMNS")
print("=" * 70)

print(onet_cip_profile.columns.tolist())

print("\nFirst 5 rows:")
display(onet_cip_profile.head())


print("\n" + "=" * 70)
print("VALIDATED MAPPING — COLUMNS")
print("=" * 70)

print(validated_mapping.columns.tolist())

print("\nFirst 5 rows:")
display(validated_mapping.head())

O*NET SKILLS PROFILE — COLUMNS
['onet_soc_code', 'element_id', 'skill', 'importance', 'level', 'selected_occupation', 'onet_title']

First 5 rows:


,onet_soc_code,element_id,skill,importance,level,selected_occupation,onet_title
0,11-9051.00,2.A.1.a,Reading Comprehension,3.75,3.88,Restaurant Manager,Food Service Managers
1,11-9051.00,2.A.1.b,Active Listening,3.88,3.50,Restaurant Manager,Food Service Managers
2,11-9051.00,2.A.1.c,Writing,3.00,3.12,Restaurant Manager,Food Service Managers
3,11-9051.00,2.A.1.d,Speaking,3.88,4.00,Restaurant Manager,Food Service Managers
4,11-9051.00,2.A.1.e,Mathematics,2.88,2.88,Restaurant Manager,Food Service Managers



O*NET + CIP PROFILE — COLUMNS
['selected_occupation', 'onet_soc_code', 'onet_title', 'skill_count', 'knowledge_count', 'ability_count', 'task_count', 'education_record_count', 'training_record_count', 'job_zone', 'cip_program_count', 'cip_soc_mapping_count']

First 5 rows:


,selected_occupation,onet_soc_code,onet_title,skill_count,knowledge_count,ability_count,task_count,education_record_count,training_record_count,job_zone,cip_program_count,cip_soc_mapping_count
0,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",10,33,52,31,12,29,2,2,2
1,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",10,33,52,28,12,29,3,1,1
2,Continuing Care Assistant,31-1131.00,Nursing Assistants,10,33,52,33,12,29,3,3,3
3,Continuing Care Assistant,31-1132.00,Orderlies,10,33,52,22,12,29,2,1,1
4,Delivery Driver,53-3033.00,Light Truck Drivers,10,33,52,13,12,29,2,1,1



VALIDATED MAPPING — COLUMNS
['selected_occupation', 'noc_2021_code', 'noc21_code', 'noc2016_code', 'mapping_note', 'noc2016_match_code', 'NOC 2016 Version 1.3 Code', 'NOC 2016 Version 1.3 Title', 'Partial', 'SOC 2018 (US) Code', 'SOC 2018 (US) Title', 'Explanatory Notes', 'O*NET-SOC 2019 Code', 'O*NET-SOC 2019 Title', '2018 SOC Code', '2018 SOC Title']

First 5 rows:


,selected_occupation,noc_2021_code,noc21_code,noc2016_code,mapping_note,noc2016_match_code,NOC 2016 Version 1.3 Code,NOC 2016 Version 1.3 Title,Partial,SOC 2018 (US) Code,SOC 2018 (US) Title,Explanatory Notes,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,Software Developer,21232,21232,2174,Breakdown mapping,2174,2174,Computer programmers and interactive media developers,*,15-1252,Software Developers,Only software developers in interactive media,15-1252.00,Software Developers,15-1252,Software Developers
1,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1211,Computer Systems Analysts,"Only computer systems analysts, excluding analysts and testers in quality assurance and systems ...",15-1211.00,Computer Systems Analysts,15-1211,Computer Systems Analysts
2,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1211,Computer Systems Analysts,"Only computer systems analysts, excluding analysts and testers in quality assurance and systems ...",15-1211.01,Health Informatics Specialists,15-1211,Computer Systems Analysts
3,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1212,Information Security Analysts,"Only analysts, consultants and related specialists in information security",15-1212.00,Information Security Analysts,15-1212,Information Security Analysts
4,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1253,Software Quality Assurance Analysts and Testers,"Only analysts and testers in quality assurance, including systems auditors",15-1253.00,Software Quality Assurance Analysts and Testers,15-1253,Software Quality Assurance Analysts and Testers


In [336]:
# ================================================================
# CELL 6 — CV SOURCE DATASET AUDIT
# ================================================================

cv_files = sorted(
    glob.glob(
        os.path.join(CV_PATH, "*.docx")
    )
)

cv_file_audit = pd.DataFrame({
    "filename": [
        os.path.basename(file)
        for file in cv_files
    ],
    
    "file_extension": [
        Path(file).suffix.lower()
        for file in cv_files
    ]
})

print("=" * 70)
print("WEEK 5 — CV SOURCE DATASET AUDIT")
print("=" * 70)

print("\nCV folder:")
print(CV_PATH)

print("\nTotal CV files:")
print(len(cv_files))

print("\nFile types:")
display(
    cv_file_audit["file_extension"]
    .value_counts()
    .rename_axis("file_type")
    .reset_index(name="file_count")
)

print("\nFirst 20 filenames:")
for i, filename in enumerate(
    cv_file_audit["filename"].head(20),
    start=1
):
    print(f"{i:02d}. {filename}")

WEEK 5 — CV SOURCE DATASET AUDIT

CV folder:
C:\Users\Admin\Capstone_Project\Data\CV

Total CV files:
65

File types:


,file_type,file_count
0,.docx,65



First 20 filenames:
01. 1.docx
02. 10.docx
03. 11.docx
04. 12.docx
05. 13.docx
06. 14.docx
07. 15.docx
08. 16.docx
09. 17.docx
10. 18.docx
11. 19.docx
12. 2.docx
13. 20.docx
14. 21.docx
15. 22.docx
16. 23.docx
17. 24.docx
18. 25.docx
19. 26.docx
20. 27.docx


In [337]:
# ================================================================
# CELL 7 — DOCX TEXT EXTRACTION FUNCTION
# ================================================================

def extract_docx_text(file_path):
    
    try:
        document = Document(file_path)
        
        paragraphs = [
            paragraph.text.strip()
            for paragraph in document.paragraphs
            if paragraph.text.strip()
        ]
        
        text = "\n".join(paragraphs)
        
        return text
    
    except Exception as error:
        
        print(
            f"Error reading "
            f"{os.path.basename(file_path)}: {error}"
        )
        
        return ""

In [338]:
# ================================================================
# CELL 8 — EXTRACT TEXT FROM ALL CVS
# ================================================================

cv_records = []

for cv_id, file_path in enumerate(
    cv_files,
    start=1
):
    
    text = extract_docx_text(file_path)
    
    cv_records.append({
        "cv_id": cv_id,
        "filename": os.path.basename(file_path),
        "raw_text": text,
        "character_count": len(text),
        "word_count": len(text.split()),
        "paragraph_count": len(
            [
                paragraph
                for paragraph in text.split("\n")
                if paragraph.strip()
            ]
        )
    })


cv_data = pd.DataFrame(cv_records)

print("=" * 70)
print("WEEK 5 — CV TEXT EXTRACTION")
print("=" * 70)

print("\nShape:")
print(cv_data.shape)

print("\nColumns:")
print(cv_data.columns.tolist())

print("\nFirst 10 CVs:")
display(
    cv_data[
        [
            "cv_id",
            "filename",
            "word_count",
            "paragraph_count"
        ]
    ].head(10)
)

WEEK 5 — CV TEXT EXTRACTION

Shape:
(65, 6)

Columns:
['cv_id', 'filename', 'raw_text', 'character_count', 'word_count', 'paragraph_count']

First 10 CVs:


,cv_id,filename,word_count,paragraph_count
0,1,1.docx,315,37
1,2,10.docx,266,31
2,3,11.docx,231,24
3,4,12.docx,0,0
4,5,13.docx,242,37
5,6,14.docx,498,55
6,7,15.docx,272,38
7,8,16.docx,179,40
8,9,17.docx,296,37
9,10,18.docx,282,43


In [339]:
# ================================================================
# CELL 9 — CV CONTENT QUALITY ASSESSMENT
# ================================================================

empty_cvs = cv_data[
    cv_data["word_count"] == 0
]

very_short_cvs = cv_data[
    (cv_data["word_count"] > 0)
    &
    (cv_data["word_count"] < 100)
]

usable_cvs = cv_data[
    cv_data["word_count"] >= 100
].copy()

print("=" * 70)
print("WEEK 5 — CV CONTENT AUDIT")
print("=" * 70)

print("\nTotal CVs:", len(cv_data))

print(
    "CVs with text:",
    (cv_data["word_count"] > 0).sum()
)

print(
    "Empty CVs:",
    len(empty_cvs)
)

print(
    "Very short CVs (<100 words):",
    len(very_short_cvs)
)

print(
    "Usable CVs (>=100 words):",
    len(usable_cvs)
)


print("\n" + "=" * 70)
print("CV WORD COUNT SUMMARY")
print("=" * 70)

print(
    cv_data["word_count"].describe()
)


print("\n" + "=" * 70)
print("CVs EXCLUDED OR REQUIRING REVIEW")
print("=" * 70)

display(
    cv_data[
        cv_data["word_count"] < 100
    ][
        [
            "cv_id",
            "filename",
            "word_count",
            "character_count"
        ]
    ]
    .sort_values("word_count")
)

WEEK 5 — CV CONTENT AUDIT

Total CVs: 65
CVs with text: 64
Empty CVs: 1
Very short CVs (<100 words): 1
Usable CVs (>=100 words): 63

CV WORD COUNT SUMMARY
count     65.000000
mean     307.015385
std      130.117074
min        0.000000
25%      221.000000
50%      297.000000
75%      362.000000
max      824.000000
Name: word_count, dtype: float64

CVs EXCLUDED OR REQUIRING REVIEW


,cv_id,filename,word_count,character_count
3,4,12.docx,0,0
54,55,59.docx,73,480


In [340]:
# ================================================================
# CELL 10 — CREATE ANONYMIZED CANDIDATE IDS
# ================================================================

candidate_data = usable_cvs.copy()

candidate_data = (
    candidate_data
    .sort_values("cv_id")
    .reset_index(drop=True)
)

candidate_data["candidate_id"] = [
    f"Candidate_{i:03d}"
    for i in range(
        1,
        len(candidate_data) + 1
    )
]

candidate_data = candidate_data[
    [
        "candidate_id",
        "cv_id",
        "filename",
        "raw_text",
        "character_count",
        "word_count",
        "paragraph_count"
    ]
]

print("=" * 70)
print("WEEK 5 — ANONYMIZED CANDIDATE DATASET")
print("=" * 70)

print("\nUsable candidates:")
print(len(candidate_data))

print("\nAnonymous candidate IDs:")
print(candidate_data["candidate_id"].nunique())

display(
    candidate_data[
        [
            "candidate_id",
            "cv_id",
            "word_count",
            "paragraph_count"
        ]
    ]
    .head(10)
)

WEEK 5 — ANONYMIZED CANDIDATE DATASET

Usable candidates:
63

Anonymous candidate IDs:
63


,candidate_id,cv_id,word_count,paragraph_count
0,Candidate_001,1,315,37
1,Candidate_002,2,266,31
2,Candidate_003,3,231,24
3,Candidate_004,5,242,37
4,Candidate_005,6,498,55
5,Candidate_006,7,272,38
6,Candidate_007,8,179,40
7,Candidate_008,9,296,37
8,Candidate_009,10,282,43
9,Candidate_010,11,320,43


In [341]:
# ================================================================
# CELL 11 — PRIVACY AND IDENTIFIER MASKING
# ================================================================

def anonymize_text(text):
    
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # ------------------------------------------------------------
    # EMAIL ADDRESSES
    # ------------------------------------------------------------
    
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "[EMAIL_REMOVED]",
        text
    )
    
    
    # ------------------------------------------------------------
    # PHONE NUMBERS
    # ------------------------------------------------------------
    
    text = re.sub(
        r"(\+?\d{1,3}[\s\-.]?)?"
        r"(\(?\d{2,4}\)?[\s\-.]?)?"
        r"\d{3,4}[\s\-.]?\d{3,4}",
        "[PHONE_REMOVED]",
        text
    )
    
    
    # ------------------------------------------------------------
    # URLS
    # ------------------------------------------------------------
    
    text = re.sub(
        r"https?://\S+|www\.\S+",
        "[URL_REMOVED]",
        text
    )
    
    
    # ------------------------------------------------------------
    # LINKEDIN / GITHUB USER URLS
    # ------------------------------------------------------------
    
    text = re.sub(
        r"(linkedin\.com/\S+|github\.com/\S+)",
        "[PROFILE_LINK_REMOVED]",
        text,
        flags=re.IGNORECASE
    )
    
    
    return text

In [342]:
# ================================================================
# CELL 12 — APPLY ANONYMIZATION
# ================================================================

candidate_data["clean_text"] = (
    candidate_data["raw_text"]
    .apply(anonymize_text)
)

candidate_data["clean_character_count"] = (
    candidate_data["clean_text"]
    .str.len()
)

print("=" * 70)
print("WEEK 5 — PRIVACY PROCESSING")
print("=" * 70)

print("\nCandidates processed:")
print(len(candidate_data))

print("\nCandidate ID count:")
print(candidate_data["candidate_id"].nunique())

print("\nExample anonymized CV text:")

example_text = (
    candidate_data
    .loc[0, "clean_text"]
)

print(example_text[:1000])

WEEK 5 — PRIVACY PROCESSING

Candidates processed:
63

Candidate ID count:
63

Example anonymized CV text:
****************
Java full stack developer
(60% – IN JAVA BACKEND DEVELOPMENT AND 40% – IN WEB FRONTEND)
3+ years of experience FullStack development and 5+ total experience in software development
phone**********	 		linkedIn
location Bat Yam     		email
Professional Skills
Programming languages: Java, JavaScript.
Web:  HTML5/CSS3, JQuery, Bootstrap3
Environments – IDE/tools:  Eclipse, IntelliJ IDEA, GitHub.
Databases: SQL (MySQL), NoSQL (MongoDB).
Technologies & Frameworks: OOP, AOP, Spring MVC, JPA, Hibernate, JDBC, SpringBoot, Spring Security, Spring Web, REST, JSON, Maven, Junit, Git, Postman.
Systems: Linux, Windows.
Professional Experience
2020 -  now:   Backend JAVA developer,   (Israel, Rehovot)
Latest project: Participation in the development of a microservice architecture for a system for storing educational information.
Areas of responsibility:
Developed backend website

In [343]:
# ================================================================
# CELL 13 — CREATE FINAL ANALYTICAL CANDIDATE DATASET
# ================================================================

candidate_profiles = candidate_data[
    [
        "candidate_id",
        "cv_id",
        "word_count",
        "paragraph_count",
        "clean_text"
    ]
].copy()

print("=" * 70)
print("WEEK 5 — CANDIDATE ANALYTICAL DATASET")
print("=" * 70)

print("\nShape:")
print(candidate_profiles.shape)

print("\nColumns:")
print(candidate_profiles.columns.tolist())

display(
    candidate_profiles.head()
)

WEEK 5 — CANDIDATE ANALYTICAL DATASET

Shape:
(63, 5)

Columns:
['candidate_id', 'cv_id', 'word_count', 'paragraph_count', 'clean_text']


,candidate_id,cv_id,word_count,paragraph_count,clean_text
0,Candidate_001,1,315,37,****************\nJava full stack developer\n(60% – IN JAVA BACKEND DEVELOPMENT AND 40% – IN WEB...
1,Candidate_002,2,266,31,******************\nWEB FRONTEND DEVELOPER\n5 years of experience in frontend development\nEXPER...
2,Candidate_003,3,231,24,"********************\nExperience in C, C++ and C# programming\nWorking knowledge of Agile, Jira ..."
3,Candidate_004,5,242,37,*****************\nFront-end Developer\nMobile phone: **********\nLinkedln:\t ****...
4,Candidate_005,6,498,55,"Rehovot, Israel \nAbout me\nSenior solution architect with wide experience of creating and main..."


In [344]:
# ================================================================
# CELL 14 — DEFINE CANDIDATE FEATURE EXTRACTION DICTIONARIES
# ================================================================

import re

# ------------------------------------------------
# TECHNICAL / PROGRAMMING SKILLS
# ------------------------------------------------

technical_skill_keywords = [
    "java",
    "javascript",
    "typescript",
    "python",
    "c++",
    "c#",
    "sql",
    "html",
    "css",
    "php",
    "kotlin",
    "swift",
    "ruby",
    "go",
    "scala",
    "r",
    "matlab"
]


# ------------------------------------------------
# FRAMEWORKS / TECHNOLOGIES
# ------------------------------------------------

technology_keywords = [
    "spring",
    "spring boot",
    "spring mvc",
    "hibernate",
    "django",
    "flask",
    "react",
    "angular",
    "vue",
    "node.js",
    "nodejs",
    ".net",
    "asp.net",
    "bootstrap",
    "jquery",
    "rest",
    "restful",
    "api",
    "microservices",
    "docker",
    "kubernetes",
    "aws",
    "azure",
    "google cloud"
]


# ------------------------------------------------
# DATABASES
# ------------------------------------------------

database_keywords = [
    "mysql",
    "postgresql",
    "mongodb",
    "oracle",
    "sql server",
    "sqlite",
    "redis",
    "nosql"
]


# ------------------------------------------------
# SOFTWARE / DEVELOPMENT TOOLS
# ------------------------------------------------

tool_keywords = [
    "git",
    "github",
    "gitlab",
    "jira",
    "confluence",
    "postman",
    "intellij",
    "eclipse",
    "visual studio",
    "vs code",
    "android studio",
    "maven",
    "gradle",
    "jenkins",
    "terraform"
]


# ------------------------------------------------
# PROFESSIONAL / TRANSFERABLE SKILLS
# ------------------------------------------------

professional_skill_keywords = [
    "communication",
    "teamwork",
    "leadership",
    "problem solving",
    "critical thinking",
    "analytical",
    "management",
    "collaboration",
    "planning",
    "organization",
    "agile",
    "scrum",
    "customer service",
    "presentation",
    "mentoring"
]


# ------------------------------------------------
# CERTIFICATIONS
# ------------------------------------------------

certification_keywords = [
    "certified",
    "certification",
    "certificate",
    "aws certified",
    "azure certified",
    "oracle certified",
    "scrum master",
    "pmp"
]


# ------------------------------------------------
# EDUCATION KEYWORDS
# ------------------------------------------------

degree_keywords = {
    "PhD": [
        "phd",
        "ph.d",
        "doctor of philosophy"
    ],
    
    "Master": [
        "master",
        "msc",
        "m.sc",
        "mba",
        "m.eng"
    ],
    
    "Bachelor": [
        "bachelor",
        "b.sc",
        "bsc",
        "b.eng",
        "undergraduate degree"
    ],
    
    "Diploma": [
        "diploma"
    ],
    
    "Certificate": [
        "certificate"
    ]
}


# ------------------------------------------------
# FIELD OF STUDY KEYWORDS
# ------------------------------------------------

field_of_study_keywords = [
    "computer science",
    "software engineering",
    "information technology",
    "computer engineering",
    "information systems",
    "data science",
    "mathematics",
    "business administration",
    "accounting",
    "finance",
    "engineering",
    "education"
]


print("=" * 70)
print("WEEK 5 — CANDIDATE FEATURE EXTRACTION DICTIONARIES")
print("=" * 70)

print("\nTechnical skills:", len(technical_skill_keywords))
print("Technologies:", len(technology_keywords))
print("Databases:", len(database_keywords))
print("Tools:", len(tool_keywords))
print("Professional skills:", len(professional_skill_keywords))
print("Fields of study:", len(field_of_study_keywords))

WEEK 5 — CANDIDATE FEATURE EXTRACTION DICTIONARIES

Technical skills: 17
Technologies: 24
Databases: 8
Tools: 15
Professional skills: 15
Fields of study: 12


In [345]:
# ================================================================
# CELL 15 — KEYWORD EXTRACTION FUNCTION
# ================================================================

def extract_keywords(text, keywords):
    
    text_lower = str(text).lower()
    
    matches = []
    
    for keyword in keywords:
        
        pattern = r"(?<!\w)" + re.escape(keyword.lower()) + r"(?!\w)"
        
        if re.search(pattern, text_lower):
            matches.append(keyword)
    
    return sorted(list(set(matches)))


print("Keyword extraction function created successfully.")

Keyword extraction function created successfully.


In [346]:
# ================================================================
# CELL 16 — EXTRACT CANDIDATE SKILL FEATURES
# ================================================================

candidate_features = candidate_profiles.copy()


# ------------------------------------------------
# PROGRAMMING / TECHNICAL SKILLS
# ------------------------------------------------

candidate_features["technical_skills"] = (
    candidate_features["clean_text"]
    .apply(
        lambda x: extract_keywords(
            x,
            technical_skill_keywords
        )
    )
)


# ------------------------------------------------
# TECHNOLOGIES / FRAMEWORKS
# ------------------------------------------------

candidate_features["technologies"] = (
    candidate_features["clean_text"]
    .apply(
        lambda x: extract_keywords(
            x,
            technology_keywords
        )
    )
)


# ------------------------------------------------
# DATABASES
# ------------------------------------------------

candidate_features["databases"] = (
    candidate_features["clean_text"]
    .apply(
        lambda x: extract_keywords(
            x,
            database_keywords
        )
    )
)


# ------------------------------------------------
# SOFTWARE TOOLS
# ------------------------------------------------

candidate_features["software_tools"] = (
    candidate_features["clean_text"]
    .apply(
        lambda x: extract_keywords(
            x,
            tool_keywords
        )
    )
)


# ------------------------------------------------
# PROFESSIONAL SKILLS
# ------------------------------------------------

candidate_features["professional_skills"] = (
    candidate_features["clean_text"]
    .apply(
        lambda x: extract_keywords(
            x,
            professional_skill_keywords
        )
    )
)


# ------------------------------------------------
# CERTIFICATIONS
# ------------------------------------------------

candidate_features["certifications"] = (
    candidate_features["clean_text"]
    .apply(
        lambda x: extract_keywords(
            x,
            certification_keywords
        )
    )
)


print("=" * 70)
print("WEEK 5 — CANDIDATE FEATURE EXTRACTION")
print("=" * 70)

print("\nShape:")
print(candidate_features.shape)

print("\nNew feature columns:")
print([
    "technical_skills",
    "technologies",
    "databases",
    "software_tools",
    "professional_skills",
    "certifications"
])

display(
    candidate_features[
        [
            "candidate_id",
            "technical_skills",
            "technologies",
            "databases",
            "software_tools",
            "professional_skills"
        ]
    ].head(10)
)

WEEK 5 — CANDIDATE FEATURE EXTRACTION

Shape:
(63, 11)

New feature columns:
['technical_skills', 'technologies', 'databases', 'software_tools', 'professional_skills', 'certifications']


,candidate_id,technical_skills,technologies,databases,software_tools,professional_skills
0,Candidate_001,"[html, java, javascript, r, sql]","[api, bootstrap, hibernate, jquery, rest, restful, spring, spring boot, spring mvc]","[mongodb, mysql, nosql]","[eclipse, git, github, intellij, maven, postman]",[]
1,Candidate_002,"[css, html, javascript, php, typescript]","[angular, bootstrap, jquery, nodejs, react]",[mysql],"[github, vs code]",[]
2,Candidate_003,"[c#, c++, java]",[],[],"[git, jira]",[agile]
3,Candidate_004,"[css, javascript, php, python]","[bootstrap, django, flask, jquery, node.js, react]","[mysql, postgresql, sqlite]","[git, github, jira]",[organization]
4,Candidate_005,"[c#, css, java, javascript, php, sql]","[.net, asp.net, jquery, nodejs, react]","[mysql, sql server]",[visual studio],[organization]
5,Candidate_006,"[css, html, javascript, python, typescript]","[angular, api, bootstrap, jquery, nodejs, rest]","[mongodb, mysql, sqlite]",[],"[agile, management, scrum]"
6,Candidate_007,"[css, html, java, javascript, kotlin, python, scala, sql]","[docker, spring]",[oracle],"[git, gitlab]","[agile, management, scrum]"
7,Candidate_008,"[css, html, javascript, python, sql]","[api, django, docker, jquery, react, vue]","[mysql, postgresql]",[],[management]
8,Candidate_009,[swift],"[api, rest]",[],"[git, github]",[planning]
9,Candidate_010,"[css, html, java, javascript, sql, typescript]","[angular, api, bootstrap, hibernate, jquery, microservices, node.js, react, rest, restful, sprin...","[mongodb, mysql, nosql, postgresql]","[eclipse, git, github, intellij, jira, maven, vs code]","[agile, management, scrum]"


In [347]:
# ================================================================
# CELL 17 — EXTRACT DEGREE LEVEL
# ================================================================

def extract_degree(text):
    
    text_lower = str(text).lower()
    
    # Ordered from highest to lowest
    for degree, keywords in degree_keywords.items():
        
        for keyword in keywords:
            
            pattern = r"(?<!\w)" + re.escape(keyword.lower()) + r"(?!\w)"
            
            if re.search(pattern, text_lower):
                return degree
    
    return "Not Identified"


candidate_features["degree_level"] = (
    candidate_features["clean_text"]
    .apply(extract_degree)
)


print("=" * 70)
print("WEEK 5 — DEGREE LEVEL EXTRACTION")
print("=" * 70)

print("\nDegree distribution:")

display(
    candidate_features["degree_level"]
    .value_counts()
    .reset_index()
    .rename(
        columns={
            "index": "degree_level",
            "degree_level": "candidate_count"
        }
    )
)

WEEK 5 — DEGREE LEVEL EXTRACTION

Degree distribution:


,candidate_count,count
0,Not Identified,30
1,Master,17
2,Bachelor,12
3,Certificate,3
4,Diploma,1


In [348]:
# ================================================================
# CELL 18 — EXTRACT FIELD OF STUDY
# ================================================================

def extract_field_of_study(text):
    
    text_lower = str(text).lower()
    
    matches = []
    
    for field in field_of_study_keywords:
        
        if field.lower() in text_lower:
            matches.append(field)
    
    if len(matches) == 0:
        return "Not Identified"
    
    return matches[0]


candidate_features["field_of_study"] = (
    candidate_features["clean_text"]
    .apply(extract_field_of_study)
)


print("=" * 70)
print("WEEK 5 — FIELD OF STUDY EXTRACTION")
print("=" * 70)

print("\nField of study distribution:")

display(
    candidate_features["field_of_study"]
    .value_counts()
    .reset_index()
)

WEEK 5 — FIELD OF STUDY EXTRACTION

Field of study distribution:


,field_of_study,count
0,education,20
1,computer science,15
2,engineering,5
3,accounting,5
4,mathematics,5
5,software engineering,3
6,finance,2
7,Not Identified,2
8,computer engineering,2
9,information systems,2


In [349]:
# ================================================================
# CELL 19 — REFINED FIELD OF STUDY EXTRACTION
# ================================================================

field_of_study_keywords = [
    
    # Computing and technology
    "computer science",
    "software engineering",
    "computer engineering",
    "information technology",
    "information systems",
    "data science",
    "cyber security",
    "cybersecurity",
    
    # Business
    "business administration",
    "accounting",
    "finance",
    
    # Mathematics and engineering
    "mathematics",
    "applied mathematics",
    "electrical engineering",
    "mechanical engineering",
    "industrial engineering",
    "engineering",
    
    # Education-related academic fields
    "teacher education",
    "educational studies",
    "education studies"
]


def extract_field_of_study(text):
    
    text_lower = str(text).lower()
    
    matches = []
    
    for field in field_of_study_keywords:
        
        pattern = (
            r"(?<!\w)"
            + re.escape(field.lower())
            + r"(?!\w)"
        )
        
        if re.search(pattern, text_lower):
            matches.append(field)
    
    if len(matches) == 0:
        return "Not Identified"
    
    # Return the longest / most specific match
    matches = sorted(
        matches,
        key=len,
        reverse=True
    )
    
    return matches[0]


candidate_features["field_of_study"] = (
    candidate_features["clean_text"]
    .apply(extract_field_of_study)
)


print("=" * 70)
print("WEEK 5 — REFINED FIELD OF STUDY EXTRACTION")
print("=" * 70)

print("\nField of study distribution:")

display(
    candidate_features["field_of_study"]
    .value_counts()
    .reset_index()
)

WEEK 5 — REFINED FIELD OF STUDY EXTRACTION

Field of study distribution:


,field_of_study,count
0,Not Identified,24
1,computer science,9
2,engineering,7
3,information technology,3
4,accounting,3
5,software engineering,3
6,mathematics,3
7,applied mathematics,3
8,information systems,3
9,business administration,2


In [350]:
# ================================================================
# CELL 20 — EDUCATION EXTRACTION REVIEW
# ================================================================

education_review = candidate_features[
    [
        "candidate_id",
        "degree_level",
        "field_of_study",
        "clean_text"
    ]
].copy()

education_review["text_preview"] = (
    education_review["clean_text"]
    .str.replace("\n", " ", regex=False)
    .str[:300]
)

education_review = education_review[
    [
        "candidate_id",
        "degree_level",
        "field_of_study",
        "text_preview"
    ]
]

print("=" * 70)
print("WEEK 5 — EDUCATION FEATURE REVIEW")
print("=" * 70)

print("\nCandidates with identified field of study:")

display(
    education_review[
        education_review["field_of_study"]
        != "Not Identified"
    ].head(20)
)

WEEK 5 — EDUCATION FEATURE REVIEW

Candidates with identified field of study:


,candidate_id,degree_level,field_of_study,text_preview
0,Candidate_001,Master,information technology,**************** Java full stack developer (60% – IN JAVA BACKEND DEVELOPMENT AND 40% – IN WEB F...
1,Candidate_002,Diploma,engineering,****************** WEB FRONTEND DEVELOPER 5 years of experience in frontend development EXPERIEN...
2,Candidate_003,Master,business administration,"******************** Experience in C, C++ and C# programming Working knowledge of Agile, Jira an..."
3,Candidate_004,Not Identified,computer science,***************** Front-end Developer Mobile phone: ********** Linkedln:\t *******...
4,Candidate_005,Master,engineering,"Rehovot, Israel About me Senior solution architect with wide experience of creating and mainta..."
5,Candidate_006,Master,accounting,"Frontend Developer - 8 years experience Phone: Email: Residence: Holon Languages: Hebrew, Engl..."
6,Candidate_007,Master,computer science,"Senior Software Engineer (Java/Kotlin Backend Developer) Israel, Beer Sheva Contacts Education T..."
8,Candidate_009,Not Identified,engineering,- WhatsApp - Israel GitHub: Executive Summary Simultaneously managing multiple projects with ...
9,Candidate_010,Master,computer science,Full Stack Java Developer 5 years of total experience in development web frontend (70%) and dev...
11,Candidate_012,Bachelor,information technology,"Full-Stack Developer SUMMARY Main stack: PHP, Node.js, React.js (Redux, Mobx), MySQL/PostgreSQL...."


In [351]:
# ================================================================
# CELL 21A — LOAD WEEK 4 O*NET SKILL PROFILE
# ================================================================

import pandas as pd
import os

onet_skills_path = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Outputs\Tables\onet_skills_profile.csv"
)

occupation_skill_profiles = pd.read_csv(
    onet_skills_path
)

print("=" * 70)
print("WEEK 5 — LOAD WEEK 4 O*NET SKILL PROFILE")
print("=" * 70)

print("\nFile exists:")
print(os.path.exists(onet_skills_path))

print("\nShape:")
print(occupation_skill_profiles.shape)

print("\nColumns:")
print(occupation_skill_profiles.columns.tolist())

print("\nFirst 10 rows:")
display(
    occupation_skill_profiles.head(10)
)

WEEK 5 — LOAD WEEK 4 O*NET SKILL PROFILE

File exists:
True

Shape:
(200, 7)

Columns:
['onet_soc_code', 'element_id', 'skill', 'importance', 'level', 'selected_occupation', 'onet_title']

First 10 rows:


,onet_soc_code,element_id,skill,importance,level,selected_occupation,onet_title
0,11-9051.00,2.A.1.a,Reading Comprehension,3.75,3.88,Restaurant Manager,Food Service Managers
1,11-9051.00,2.A.1.b,Active Listening,3.88,3.50,Restaurant Manager,Food Service Managers
2,11-9051.00,2.A.1.c,Writing,3.00,3.12,Restaurant Manager,Food Service Managers
3,11-9051.00,2.A.1.d,Speaking,3.88,4.00,Restaurant Manager,Food Service Managers
4,11-9051.00,2.A.1.e,Mathematics,2.88,2.88,Restaurant Manager,Food Service Managers
5,11-9051.00,2.A.1.f,Science,1.62,0.62,Restaurant Manager,Food Service Managers
6,11-9051.00,2.A.2.a,Critical Thinking,3.62,3.75,Restaurant Manager,Food Service Managers
7,11-9051.00,2.A.2.b,Active Learning,3.12,3.75,Restaurant Manager,Food Service Managers
8,11-9051.00,2.A.2.c,Learning Strategies,3.12,3.12,Restaurant Manager,Food Service Managers
9,11-9051.00,2.A.2.d,Monitoring,3.88,3.88,Restaurant Manager,Food Service Managers


In [352]:
# ================================================================
# CELL 21B — VALIDATE WEEK 4 O*NET SKILL PROFILE
# ================================================================

print("=" * 70)
print("WEEK 5 — O*NET SKILL PROFILE VALIDATION")
print("=" * 70)

required_columns = [
    "selected_occupation",
    "skill",
    "importance",
    "level"
]

print("\nRequired columns present:")

for column in required_columns:
    print(
        f"{column}:",
        column in occupation_skill_profiles.columns
    )

print("\nUnique occupations:")
print(
    occupation_skill_profiles[
        "selected_occupation"
    ].nunique()
)

print("\nTotal skill records:")
print(
    len(occupation_skill_profiles)
)

display(
    occupation_skill_profiles
    .groupby("selected_occupation")
    .agg(
        total_skill_records=("skill", "size"),
        unique_skills=("skill", "nunique")
    )
    .reset_index()
)

WEEK 5 — O*NET SKILL PROFILE VALIDATION

Required columns present:
selected_occupation: True
skill: True
importance: True
level: True

Unique occupations:
15

Total skill records:
200


,selected_occupation,total_skill_records,unique_skills
0,Administrative Assistant,10,10
1,Bookkeeper,10,10
2,Continuing Care Assistant,20,10
3,Delivery Driver,10,10
4,"Driver, Truck",10,10
5,Food Service Supervisor,10,10
6,Information Technology (IT) Analyst,40,10
7,Inside Sales Representative,10,10
8,Licensed Practical Nurse (L.P.N.),10,10
9,Office Administrator,20,10


In [353]:
# ================================================================
# CELL 21 — BUILD CONTROLLED O*NET SKILL VOCABULARY
# ================================================================

onet_skill_vocabulary = (
    occupation_skill_profiles["skill"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("=" * 70)
print("WEEK 5 — CONTROLLED O*NET SKILL VOCABULARY")
print("=" * 70)

print("\nNumber of unique O*NET skills:")
print(len(onet_skill_vocabulary))

print("\nO*NET skills:")
print(onet_skill_vocabulary)

WEEK 5 — CONTROLLED O*NET SKILL VOCABULARY

Number of unique O*NET skills:
10

O*NET skills:
['Active Learning', 'Active Listening', 'Critical Thinking', 'Learning Strategies', 'Mathematics', 'Monitoring', 'Reading Comprehension', 'Science', 'Speaking', 'Writing']


In [354]:
# ================================================================
# CELL 22 — CONSOLIDATE O*NET SKILL PROFILES
# ================================================================

# Some Canadian occupations map to multiple O*NET occupations.
# Consolidate repeated skills by calculating the mean importance
# and mean level across the valid O*NET mappings.

occupation_skill_profiles_final = (
    occupation_skill_profiles
    .groupby(
        [
            "selected_occupation",
            "skill"
        ],
        as_index=False
    )
    .agg(
        importance=("importance", "mean"),
        level=("level", "mean"),
        mapping_count=("onet_soc_code", "nunique")
    )
)

occupation_skill_profiles_final = (
    occupation_skill_profiles_final
    .sort_values(
        [
            "selected_occupation",
            "importance"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("WEEK 5 — FINAL OCCUPATION SKILL PROFILES")
print("=" * 70)

print("\nShape:")
print(occupation_skill_profiles_final.shape)

print("\nUnique occupations:")
print(
    occupation_skill_profiles_final[
        "selected_occupation"
    ].nunique()
)

print("\nUnique occupation-skill combinations:")
print(
    occupation_skill_profiles_final.shape[0]
)

display(
    occupation_skill_profiles_final.head(20)
)

WEEK 5 — FINAL OCCUPATION SKILL PROFILES

Shape:
(150, 5)

Unique occupations:
15

Unique occupation-skill combinations:
150


,selected_occupation,skill,importance,level,mapping_count
0,Administrative Assistant,Active Listening,4.00,3.75,1
1,Administrative Assistant,Speaking,4.00,3.62,1
2,Administrative Assistant,Reading Comprehension,3.88,3.88,1
3,Administrative Assistant,Writing,3.75,3.50,1
4,Administrative Assistant,Monitoring,3.12,3.25,1
5,Administrative Assistant,Critical Thinking,3.00,3.62,1
6,Administrative Assistant,Active Learning,2.88,3.00,1
7,Administrative Assistant,Learning Strategies,2.12,1.88,1
8,Administrative Assistant,Mathematics,2.00,1.62,1
9,Administrative Assistant,Science,1.00,0.00,1


In [355]:
# ================================================================
# CELL 23 — VALIDATE FINAL OCCUPATION SKILL PROFILES
# ================================================================

occupation_skill_validation = (
    occupation_skill_profiles_final
    .groupby("selected_occupation")
    .agg(
        total_skill_records=("skill", "size"),
        unique_skills=("skill", "nunique"),
        minimum_importance=("importance", "min"),
        maximum_importance=("importance", "max")
    )
    .reset_index()
)

occupation_skill_validation[
    "duplicate_skill_records"
] = (
    occupation_skill_validation["total_skill_records"]
    - occupation_skill_validation["unique_skills"]
)

print("=" * 70)
print("WEEK 5 — FINAL OCCUPATION SKILL PROFILE VALIDATION")
print("=" * 70)

display(occupation_skill_validation)

print("\nAll occupations have exactly 10 unique skills:")

print(
    occupation_skill_validation[
        "unique_skills"
    ].eq(10).all()
)

print("\nTotal final occupation-skill records:")

print(
    len(occupation_skill_profiles_final)
)

WEEK 5 — FINAL OCCUPATION SKILL PROFILE VALIDATION


,selected_occupation,total_skill_records,unique_skills,minimum_importance,maximum_importance,duplicate_skill_records
0,Administrative Assistant,10,10,1.00,4.000,0
1,Bookkeeper,10,10,1.12,3.380,0
2,Continuing Care Assistant,10,10,1.50,3.370,0
3,Delivery Driver,10,10,1.12,3.120,0
4,"Driver, Truck",10,10,1.00,3.120,0
5,Food Service Supervisor,10,10,1.00,3.880,0
6,Information Technology (IT) Analyst,10,10,2.25,4.065,0
7,Inside Sales Representative,10,10,1.00,3.750,0
8,Licensed Practical Nurse (L.P.N.),10,10,2.62,3.880,0
9,Office Administrator,10,10,1.19,4.000,0



All occupations have exactly 10 unique skills:
True

Total final occupation-skill records:
150


In [356]:
# ================================================================
# CELL 24 — DEFINE CANDIDATE RAW SKILL VOCABULARY
# ================================================================

candidate_skill_dictionary = {

    "Programming Languages": [
        "Python",
        "Java",
        "JavaScript",
        "TypeScript",
        "C++",
        "C#",
        "C",
        "Kotlin",
        "PHP",
        "SQL",
        "R"
    ],

    "Web Technologies": [
        "HTML",
        "CSS",
        "React",
        "Angular",
        "Vue",
        "Node.js",
        "NodeJS",
        "REST",
        "REST API",
        "GraphQL",
        "JSON"
    ],

    "Frameworks": [
        "Spring",
        "Spring Boot",
        "Spring MVC",
        ".NET",
        ".NET Core",
        "ASP.NET",
        "Django",
        "Flask",
        "Hibernate",
        "JPA"
    ],

    "Databases": [
        "MySQL",
        "PostgreSQL",
        "MongoDB",
        "Oracle",
        "SQLite",
        "NoSQL"
    ],

    "Cloud and DevOps": [
        "AWS",
        "Azure",
        "Docker",
        "Kubernetes",
        "Jenkins",
        "CI/CD",
        "GitHub Actions"
    ],

    "Data and Analytics": [
        "Excel",
        "Tableau",
        "Power BI",
        "Pandas",
        "NumPy",
        "Apache Spark",
        "Machine Learning",
        "Data Analysis"
    ],

    "Development Tools": [
        "Git",
        "GitHub",
        "Jira",
        "Postman",
        "Maven",
        "Gradle",
        "IntelliJ",
        "Eclipse"
    ],

    "Professional Methods": [
        "Agile",
        "Scrum",
        "Project Management",
        "Problem Solving",
        "Teamwork",
        "Communication"
    ]
}


print("=" * 70)
print("WEEK 5 — CANDIDATE RAW SKILL DICTIONARY")
print("=" * 70)

total_skills = sum(
    len(skills)
    for skills in candidate_skill_dictionary.values()
)

print("\nSkill categories:")
print(
    len(candidate_skill_dictionary)
)

print("\nTotal skill terms:")
print(total_skills)

for category, skills in candidate_skill_dictionary.items():
    print(f"\n{category}:")
    print(skills)

WEEK 5 — CANDIDATE RAW SKILL DICTIONARY

Skill categories:
8

Total skill terms:
67

Programming Languages:
['Python', 'Java', 'JavaScript', 'TypeScript', 'C++', 'C#', 'C', 'Kotlin', 'PHP', 'SQL', 'R']

Web Technologies:
['HTML', 'CSS', 'React', 'Angular', 'Vue', 'Node.js', 'NodeJS', 'REST', 'REST API', 'GraphQL', 'JSON']

Frameworks:
['Spring', 'Spring Boot', 'Spring MVC', '.NET', '.NET Core', 'ASP.NET', 'Django', 'Flask', 'Hibernate', 'JPA']

Databases:
['MySQL', 'PostgreSQL', 'MongoDB', 'Oracle', 'SQLite', 'NoSQL']

Cloud and DevOps:
['AWS', 'Azure', 'Docker', 'Kubernetes', 'Jenkins', 'CI/CD', 'GitHub Actions']

Data and Analytics:
['Excel', 'Tableau', 'Power BI', 'Pandas', 'NumPy', 'Apache Spark', 'Machine Learning', 'Data Analysis']

Development Tools:
['Git', 'GitHub', 'Jira', 'Postman', 'Maven', 'Gradle', 'IntelliJ', 'Eclipse']

Professional Methods:
['Agile', 'Scrum', 'Project Management', 'Problem Solving', 'Teamwork', 'Communication']


In [357]:
# ================================================================
# CELL 25 — EXTRACT RAW CANDIDATE SKILLS
# ================================================================

import re


def extract_candidate_skills(text):

    text_lower = str(text).lower()

    extracted_skills = []

    for category, skills in candidate_skill_dictionary.items():

        for skill in skills:

            skill_lower = skill.lower()

            pattern = (
                r"(?<!\w)"
                + re.escape(skill_lower)
                + r"(?!\w)"
            )

            if re.search(pattern, text_lower):

                extracted_skills.append(skill)

    return sorted(
        list(set(extracted_skills))
    )


candidate_features["raw_skills"] = (
    candidate_features["clean_text"]
    .apply(extract_candidate_skills)
)

candidate_features["raw_skill_count"] = (
    candidate_features["raw_skills"]
    .apply(len)
)


print("=" * 70)
print("WEEK 5 — RAW CANDIDATE SKILL EXTRACTION")
print("=" * 70)

print("\nCandidates:")
print(len(candidate_features))

print("\nCandidates with at least one extracted skill:")
print(
    (
        candidate_features["raw_skill_count"] > 0
    ).sum()
)

print("\nCandidates with no extracted skills:")
print(
    (
        candidate_features["raw_skill_count"] == 0
    ).sum()
)

print("\nSkill count summary:")

display(
    candidate_features[
        "raw_skill_count"
    ].describe()
)

WEEK 5 — RAW CANDIDATE SKILL EXTRACTION

Candidates:
63

Candidates with at least one extracted skill:
63

Candidates with no extracted skills:
0

Skill count summary:


count    63.000000
mean     12.777778
std       7.187454
min       1.000000
25%       7.500000
50%      12.000000
75%      16.500000
max      37.000000
Name: raw_skill_count, dtype: float64

In [358]:
# ================================================================
# CELL 26 — REVIEW CANDIDATE SKILL EXTRACTION
# ================================================================

skill_review = candidate_features[
    [
        "candidate_id",
        "raw_skill_count",
        "raw_skills"
    ]
].copy()


print("=" * 70)
print("WEEK 5 — CANDIDATE SKILL EXTRACTION REVIEW")
print("=" * 70)

display(
    skill_review.head(20)
)

WEEK 5 — CANDIDATE SKILL EXTRACTION REVIEW


,candidate_id,raw_skill_count,raw_skills
0,Candidate_001,21,"[Eclipse, Git, GitHub, HTML, Hibernate, IntelliJ, JPA, JSON, Java, JavaScript, Maven, MongoDB, M..."
1,Candidate_002,10,"[Angular, CSS, GitHub, HTML, JavaScript, MySQL, NodeJS, PHP, React, TypeScript]"
2,Candidate_003,7,"[Agile, C, C#, C++, Git, Java, Jira]"
3,Candidate_004,14,"[CSS, Django, Flask, Git, GitHub, JavaScript, Jira, MySQL, Node.js, PHP, PostgreSQL, Python, Rea..."
4,Candidate_005,12,"[.NET, ASP.NET, C, C#, CSS, Java, JavaScript, MySQL, NodeJS, PHP, React, SQL]"
5,Candidate_006,14,"[Agile, Angular, CSS, HTML, JavaScript, MongoDB, MySQL, NodeJS, Python, REST, REST API, SQLite, ..."
6,Candidate_007,13,"[Agile, CSS, Docker, Git, HTML, Java, JavaScript, Kotlin, Oracle, Python, SQL, Scrum, Spring]"
7,Candidate_008,11,"[CSS, Django, Docker, HTML, JavaScript, MySQL, PostgreSQL, Python, React, SQL, Vue]"
8,Candidate_009,4,"[Git, GitHub, REST, REST API]"
9,Candidate_010,29,"[Agile, Angular, CSS, Eclipse, Excel, Git, GitHub, HTML, Hibernate, IntelliJ, JSON, Java, JavaSc..."


In [359]:
# ================================================================
# CELL 27 — DEFINE CANDIDATE-TO-O*NET SKILL EVIDENCE MAPPING
# ================================================================

candidate_to_onet_mapping = {

    "Active Learning": [
        "Agile",
        "Scrum",
        "Python",
        "Java",
        "JavaScript",
        "TypeScript",
        "Kotlin",
        "PHP",
        "C++",
        "C#",
        "C",
        "Machine Learning"
    ],

    "Active Listening": [
        "Communication",
        "Teamwork",
        "Project Management",
        "Agile",
        "Scrum"
    ],

    "Critical Thinking": [
        "Python",
        "Java",
        "JavaScript",
        "TypeScript",
        "C++",
        "C#",
        "C",
        "Kotlin",
        "PHP",
        "SQL",
        "R",
        "Machine Learning",
        "Data Analysis",
        "Problem Solving"
    ],

    "Learning Strategies": [
        "Agile",
        "Scrum",
        "Machine Learning",
        "Python",
        "Java",
        "JavaScript",
        "TypeScript"
    ],

    "Mathematics": [
        "SQL",
        "Python",
        "R",
        "Excel",
        "Pandas",
        "NumPy",
        "Data Analysis",
        "Machine Learning",
        "Apache Spark"
    ],

    "Monitoring": [
        "Jira",
        "Jenkins",
        "CI/CD",
        "GitHub Actions",
        "Docker",
        "Kubernetes"
    ],

    "Reading Comprehension": [
        "Python",
        "Java",
        "JavaScript",
        "TypeScript",
        "C++",
        "C#",
        "SQL",
        "Git",
        "GitHub",
        "Jira",
        "REST API",
        "GraphQL"
    ],

    "Science": [
        "Python",
        "R",
        "NumPy",
        "Pandas",
        "Machine Learning",
        "Apache Spark"
    ],

    "Speaking": [
        "Communication",
        "Teamwork",
        "Project Management",
        "Agile",
        "Scrum"
    ],

    "Writing": [
        "Git",
        "GitHub",
        "Jira",
        "REST API",
        "GraphQL",
        "JSON",
        "Project Management"
    ]
}


print("=" * 70)
print("WEEK 5 — CANDIDATE-TO-O*NET SKILL EVIDENCE MAPPING")
print("=" * 70)

print("\nNumber of O*NET skill categories:")
print(len(candidate_to_onet_mapping))

print("\nMapping:")

for onet_skill, evidence in candidate_to_onet_mapping.items():

    print(f"\n{onet_skill}")
    print("Evidence terms:")
    print(evidence)

WEEK 5 — CANDIDATE-TO-O*NET SKILL EVIDENCE MAPPING

Number of O*NET skill categories:
10

Mapping:

Active Learning
Evidence terms:
['Agile', 'Scrum', 'Python', 'Java', 'JavaScript', 'TypeScript', 'Kotlin', 'PHP', 'C++', 'C#', 'C', 'Machine Learning']

Active Listening
Evidence terms:
['Communication', 'Teamwork', 'Project Management', 'Agile', 'Scrum']

Critical Thinking
Evidence terms:
['Python', 'Java', 'JavaScript', 'TypeScript', 'C++', 'C#', 'C', 'Kotlin', 'PHP', 'SQL', 'R', 'Machine Learning', 'Data Analysis', 'Problem Solving']

Learning Strategies
Evidence terms:
['Agile', 'Scrum', 'Machine Learning', 'Python', 'Java', 'JavaScript', 'TypeScript']

Mathematics
Evidence terms:
['SQL', 'Python', 'R', 'Excel', 'Pandas', 'NumPy', 'Data Analysis', 'Machine Learning', 'Apache Spark']

Monitoring
Evidence terms:
['Jira', 'Jenkins', 'CI/CD', 'GitHub Actions', 'Docker', 'Kubernetes']

Reading Comprehension
Evidence terms:
['Python', 'Java', 'JavaScript', 'TypeScript', 'C++', 'C#', 'SQL',

In [360]:
# ================================================================
# CELL 28 — MAP CANDIDATES TO O*NET SKILL EVIDENCE
# ================================================================

def map_candidate_to_onet_skills(raw_skills):
    
    # Convert candidate skills to lowercase
    candidate_skill_set = {
        str(skill).lower().strip()
        for skill in raw_skills
    }
    
    mapped_skills = {}
    
    for onet_skill, evidence_terms in candidate_to_onet_mapping.items():
        
        matched_evidence = []
        
        for term in evidence_terms:
            
            if term.lower() in candidate_skill_set:
                matched_evidence.append(term)
        
        mapped_skills[onet_skill] = {
            "has_evidence": int(len(matched_evidence) > 0),
            "matched_evidence": sorted(
                list(set(matched_evidence))
            ),
            "evidence_count": len(
                set(matched_evidence)
            )
        }
    
    return mapped_skills


candidate_features["onet_skill_mapping"] = (
    candidate_features["raw_skills"]
    .apply(map_candidate_to_onet_skills)
)


print("=" * 70)
print("WEEK 5 — CANDIDATE-TO-O*NET SKILL MAPPING")
print("=" * 70)

print("\nCandidates processed:")
print(len(candidate_features))

print("\nExample candidate mapping:\n")

example_candidate = (
    candidate_features.iloc[0]
)

print("Candidate ID:")
print(example_candidate["candidate_id"])

print("\nRaw skills:")
print(example_candidate["raw_skills"])

print("\nMapped O*NET skills:")

for skill, result in example_candidate["onet_skill_mapping"].items():
    
    print(f"\n{skill}")
    print("Has evidence:", result["has_evidence"])
    print("Matched evidence:", result["matched_evidence"])
    print("Evidence count:", result["evidence_count"])

WEEK 5 — CANDIDATE-TO-O*NET SKILL MAPPING

Candidates processed:
63

Example candidate mapping:

Candidate ID:
Candidate_001

Raw skills:
['Eclipse', 'Git', 'GitHub', 'HTML', 'Hibernate', 'IntelliJ', 'JPA', 'JSON', 'Java', 'JavaScript', 'Maven', 'MongoDB', 'MySQL', 'NoSQL', 'Postman', 'R', 'REST', 'SQL', 'Spring', 'Spring Boot', 'Spring MVC']

Mapped O*NET skills:

Active Learning
Has evidence: 1
Matched evidence: ['Java', 'JavaScript']
Evidence count: 2

Active Listening
Has evidence: 0
Matched evidence: []
Evidence count: 0

Critical Thinking
Has evidence: 1
Matched evidence: ['Java', 'JavaScript', 'R', 'SQL']
Evidence count: 4

Learning Strategies
Has evidence: 1
Matched evidence: ['Java', 'JavaScript']
Evidence count: 2

Mathematics
Has evidence: 1
Matched evidence: ['R', 'SQL']
Evidence count: 2

Monitoring
Has evidence: 0
Matched evidence: []
Evidence count: 0

Reading Comprehension
Has evidence: 1
Matched evidence: ['Git', 'GitHub', 'Java', 'JavaScript', 'SQL']
Evidence count: 5

In [361]:
# ================================================================
# CELL 29 — BUILD CANDIDATE O*NET SKILL MATRIX
# ================================================================

candidate_onet_records = []

for _, row in candidate_features.iterrows():
    
    candidate_id = row["candidate_id"]
    skill_mapping = row["onet_skill_mapping"]
    
    record = {
        "candidate_id": candidate_id
    }
    
    # Create one column for each O*NET skill
    for onet_skill in candidate_to_onet_mapping.keys():
        
        record[onet_skill] = (
            skill_mapping[onet_skill]["has_evidence"]
        )
        
        record[f"{onet_skill}_evidence_count"] = (
            skill_mapping[onet_skill]["evidence_count"]
        )
    
    candidate_onet_records.append(record)


candidate_onet_skill_matrix = pd.DataFrame(
    candidate_onet_records
)


print("=" * 70)
print("WEEK 5 — CANDIDATE O*NET SKILL MATRIX")
print("=" * 70)

print("\nShape:")
print(candidate_onet_skill_matrix.shape)

print("\nNumber of candidates:")
print(
    candidate_onet_skill_matrix["candidate_id"].nunique()
)

print("\nFirst 10 candidates:")

display(
    candidate_onet_skill_matrix.head(10)
)

WEEK 5 — CANDIDATE O*NET SKILL MATRIX

Shape:
(63, 21)

Number of candidates:
63

First 10 candidates:


,candidate_id,Active Learning,Active Learning_evidence_count,Active Listening,Active Listening_evidence_count,Critical Thinking,Critical Thinking_evidence_count,Learning Strategies,Learning Strategies_evidence_count,Mathematics,Mathematics_evidence_count,Monitoring,Monitoring_evidence_count,Reading Comprehension,Reading Comprehension_evidence_count,Science,Science_evidence_count,Speaking,Speaking_evidence_count,Writing,Writing_evidence_count
0,Candidate_001,1,2,0,0,1,4,1,2,1,2,0,0,1,5,1,1,0,0,1,3
1,Candidate_002,1,3,0,0,1,3,1,2,0,0,0,0,1,3,0,0,0,0,1,1
2,Candidate_003,1,5,1,1,1,4,1,2,0,0,1,1,1,5,0,0,1,1,1,2
3,Candidate_004,1,3,0,0,1,3,1,2,1,1,1,1,1,5,1,1,0,0,1,3
4,Candidate_005,1,5,0,0,1,6,1,2,1,1,0,0,1,4,0,0,0,0,0,0
5,Candidate_006,1,5,1,2,1,3,1,5,1,1,0,0,1,4,1,1,1,2,1,1
6,Candidate_007,1,6,1,2,1,5,1,5,1,2,1,1,1,5,1,1,1,2,1,1
7,Candidate_008,1,2,0,0,1,3,1,2,1,2,1,1,1,3,1,1,0,0,0,0
8,Candidate_009,0,0,0,0,0,0,0,0,0,0,0,0,1,3,0,0,0,0,1,3
9,Candidate_010,1,5,1,3,1,4,1,5,1,2,1,1,1,7,0,0,1,3,1,5


In [362]:
# ================================================================
# CELL 30 — VALIDATE CANDIDATE O*NET SKILL COVERAGE
# ================================================================

# List of the 10 controlled O*NET skills
onet_skill_columns = list(
    candidate_to_onet_mapping.keys()
)

# ------------------------------------------------
# Calculate total O*NET skills with evidence
# for each candidate
# ------------------------------------------------

candidate_onet_skill_matrix[
    "onet_skill_match_count"
] = (
    candidate_onet_skill_matrix[
        onet_skill_columns
    ]
    .sum(axis=1)
)

# ------------------------------------------------
# Calculate total evidence terms
# for each candidate
# ------------------------------------------------

evidence_count_columns = [
    f"{skill}_evidence_count"
    for skill in onet_skill_columns
]

candidate_onet_skill_matrix[
    "total_onet_evidence_count"
] = (
    candidate_onet_skill_matrix[
        evidence_count_columns
    ]
    .sum(axis=1)
)


print("=" * 70)
print("WEEK 5 — CANDIDATE O*NET SKILL COVERAGE VALIDATION")
print("=" * 70)

print("\nCandidates:")
print(
    candidate_onet_skill_matrix[
        "candidate_id"
    ].nunique()
)

print("\nCandidates with at least one O*NET skill match:")
print(
    (
        candidate_onet_skill_matrix[
            "onet_skill_match_count"
        ] > 0
    ).sum()
)

print("\nCandidates with no O*NET skill matches:")
print(
    (
        candidate_onet_skill_matrix[
            "onet_skill_match_count"
        ] == 0
    ).sum()
)

print("\nO*NET skill matches per candidate:")

display(
    candidate_onet_skill_matrix[
        "onet_skill_match_count"
    ]
    .describe()
)

print("\nTotal evidence terms per candidate:")

display(
    candidate_onet_skill_matrix[
        "total_onet_evidence_count"
    ]
    .describe()
)

print("\nCandidate coverage review:")

display(
    candidate_onet_skill_matrix[
        [
            "candidate_id",
            "onet_skill_match_count",
            "total_onet_evidence_count"
        ]
    ]
    .sort_values(
        "onet_skill_match_count",
        ascending=False
    )
    .head(20)
)

WEEK 5 — CANDIDATE O*NET SKILL COVERAGE VALIDATION

Candidates:
63

Candidates with at least one O*NET skill match:
63

Candidates with no O*NET skill matches:
0

O*NET skill matches per candidate:


count    63.000000
mean      7.238095
std       1.729388
min       2.000000
25%       6.000000
50%       7.000000
75%       8.000000
max      10.000000
Name: onet_skill_match_count, dtype: float64


Total evidence terms per candidate:


count    63.000000
mean     18.968254
std       8.765711
min       5.000000
25%      12.500000
50%      18.000000
75%      24.000000
max      48.000000
Name: total_onet_evidence_count, dtype: float64


Candidate coverage review:


,candidate_id,onet_skill_match_count,total_onet_evidence_count
31,Candidate_032,10,30
6,Candidate_007,10,30
28,Candidate_029,10,17
38,Candidate_039,10,13
14,Candidate_015,10,31
60,Candidate_061,10,36
17,Candidate_018,10,18
5,Candidate_006,9,24
53,Candidate_054,9,48
21,Candidate_022,9,35


In [363]:
# ================================================================
# CELL 31 — PREPARE OCCUPATION-SPECIFIC SKILL WEIGHTS
# ================================================================

# Keep the columns required for the baseline recommendation model
occupation_skill_weights = (
    occupation_skill_profiles_final[
        [
            "selected_occupation",
            "skill",
            "importance",
            "level"
        ]
    ]
    .copy()
)

# Normalize importance within each occupation so that
# the occupation-specific weights sum to 1
occupation_skill_weights[
    "normalized_importance"
] = (
    occupation_skill_weights["importance"]
    /
    occupation_skill_weights
    .groupby("selected_occupation")["importance"]
    .transform("sum")
)


print("=" * 70)
print("WEEK 5 — OCCUPATION-SPECIFIC SKILL WEIGHTS")
print("=" * 70)

print("\nNumber of occupations:")
print(
    occupation_skill_weights[
        "selected_occupation"
    ].nunique()
)

print("\nTotal occupation-skill records:")
print(
    len(occupation_skill_weights)
)

print("\nWeight validation:")

weight_validation = (
    occupation_skill_weights
    .groupby("selected_occupation")
    .agg(
        skill_count=("skill", "nunique"),
        importance_sum=("importance", "sum"),
        normalized_weight_sum=(
            "normalized_importance",
            "sum"
        )
    )
    .reset_index()
)

display(weight_validation)

print("\nAll occupations have 10 skills:")

print(
    weight_validation[
        "skill_count"
    ].eq(10).all()
)

print("\nAll normalized weights sum to approximately 1:")

print(
    np.isclose(
        weight_validation[
            "normalized_weight_sum"
        ],
        1.0
    ).all()
)

print("\nExample occupation weights:")

display(
    occupation_skill_weights[
        occupation_skill_weights[
            "selected_occupation"
        ] == "Software Developer"
    ]
    .sort_values(
        "normalized_importance",
        ascending=False
    )
)

WEEK 5 — OCCUPATION-SPECIFIC SKILL WEIGHTS

Number of occupations:
15

Total occupation-skill records:
150

Weight validation:


,selected_occupation,skill_count,importance_sum,normalized_weight_sum
0,Administrative Assistant,10,29.7500,1.0
1,Bookkeeper,10,28.2400,1.0
2,Continuing Care Assistant,10,25.9800,1.0
3,Delivery Driver,10,25.7200,1.0
4,"Driver, Truck",10,25.2500,1.0
5,Food Service Supervisor,10,31.0000,1.0
6,Information Technology (IT) Analyst,10,34.4125,1.0
7,Inside Sales Representative,10,28.8600,1.0
8,Licensed Practical Nurse (L.P.N.),10,34.3900,1.0
9,Office Administrator,10,33.4400,1.0



All occupations have 10 skills:
True

All normalized weights sum to approximately 1:
True

Example occupation weights:


,selected_occupation,skill,importance,level,normalized_importance
140,Software Developer,Critical Thinking,3.88,4.12,0.124679
141,Software Developer,Active Learning,3.50,3.62,0.112468
142,Software Developer,Reading Comprehension,3.50,4.25,0.112468
143,Software Developer,Active Listening,3.38,3.88,0.108612
144,Software Developer,Writing,3.25,3.62,0.104434
145,Software Developer,Speaking,3.12,3.62,0.100257
146,Software Developer,Monitoring,3.00,3.50,0.096401
147,Software Developer,Mathematics,2.75,3.25,0.088368
148,Software Developer,Learning Strategies,2.62,3.12,0.084190
149,Software Developer,Science,2.12,1.88,0.068123


In [364]:
# ================================================================
# CELL 32 — CALCULATE WEIGHTED CANDIDATE-OCCUPATION MATCH SCORES
# ================================================================

recommendation_results = []

# ------------------------------------------------
# Compare each candidate against each occupation
# ------------------------------------------------

for _, candidate_row in candidate_onet_skill_matrix.iterrows():

    candidate_id = candidate_row["candidate_id"]

    # Loop through the 15 selected occupations
    for occupation in occupation_skill_weights[
        "selected_occupation"
    ].unique():

        occupation_profile = (
            occupation_skill_weights[
                occupation_skill_weights[
                    "selected_occupation"
                ] == occupation
            ]
        )

        weighted_score = 0

        # --------------------------------------------
        # Calculate occupation-specific weighted overlap
        # --------------------------------------------

        for _, skill_row in occupation_profile.iterrows():

            skill = skill_row["skill"]

            skill_weight = (
                skill_row["normalized_importance"]
            )

            # Candidate has evidence for skill = 1
            # Candidate has no evidence = 0
            candidate_match = candidate_row[skill]

            weighted_score += (
                candidate_match * skill_weight
            )

        recommendation_results.append(
            {
                "candidate_id": candidate_id,
                "selected_occupation": occupation,
                "recommendation_score": weighted_score,
                "recommendation_percentage": (
                    weighted_score * 100
                )
            }
        )


# ------------------------------------------------
# Create final recommendation DataFrame
# ------------------------------------------------

candidate_occupation_scores = pd.DataFrame(
    recommendation_results
)

# ------------------------------------------------
# Rank occupations for each candidate
# ------------------------------------------------

candidate_occupation_scores[
    "occupation_rank"
] = (
    candidate_occupation_scores
    .groupby("candidate_id")[
        "recommendation_score"
    ]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)


# ------------------------------------------------
# Sort final results
# ------------------------------------------------

candidate_occupation_scores = (
    candidate_occupation_scores
    .sort_values(
        [
            "candidate_id",
            "occupation_rank",
            "selected_occupation"
        ]
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("WEEK 5 — BASELINE WEIGHTED OCCUPATION RECOMMENDATION")
print("=" * 70)

print("\nShape:")
print(candidate_occupation_scores.shape)

print("\nExpected candidate-occupation comparisons:")
print(
    candidate_onet_skill_matrix[
        "candidate_id"
    ].nunique()
    *
    occupation_skill_weights[
        "selected_occupation"
    ].nunique()
)

print("\nActual candidate-occupation comparisons:")
print(
    len(candidate_occupation_scores)
)

print("\nUnique candidates:")
print(
    candidate_occupation_scores[
        "candidate_id"
    ].nunique()
)

print("\nUnique occupations:")
print(
    candidate_occupation_scores[
        "selected_occupation"
    ].nunique()
)

print("\nRecommendation score range:")

print(
    "Minimum:",
    round(
        candidate_occupation_scores[
            "recommendation_percentage"
        ].min(),
        2
    )
)

print(
    "Maximum:",
    round(
        candidate_occupation_scores[
            "recommendation_percentage"
        ].max(),
        2
    )
)

print("\nFirst 20 candidate-occupation results:")

display(
    candidate_occupation_scores.head(20)
)

WEEK 5 — BASELINE WEIGHTED OCCUPATION RECOMMENDATION

Shape:
(945, 5)

Expected candidate-occupation comparisons:
945

Actual candidate-occupation comparisons:
945

Unique candidates:
63

Unique occupations:
15

Recommendation score range:
Minimum: 20.16
Maximum: 100.0

First 20 candidate-occupation results:


,candidate_id,selected_occupation,recommendation_score,recommendation_percentage,occupation_rank
0,Candidate_001,Software Developer,0.694730,69.473008,1
1,Candidate_001,Information Technology (IT) Analyst,0.673956,67.395568,2
2,Candidate_001,Secondary School Teacher,0.668896,66.889632,3
3,Candidate_001,Bookkeeper,0.668201,66.820113,4
4,Candidate_001,Licensed Practical Nurse (L.P.N.),0.661530,66.152951,5
5,Candidate_001,Office Administrator,0.656100,65.610048,6
6,Candidate_001,Office Manager,0.654676,65.467626,7
7,Candidate_001,Restaurant Manager,0.644580,64.458015,8
8,Candidate_001,"Driver, Truck",0.643564,64.356436,9
9,Candidate_001,Continuing Care Assistant,0.639530,63.953041,10


In [365]:
# ================================================================
# CELL 33 — REVIEW RECOMMENDATION SCORE DISTRIBUTION
# ================================================================

print("=" * 70)
print("WEEK 5 — RECOMMENDATION SCORE DISTRIBUTION REVIEW")
print("=" * 70)

# ------------------------------------------------
# Overall score distribution
# ------------------------------------------------

print("\nOverall recommendation percentage summary:")

display(
    candidate_occupation_scores[
        "recommendation_percentage"
    ].describe()
)

# ------------------------------------------------
# Score range for each candidate
# ------------------------------------------------

candidate_score_range = (
    candidate_occupation_scores
    .groupby("candidate_id")
    .agg(
        best_score=(
            "recommendation_percentage",
            "max"
        ),
        lowest_score=(
            "recommendation_percentage",
            "min"
        )
    )
    .reset_index()
)

candidate_score_range["score_range"] = (
    candidate_score_range["best_score"]
    - candidate_score_range["lowest_score"]
)

print("\nCandidate score range summary:")

display(
    candidate_score_range[
        "score_range"
    ].describe()
)

# ------------------------------------------------
# Review best and second-best score separation
# ------------------------------------------------

top_two_scores = (
    candidate_occupation_scores
    .sort_values(
        [
            "candidate_id",
            "recommendation_score"
        ],
        ascending=[True, False]
    )
    .groupby("candidate_id")
    .head(2)
    .copy()
)

top_two_scores["position"] = (
    top_two_scores
    .groupby("candidate_id")
    .cumcount()
    + 1
)

top_two_pivot = (
    top_two_scores
    .pivot(
        index="candidate_id",
        columns="position",
        values="recommendation_percentage"
    )
    .reset_index()
    .rename(
        columns={
            1: "top_1_score",
            2: "top_2_score"
        }
    )
)

top_two_pivot["top_score_gap"] = (
    top_two_pivot["top_1_score"]
    - top_two_pivot["top_2_score"]
)

print("\nTop 1 vs Top 2 score gap summary:")

display(
    top_two_pivot[
        "top_score_gap"
    ].describe()
)

print("\nCandidates with the largest Top 1 vs Top 2 separation:")

display(
    top_two_pivot
    .sort_values(
        "top_score_gap",
        ascending=False
    )
    .head(15)
)

print("\nCandidates with the smallest Top 1 vs Top 2 separation:")

display(
    top_two_pivot
    .sort_values(
        "top_score_gap",
        ascending=True
    )
    .head(15)
)

WEEK 5 — RECOMMENDATION SCORE DISTRIBUTION REVIEW

Overall recommendation percentage summary:


count    945.000000
mean      73.194986
std       17.931348
min       20.161290
25%       59.253499
50%       73.521851
75%       88.118812
max      100.000000
Name: recommendation_percentage, dtype: float64


Candidate score range summary:


count    6.300000e+01
mean     5.002727e+00
std      2.296794e+00
min      4.263256e-14
25%      4.649469e+00
50%      4.744771e+00
75%      6.003867e+00
max      9.012756e+00
Name: score_range, dtype: float64


Top 1 vs Top 2 score gap summary:


count    63.000000
mean      0.840049
std       0.715777
min       0.000000
25%       0.348109
50%       0.684786
75%       1.382511
max       2.471222
Name: top_score_gap, dtype: float64


Candidates with the largest Top 1 vs Top 2 separation:


position,candidate_id,top_1_score,top_2_score,top_score_gap
8,Candidate_009,25.647059,23.175837,2.471222
13,Candidate_014,59.029563,56.800446,2.229117
59,Candidate_060,59.029563,56.800446,2.229117
41,Candidate_042,59.029563,56.800446,2.229117
39,Candidate_040,59.029563,56.800446,2.229117
0,Candidate_001,69.473008,67.395568,2.077439
58,Candidate_059,69.473008,67.395568,2.077439
37,Candidate_038,69.473008,67.395568,2.077439
24,Candidate_025,69.473008,67.395568,2.077439
57,Candidate_058,79.428571,77.697842,1.730730



Candidates with the smallest Top 1 vs Top 2 separation:


position,candidate_id,top_1_score,top_2_score,top_score_gap
31,Candidate_032,100.000000,100.000000,0.000000
28,Candidate_029,100.000000,100.000000,0.000000
17,Candidate_018,100.000000,100.000000,0.000000
14,Candidate_015,100.000000,100.000000,0.000000
60,Candidate_061,100.000000,100.000000,0.000000
6,Candidate_007,100.000000,100.000000,0.000000
38,Candidate_039,100.000000,100.000000,0.000000
50,Candidate_051,45.411765,45.404884,0.006880
30,Candidate_031,86.151261,86.139986,0.011274
16,Candidate_017,86.151261,86.139986,0.011274


In [366]:
# ================================================================
# CELL 34 — LOAD WEEK 4 CIP-INTEGRATED OCCUPATION PROFILE
# ================================================================

import pandas as pd
import os

# ------------------------------------------------
# Path to Week 4 CIP-integrated occupation profile
# ------------------------------------------------

cip_file_path = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Outputs\Tables\onet_cip_integrated_occupation_profile.csv"
)

# ------------------------------------------------
# Validate file
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — LOAD WEEK 4 CIP-INTEGRATED OCCUPATION PROFILE")
print("=" * 70)

print("\nFile path:")
print(cip_file_path)

print("\nFile exists:")
print(os.path.exists(cip_file_path))

# ------------------------------------------------
# Load dataset
# ------------------------------------------------

if os.path.exists(cip_file_path):

    cip_occupation_profiles = pd.read_csv(
        cip_file_path
    )

    print("\nShape:")
    print(cip_occupation_profiles.shape)

    print("\nColumns:")
    print(
        cip_occupation_profiles.columns.tolist()
    )

    print("\nUnique selected occupations:")

    if "selected_occupation" in cip_occupation_profiles.columns:

        print(
            cip_occupation_profiles[
                "selected_occupation"
            ].nunique()
        )

        print("\nOccupation names:")

        print(
            sorted(
                cip_occupation_profiles[
                    "selected_occupation"
                ]
                .dropna()
                .unique()
            )
        )

    print("\nFirst 10 rows:")

    display(
        cip_occupation_profiles.head(10)
    )

else:

    print(
        "\nERROR: CIP-integrated occupation file was not found."
    )

WEEK 5 — LOAD WEEK 4 CIP-INTEGRATED OCCUPATION PROFILE

File path:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_cip_integrated_occupation_profile.csv

File exists:
True

Shape:
(20, 12)

Columns:
['selected_occupation', 'onet_soc_code', 'onet_title', 'skill_count', 'knowledge_count', 'ability_count', 'task_count', 'education_record_count', 'training_record_count', 'job_zone', 'cip_program_count', 'cip_soc_mapping_count']

Unique selected occupations:
15

Occupation names:
['Administrative Assistant', 'Bookkeeper', 'Continuing Care Assistant', 'Delivery Driver', 'Driver, Truck', 'Food Service Supervisor', 'Information Technology (IT) Analyst', 'Inside Sales Representative', 'Licensed Practical Nurse (L.P.N.)', 'Office Administrator', 'Office Manager', 'Restaurant Manager', 'Retail Sales Associate', 'Secondary School Teacher', 'Software Developer']

First 10 rows:


,selected_occupation,onet_soc_code,onet_title,skill_count,knowledge_count,ability_count,task_count,education_record_count,training_record_count,job_zone,cip_program_count,cip_soc_mapping_count
0,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",10,33,52,31,12,29,2,2,2
1,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",10,33,52,28,12,29,3,1,1
2,Continuing Care Assistant,31-1131.00,Nursing Assistants,10,33,52,33,12,29,3,3,3
3,Continuing Care Assistant,31-1132.00,Orderlies,10,33,52,22,12,29,2,1,1
4,Delivery Driver,53-3033.00,Light Truck Drivers,10,33,52,13,12,29,2,1,1
5,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,10,33,52,29,12,29,2,1,1
6,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,10,33,52,26,12,29,2,5,5
7,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,10,33,52,22,12,29,4,4,4
8,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,10,33,52,17,12,29,5,4,4
9,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,10,33,52,11,12,29,4,9,9


In [367]:
# ================================================================
# CELL 35 — INSPECT OCCUPATION-LEVEL CIP INTEGRATION
# ================================================================

print("=" * 70)
print("WEEK 5 — CIP INTEGRATION STRUCTURE REVIEW")
print("=" * 70)

# ------------------------------------------------
# Count O*NET records per selected occupation
# ------------------------------------------------

occupation_cip_structure = (
    cip_occupation_profiles
    .groupby("selected_occupation")
    .agg(
        onet_record_count=(
            "onet_soc_code",
            "nunique"
        ),
        cip_program_count_min=(
            "cip_program_count",
            "min"
        ),
        cip_program_count_max=(
            "cip_program_count",
            "max"
        ),
        cip_mapping_count_min=(
            "cip_soc_mapping_count",
            "min"
        ),
        cip_mapping_count_max=(
            "cip_soc_mapping_count",
            "max"
        )
    )
    .reset_index()
)

print("\nOccupation-level CIP structure:")

display(
    occupation_cip_structure
)

print("\nOccupations with multiple O*NET records:")

multiple_onet_occupations = (
    occupation_cip_structure[
        occupation_cip_structure[
            "onet_record_count"
        ] > 1
    ]
)

display(
    multiple_onet_occupations
)

print("\nTotal selected occupations:")
print(
    occupation_cip_structure[
        "selected_occupation"
    ].nunique()
)

print("\nTotal O*NET occupation records:")
print(
    len(cip_occupation_profiles)
)

WEEK 5 — CIP INTEGRATION STRUCTURE REVIEW

Occupation-level CIP structure:


,selected_occupation,onet_record_count,cip_program_count_min,cip_program_count_max,cip_mapping_count_min,cip_mapping_count_max
0,Administrative Assistant,1,2,2,2,2
1,Bookkeeper,1,1,1,1,1
2,Continuing Care Assistant,2,1,3,1,3
3,Delivery Driver,1,1,1,1,1
4,"Driver, Truck",1,1,1,1,1
5,Food Service Supervisor,1,5,5,5,5
6,Information Technology (IT) Analyst,4,4,15,4,15
7,Inside Sales Representative,1,1,1,2,2
8,Licensed Practical Nurse (L.P.N.),1,2,2,2,2
9,Office Administrator,2,2,13,2,26



Occupations with multiple O*NET records:


,selected_occupation,onet_record_count,cip_program_count_min,cip_program_count_max,cip_mapping_count_min,cip_mapping_count_max
2,Continuing Care Assistant,2,1,3,1,3
6,Information Technology (IT) Analyst,4,4,15,4,15
9,Office Administrator,2,2,13,2,26



Total selected occupations:
15

Total O*NET occupation records:
20


In [368]:
# ================================================================
# CELL 36 — LOCATE DETAILED CIP MAPPING FILES
# ================================================================

import os
import glob
import pandas as pd

# ------------------------------------------------
# Search the Capstone Project folder
# ------------------------------------------------

project_path = r"C:\Users\Admin\Capstone_Project"

# Find all CSV files recursively
all_csv_files = glob.glob(
    os.path.join(
        project_path,
        "**",
        "*.csv"
    ),
    recursive=True
)

# ------------------------------------------------
# Identify files potentially related to
# CIP, SOC, occupation mappings, or education
# ------------------------------------------------

search_terms = [
    "cip",
    "soc",
    "crosswalk",
    "education",
    "occupation"
]

potential_files = []

for file_path in all_csv_files:

    filename = os.path.basename(
        file_path
    ).lower()

    if any(
        term in filename
        for term in search_terms
    ):

        potential_files.append(
            file_path
        )

# ------------------------------------------------
# Display results
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — LOCATE DETAILED CIP MAPPING FILES")
print("=" * 70)

print("\nTotal CSV files found:")
print(len(all_csv_files))

print("\nPotential CIP / SOC / education mapping files:")
print(len(potential_files))

for i, file_path in enumerate(
    sorted(potential_files),
    start=1
):

    print(f"\n{i}. {file_path}")

WEEK 5 — LOCATE DETAILED CIP MAPPING FILES

Total CSV files found:
69

Potential CIP / SOC / education mapping files:
13

1. C:\Users\Admin\Capstone_Project\Data\Crosswalks\NOC2016_to_SOC2018\noc2016v1_3-soc2018us-eng.csv

2. C:\Users\Admin\Capstone_Project\Data\Raw_Data\CIPCode2020.csv

3. C:\Users\Admin\Capstone_Project\Data\Raw_Data\db_onet_csv\education.csv

4. C:\Users\Admin\Capstone_Project\Data\Raw_Data\db_onet_csv\education_categories.csv

5. C:\Users\Admin\Capstone_Project\Data\Raw_Data\db_onet_csv\interests_illustrative_occupations.csv

6. C:\Users\Admin\Capstone_Project\Data\Raw_Data\db_onet_csv\occupation_data.csv

7. C:\Users\Admin\Capstone_Project\Data\Raw_Data\db_onet_csv\occupation_level_metadata.csv

8. C:\Users\Admin\Capstone_Project\Data\Raw_Data\db_onet_csv\related_occupations.csv

9. C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_cip_integrated_occupation_profile.csv

10. C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_education_profile.csv

11. C:\Users\A

In [369]:
# ================================================================
# CELL 37 — INSPECT CIP 2020 DATASET
# ================================================================

import pandas as pd
import os

# ------------------------------------------------
# Path to CIP 2020 dataset
# ------------------------------------------------

cip_raw_file_path = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Data\Raw_Data\CIPCode2020.csv"
)

print("=" * 70)
print("WEEK 5 — INSPECT CIP 2020 DATASET")
print("=" * 70)

print("\nFile path:")
print(cip_raw_file_path)

print("\nFile exists:")
print(os.path.exists(cip_raw_file_path))

# ------------------------------------------------
# Load CIP dataset
# ------------------------------------------------

cip_raw = pd.read_csv(
    cip_raw_file_path
)

print("\nShape:")
print(cip_raw.shape)

print("\nColumns:")
print(cip_raw.columns.tolist())

print("\nFirst 10 rows:")

display(
    cip_raw.head(10)
)

print("\nData types:")

display(
    cip_raw.dtypes
)

WEEK 5 — INSPECT CIP 2020 DATASET

File path:
C:\Users\Admin\Capstone_Project\Data\Raw_Data\CIPCode2020.csv

File exists:
True

Shape:
(2848, 8)

Columns:
['CIPFamily', 'CIPCode', 'Action', 'TextChange', 'CIPTitle', 'CIPDefinition', 'CrossReferences', 'Examples']

First 10 rows:


,CIPFamily,CIPCode,Action,TextChange,CIPTitle,CIPDefinition,CrossReferences,Examples
0,"=""01""","=""01""",No substantive changes,yes,AGRICULTURAL/ANIMAL/PLANT/VETERINARY SCIENCE AND RELATED FIELDS.,"Instructional programs that focus on agriculture, animal, plant, veterinary, and related science...",NaN,NaN
1,"=""01""","=""01.00""",No substantive changes,no,"Agriculture, General.",Instructional content is defined in code 01.0000.,NaN,NaN
2,"=""01""","=""01.0000""",No substantive changes,no,"Agriculture, General.",A program that focuses on the general principles and practice of agricultural research and produ...,14.0301 - Agricultural Engineering.,NaN
3,"=""01""","=""01.01""",No substantive changes,no,Agricultural Business and Management.,Instructional content for this group of programs is defined in codes 01.0101 - 01.0199.,NaN,NaN
4,"=""01""","=""01.0101""",No substantive changes,no,"Agricultural Business and Management, General.",A general program that focuses on modern business and economic principles involved in the organ...,NaN,NaN
5,"=""01""","=""01.0102""",No substantive changes,no,Agribusiness/Agricultural Business Operations.,A program that prepares individuals to manage agricultural businesses and agriculturally related...,NaN,NaN
6,"=""01""","=""01.0103""",No substantive changes,no,Agricultural Economics.,"A program that focuses on the application of economics to the analysis of resource allocation, p...",03.0204 - Environmental/Natural Resource Economics.,Examples: - Agroeconomics
7,"=""01""","=""01.0104""",No substantive changes,no,Farm/Farm and Ranch Management.,"A program that prepares individuals to manage farms, ranches, and similar enterprises. Includes...",NaN,NaN
8,"=""01""","=""01.0105""",No substantive changes,no,Agricultural/Farm Supplies Retailing and Wholesaling.,"A program that prepares individuals to sell agricultural products and supplies, provide support...","52.1803 - Retailing and Retail Operations., 52.1901 - Auctioneering., 52.0202 - Purchasing, Proc...",NaN
9,"=""01""","=""01.0106""",No substantive changes,yes,Agricultural Business Technology/Technician.,A program that prepares individuals to perform specialized support functions related to agricult...,NaN,NaN



Data types:


CIPFamily          object
CIPCode            object
Action             object
TextChange         object
CIPTitle           object
CIPDefinition      object
CrossReferences    object
Examples           object
dtype: object

In [370]:
# ================================================================
# CELL 38 — SEARCH PROJECT FILES FOR CIP MAPPING DATA
# ================================================================

import os
import glob
import pandas as pd

print("=" * 70)
print("WEEK 5 — SEARCH FOR DETAILED CIP MAPPING DATA")
print("=" * 70)

project_path = r"C:\Users\Admin\Capstone_Project"

# ------------------------------------------------
# Find CSV and Excel files
# ------------------------------------------------

csv_files = glob.glob(
    os.path.join(project_path, "**", "*.csv"),
    recursive=True
)

xlsx_files = glob.glob(
    os.path.join(project_path, "**", "*.xlsx"),
    recursive=True
)

all_files = csv_files + xlsx_files

mapping_candidates = []

# ------------------------------------------------
# Inspect columns
# ------------------------------------------------

for file_path in all_files:

    try:

        if file_path.lower().endswith(".csv"):

            sample = pd.read_csv(
                file_path,
                nrows=5
            )

            sheets_to_check = {
                "CSV": sample.columns.tolist()
            }

        else:

            excel_file = pd.ExcelFile(file_path)

            sheets_to_check = {}

            for sheet_name in excel_file.sheet_names:

                sample = pd.read_excel(
                    file_path,
                    sheet_name=sheet_name,
                    nrows=5
                )

                sheets_to_check[sheet_name] = (
                    sample.columns.tolist()
                )

        # ----------------------------------------
        # Check each CSV / Excel sheet
        # ----------------------------------------

        for source_name, columns in sheets_to_check.items():

            columns_lower = [
                str(column).lower()
                for column in columns
            ]

            has_cip = any(
                "cip" in column
                for column in columns_lower
            )

            has_soc_or_onet = any(
                (
                    "soc" in column
                    or "onet" in column
                    or "occupation" in column
                )
                for column in columns_lower
            )

            if has_cip and has_soc_or_onet:

                mapping_candidates.append(
                    {
                        "file_path": file_path,
                        "sheet": source_name,
                        "columns": columns
                    }
                )

    except Exception:
        pass

# ------------------------------------------------
# Display results
# ------------------------------------------------

print("\nPotential detailed CIP mapping sources found:")
print(len(mapping_candidates))

for i, item in enumerate(
    mapping_candidates,
    start=1
):

    print("\n" + "=" * 70)
    print(f"MAPPING SOURCE {i}")

    print("\nFile:")
    print(item["file_path"])

    print("\nSheet:")
    print(item["sheet"])

    print("\nColumns:")
    print(item["columns"])

WEEK 5 — SEARCH FOR DETAILED CIP MAPPING DATA

Potential detailed CIP mapping sources found:
8

MAPPING SOURCE 1

File:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_cip_integrated_occupation_profile.csv

Sheet:
CSV

Columns:
['selected_occupation', 'onet_soc_code', 'onet_title', 'skill_count', 'knowledge_count', 'ability_count', 'task_count', 'education_record_count', 'training_record_count', 'job_zone', 'cip_program_count', 'cip_soc_mapping_count']

MAPPING SOURCE 2

File:
C:\Users\Admin\Capstone_Project\Data\Crosswalks\CIP_to_SOC\CIP2020_SOC2018_Crosswalk.xlsx

Sheet:
CIP-SOC

Columns:
['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']

MAPPING SOURCE 3

File:
C:\Users\Admin\Capstone_Project\Data\Crosswalks\CIP_to_SOC\CIP2020_SOC2018_Crosswalk.xlsx

Sheet:
SOC-CIP

Columns:
['SOC2018Code', 'SOC2018Title', 'CIP2020Code', 'CIP2020Title']

MAPPING SOURCE 4

File:
C:\Users\Admin\Capstone_Project\Data\Crosswalks\CIP_to_SOC\CIP2020_SOC2018_Crosswalk.xlsx

Sheet:
New CIP


In [371]:
# ================================================================
# CELL 39 — LOAD CIP 2020 TO SOC 2018 CROSSWALK
# ================================================================

cip_soc_path = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Data\Crosswalks\CIP_to_SOC"
    r"\CIP2020_SOC2018_Crosswalk.xlsx"
)

# ------------------------------------------------
# Load the primary CIP-to-SOC mapping sheet
# ------------------------------------------------

cip_soc_crosswalk = pd.read_excel(
    cip_soc_path,
    sheet_name="CIP-SOC"
)

print("=" * 70)
print("WEEK 5 — CIP 2020 TO SOC 2018 CROSSWALK")
print("=" * 70)

print("\nFile path:")
print(cip_soc_path)

print("\nFile exists:")
print(os.path.exists(cip_soc_path))

print("\nShape:")
print(cip_soc_crosswalk.shape)

print("\nColumns:")
print(
    cip_soc_crosswalk.columns.tolist()
)

print("\nUnique CIP programs:")
print(
    cip_soc_crosswalk[
        "CIP2020Code"
    ].nunique()
)

print("\nUnique SOC occupations:")
print(
    cip_soc_crosswalk[
        "SOC2018Code"
    ].nunique()
)

print("\nFirst 20 mappings:")

display(
    cip_soc_crosswalk.head(20)
)

print("\nData types:")

print(
    cip_soc_crosswalk.dtypes
)

WEEK 5 — CIP 2020 TO SOC 2018 CROSSWALK

File path:
C:\Users\Admin\Capstone_Project\Data\Crosswalks\CIP_to_SOC\CIP2020_SOC2018_Crosswalk.xlsx

File exists:
True

Shape:
(6097, 4)

Columns:
['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']

Unique CIP programs:
2143

Unique SOC occupations:
868

First 20 mappings:


,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,1.0000,"Agriculture, General.",19-1011,Animal Scientists
1,1.0000,"Agriculture, General.",19-1012,Food Scientists and Technologists
2,1.0000,"Agriculture, General.",19-1013,Soil and Plant Scientists
3,1.0000,"Agriculture, General.",19-4012,Agricultural Technicians
4,1.0000,"Agriculture, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"
5,1.0101,"Agricultural Business and Management, General.",11-9013,"Farmers, Ranchers, and Other Agricultural Managers"
6,1.0101,"Agricultural Business and Management, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"
7,1.0101,"Agricultural Business and Management, General.",45-1011,"First-Line Supervisors of Farming, Fishing, and Forestry Workers"
8,1.0102,Agribusiness/Agricultural Business Operations.,11-9013,"Farmers, Ranchers, and Other Agricultural Managers"
9,1.0102,Agribusiness/Agricultural Business Operations.,25-1041,"Agricultural Sciences Teachers, Postsecondary"



Data types:
CIP2020Code     float64
CIP2020Title     object
SOC2018Code      object
SOC2018Title     object
dtype: object


In [372]:
# ================================================================
# CELL 40 — LOAD VALIDATED OCCUPATION-TO-O*NET MAPPING
# ================================================================

validated_mapping_path = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Outputs\Tables\validated_occupation_onet_mapping.csv"
)

validated_occupation_mapping = pd.read_csv(
    validated_mapping_path
)

print("=" * 70)
print("WEEK 5 — VALIDATED OCCUPATION MAPPING REVIEW")
print("=" * 70)

print("\nFile path:")
print(validated_mapping_path)

print("\nFile exists:")
print(os.path.exists(validated_mapping_path))

print("\nShape:")
print(validated_occupation_mapping.shape)

print("\nColumns:")
print(
    validated_occupation_mapping.columns.tolist()
)

print("\nFirst 20 rows:")

display(
    validated_occupation_mapping.head(20)
)

print("\nUnique selected occupations:")

print(
    validated_occupation_mapping[
        "selected_occupation"
    ].nunique()
)

print("\nUnique SOC codes:")

# Identify and display columns containing SOC
soc_columns = [
    column
    for column in validated_occupation_mapping.columns
    if "soc" in column.lower()
]

print("\nSOC-related columns:")
print(soc_columns)

for column in soc_columns:

    print(f"\n{column}:")

    print(
        validated_occupation_mapping[
            column
        ]
        .dropna()
        .astype(str)
        .unique()[:20]
    )

WEEK 5 — VALIDATED OCCUPATION MAPPING REVIEW

File path:
C:\Users\Admin\Capstone_Project\Outputs\Tables\validated_occupation_onet_mapping.csv

File exists:
True

Shape:
(20, 16)

Columns:
['selected_occupation', 'noc_2021_code', 'noc21_code', 'noc2016_code', 'mapping_note', 'noc2016_match_code', 'NOC 2016 Version 1.3 Code', 'NOC 2016 Version 1.3 Title', 'Partial', 'SOC 2018 (US) Code', 'SOC 2018 (US) Title', 'Explanatory Notes', 'O*NET-SOC 2019 Code', 'O*NET-SOC 2019 Title', '2018 SOC Code', '2018 SOC Title']

First 20 rows:


,selected_occupation,noc_2021_code,noc21_code,noc2016_code,mapping_note,noc2016_match_code,NOC 2016 Version 1.3 Code,NOC 2016 Version 1.3 Title,Partial,SOC 2018 (US) Code,SOC 2018 (US) Title,Explanatory Notes,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,Software Developer,21232,21232,2174,Breakdown mapping,2174,2174,Computer programmers and interactive media developers,*,15-1252,Software Developers,Only software developers in interactive media,15-1252.00,Software Developers,15-1252,Software Developers
1,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1211,Computer Systems Analysts,"Only computer systems analysts, excluding analysts and testers in quality assurance and systems ...",15-1211.00,Computer Systems Analysts,15-1211,Computer Systems Analysts
2,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1211,Computer Systems Analysts,"Only computer systems analysts, excluding analysts and testers in quality assurance and systems ...",15-1211.01,Health Informatics Specialists,15-1211,Computer Systems Analysts
3,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1212,Information Security Analysts,"Only analysts, consultants and related specialists in information security",15-1212.00,Information Security Analysts,15-1212,Information Security Analysts
4,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping,2171,2171,Information systems analysts and consultants,*,15-1253,Software Quality Assurance Analysts and Testers,"Only analysts and testers in quality assurance, including systems auditors",15-1253.00,Software Quality Assurance Analysts and Testers,15-1253,Software Quality Assurance Analysts and Testers
5,Administrative Assistant,13110,13110,1241,Selected relevant mapping; excluded HR manager transfer,1241,1241,Administrative assistants,NaN,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",NaN,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
6,Bookkeeper,12200,12200,1311,Direct code/name change,1311,1311,Accounting technicians and bookkeepers,NaN,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",NaN,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",43-3031,"Bookkeeping, Accounting, and Auditing Clerks"
7,Office Administrator,13100,13100,1221,Direct code/name change,1221,1221,Administrative officers,*,43-1011,First-Line Supervisors of Office and Administrative Support Workers,"Only chief invigilators and supervisors of invigilators and exam proctors, non-post-secondary",43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,43-1011,First-Line Supervisors of Office and Administrative Support Workers
8,Office Administrator,13100,13100,1221,Direct code/name change,1221,1221,Administrative officers,*,43-6011,Executive Secretaries and Executive Administrative Assistants,Only executive secretaries,43-6011.00,Executive Secretaries and Executive Administrative Assistants,43-6011,Executive Secretaries and Executive Administrative Assistants
9,Office Manager,13100,13100,1221,Direct code/name change,1221,1221,Administrative officers,*,43-1011,First-Line Supervisors of Office and Administrative Support Workers,"Only chief invigilators and supervisors of invigilators and exam proctors, non-post-secondary",43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,43-1011,First-Line Supervisors of Office and Administrative Support Workers



Unique selected occupations:
15

Unique SOC codes:

SOC-related columns:
['SOC 2018 (US) Code', 'SOC 2018 (US) Title', 'O*NET-SOC 2019 Code', 'O*NET-SOC 2019 Title', '2018 SOC Code', '2018 SOC Title']

SOC 2018 (US) Code:
['15-1252' '15-1211' '15-1212' '15-1253' '43-6014' '43-3031' '43-1011'
 '43-6011' '11-9051' '35-1012' '41-2031' '53-3032' '53-3033' '25-2031'
 '31-1131' '31-1132' '29-2061']

SOC 2018 (US) Title:
['Software Developers' 'Computer Systems Analysts'
 'Information Security Analysts'
 'Software Quality Assurance Analysts and Testers'
 'Secretaries and Administrative Assistants, Except Legal, Medical, and Executive'
 'Bookkeeping, Accounting, and Auditing Clerks'
 'First-Line Supervisors of Office and Administrative Support Workers'
 'Executive Secretaries and Executive Administrative Assistants'
 'Food Service Managers'
 'First-Line Supervisors of Food Preparation and Serving Workers'
 'Retail Salespersons' 'Heavy and Tractor-Trailer Truck Drivers'
 'Light Truck Drivers'


In [373]:

# ================================================================
# CELL 41 — CREATE DETAILED OCCUPATION-TO-CIP MAPPING
# ================================================================

# Keep the occupation and SOC information needed for the CIP merge
occupation_soc_mapping = (
    validated_occupation_mapping[
        [
            "selected_occupation",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Rename columns to match the CIP crosswalk structure
occupation_soc_mapping = (
    occupation_soc_mapping
    .rename(
        columns={
            "SOC 2018 (US) Code": "SOC2018Code",
            "SOC 2018 (US) Title": "SOC2018Title"
        }
    )
)

# Merge occupation mappings with the CIP 2020 → SOC 2018 crosswalk
occupation_cip_mapping = (
    occupation_soc_mapping
    .merge(
        cip_soc_crosswalk[
            [
                "CIP2020Code",
                "CIP2020Title",
                "SOC2018Code",
                "SOC2018Title"
            ]
        ],
        on="SOC2018Code",
        how="left",
        suffixes=(
            "_occupation",
            "_cip"
        )
    )
)

# Sort results
occupation_cip_mapping = (
    occupation_cip_mapping
    .sort_values(
        [
            "selected_occupation",
            "CIP2020Code"
        ]
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("WEEK 5 — DETAILED OCCUPATION-TO-CIP MAPPING")
print("=" * 70)

print("\nShape:")
print(occupation_cip_mapping.shape)

print("\nUnique selected occupations:")
print(
    occupation_cip_mapping[
        "selected_occupation"
    ].nunique()
)

print("\nUnique CIP programs:")
print(
    occupation_cip_mapping[
        "CIP2020Code"
    ].nunique()
)

print("\nMissing CIP mappings:")
print(
    occupation_cip_mapping[
        "CIP2020Code"
    ].isna().sum()
)

print("\nFirst 20 occupation-to-CIP mappings:")

display(
    occupation_cip_mapping.head(20)
)


WEEK 5 — DETAILED OCCUPATION-TO-CIP MAPPING

Shape:
(196, 6)

Unique selected occupations:
15

Unique CIP programs:
156

Missing CIP mappings:
0

First 20 occupation-to-CIP mappings:


,selected_occupation,SOC2018Code,SOC2018Title_occupation,CIP2020Code,CIP2020Title,SOC2018Title_cip
0,Administrative Assistant,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",52.0401,"Administrative Assistant and Secretarial Science, General.","Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
1,Administrative Assistant,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",52.0402,Executive Assistant/Executive Secretary.,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
2,Bookkeeper,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",52.0302,Accounting Technology/Technician and Bookkeeping.,"Bookkeeping, Accounting, and Auditing Clerks"
3,Continuing Care Assistant,31-1131,Nursing Assistants,51.2601,Health Aide.,Nursing Assistants
4,Continuing Care Assistant,31-1131,Nursing Assistants,51.3902,Nursing Assistant/Aide and Patient Care Assistant/Aide.,Nursing Assistants
5,Continuing Care Assistant,31-1131,Nursing Assistants,51.3999,"Practical Nursing, Vocational Nursing and Nursing Assistants, Other.",Nursing Assistants
6,Continuing Care Assistant,31-1132,Orderlies,99.9999,NO MATCH,Orderlies
7,Delivery Driver,53-3033,Light Truck Drivers,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.,Light Truck Drivers
8,"Driver, Truck",53-3032,Heavy and Tractor-Trailer Truck Drivers,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.,Heavy and Tractor-Trailer Truck Drivers
9,Food Service Supervisor,35-1012,First-Line Supervisors of Food Preparation and Serving Workers,12.0500,"Cooking and Related Culinary Arts, General.",First-Line Supervisors of Food Preparation and Serving Workers


In [374]:

# ================================================================
# CELL 42 — CLEAN AND VALIDATE OCCUPATION-TO-CIP MAPPING
# ================================================================

# Create a clean copy
occupation_cip_mapping_clean = (
    occupation_cip_mapping
    .copy()
)

# ------------------------------------------------
# Remove placeholder "NO MATCH" CIP records
# ------------------------------------------------

no_match_count = (
    occupation_cip_mapping_clean[
        "CIP2020Title"
    ]
    .eq("NO MATCH")
    .sum()
)

occupation_cip_mapping_clean = (
    occupation_cip_mapping_clean[
        occupation_cip_mapping_clean[
            "CIP2020Title"
        ]
        .ne("NO MATCH")
    ]
    .copy()
)

# ------------------------------------------------
# Remove duplicate occupation-CIP combinations
#
# A CIP program may connect to multiple SOC codes
# for the same selected occupation.
# Keep one occupation-to-CIP relationship.
# ------------------------------------------------

records_before_duplicates = len(
    occupation_cip_mapping_clean
)

occupation_cip_mapping_clean = (
    occupation_cip_mapping_clean
    .drop_duplicates(
        subset=[
            "selected_occupation",
            "CIP2020Code"
        ]
    )
    .sort_values(
        [
            "selected_occupation",
            "CIP2020Code"
        ]
    )
    .reset_index(drop=True)
)

records_after_duplicates = len(
    occupation_cip_mapping_clean
)

# ------------------------------------------------
# Occupation-level validation summary
# ------------------------------------------------

cip_validation = (
    occupation_cip_mapping_clean
    .groupby("selected_occupation")
    .agg(
        unique_cip_programs=(
            "CIP2020Code",
            "nunique"
        ),
        mapping_records=(
            "CIP2020Code",
            "size"
        )
    )
    .reset_index()
    .sort_values(
        "unique_cip_programs",
        ascending=False
    )
)

# ------------------------------------------------
# Identify occupations with no CIP programs
# ------------------------------------------------

all_occupations = (
    validated_occupation_mapping[
        "selected_occupation"
    ]
    .drop_duplicates()
)

occupations_with_cip = set(
    occupation_cip_mapping_clean[
        "selected_occupation"
    ]
)

occupations_without_cip = [
    occupation
    for occupation in all_occupations
    if occupation not in occupations_with_cip
]

# ------------------------------------------------
# Output
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — CLEANED OCCUPATION-TO-CIP MAPPING VALIDATION")
print("=" * 70)

print("\nRecords before cleaning:")
print(len(occupation_cip_mapping))

print("\n'NO MATCH' records removed:")
print(no_match_count)

print("\nRecords after removing 'NO MATCH':")
print(records_before_duplicates)

print("\nDuplicate occupation-CIP records removed:")
print(
    records_before_duplicates
    - records_after_duplicates
)

print("\nFinal mapping records:")
print(records_after_duplicates)

print("\nUnique selected occupations:")
print(
    occupation_cip_mapping_clean[
        "selected_occupation"
    ].nunique()
)

print("\nUnique CIP programs:")
print(
    occupation_cip_mapping_clean[
        "CIP2020Code"
    ].nunique()
)

print("\nOccupations without CIP programs:")
print(occupations_without_cip)

print("\nOccupation-level CIP validation:")

display(cip_validation)

print("\nFirst 20 cleaned mappings:")

display(
    occupation_cip_mapping_clean
    .head(20)
)

WEEK 5 — CLEANED OCCUPATION-TO-CIP MAPPING VALIDATION

Records before cleaning:
196

'NO MATCH' records removed:
3

Records after removing 'NO MATCH':
193

Duplicate occupation-CIP records removed:
8

Final mapping records:
185

Unique selected occupations:
13

Unique CIP programs:
155

Occupations without CIP programs:
['Retail Sales Associate', 'Inside Sales Representative']

Occupation-level CIP validation:


,selected_occupation,unique_cip_programs,mapping_records
11,Secondary School Teacher,92,92
6,Information Technology (IT) Analyst,22,22
12,Software Developer,20,20
8,Office Administrator,13,13
9,Office Manager,13,13
10,Restaurant Manager,10,10
5,Food Service Supervisor,5,5
2,Continuing Care Assistant,3,3
0,Administrative Assistant,2,2
7,Licensed Practical Nurse (L.P.N.),2,2



First 20 cleaned mappings:


,selected_occupation,SOC2018Code,SOC2018Title_occupation,CIP2020Code,CIP2020Title,SOC2018Title_cip
0,Administrative Assistant,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",52.0401,"Administrative Assistant and Secretarial Science, General.","Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
1,Administrative Assistant,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",52.0402,Executive Assistant/Executive Secretary.,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
2,Bookkeeper,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",52.0302,Accounting Technology/Technician and Bookkeeping.,"Bookkeeping, Accounting, and Auditing Clerks"
3,Continuing Care Assistant,31-1131,Nursing Assistants,51.2601,Health Aide.,Nursing Assistants
4,Continuing Care Assistant,31-1131,Nursing Assistants,51.3902,Nursing Assistant/Aide and Patient Care Assistant/Aide.,Nursing Assistants
5,Continuing Care Assistant,31-1131,Nursing Assistants,51.3999,"Practical Nursing, Vocational Nursing and Nursing Assistants, Other.",Nursing Assistants
6,Delivery Driver,53-3033,Light Truck Drivers,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.,Light Truck Drivers
7,"Driver, Truck",53-3032,Heavy and Tractor-Trailer Truck Drivers,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.,Heavy and Tractor-Trailer Truck Drivers
8,Food Service Supervisor,35-1012,First-Line Supervisors of Food Preparation and Serving Workers,12.0500,"Cooking and Related Culinary Arts, General.",First-Line Supervisors of Food Preparation and Serving Workers
9,Food Service Supervisor,35-1012,First-Line Supervisors of Food Preparation and Serving Workers,12.0503,Culinary Arts/Chef Training.,First-Line Supervisors of Food Preparation and Serving Workers


In [375]:

# ================================================================
# CELL 43 — CREATE COMPLETE OCCUPATION-CIP COVERAGE SUMMARY
# ================================================================

# ------------------------------------------------
# Start with all 15 selected occupations
# ------------------------------------------------

all_selected_occupations = (
    validated_occupation_mapping[
        ["selected_occupation"]
    ]
    .drop_duplicates()
    .sort_values("selected_occupation")
    .reset_index(drop=True)
)

# ------------------------------------------------
# Summarize valid CIP programs for occupations
# ------------------------------------------------

occupation_cip_summary = (
    occupation_cip_mapping_clean
    .groupby("selected_occupation")
    .agg(
        cip_program_count=(
            "CIP2020Code",
            "nunique"
        )
    )
    .reset_index()
)

# ------------------------------------------------
# Merge back to all 15 occupations
# ------------------------------------------------

occupation_cip_coverage = (
    all_selected_occupations
    .merge(
        occupation_cip_summary,
        on="selected_occupation",
        how="left"
    )
)

# Replace missing CIP counts with zero
occupation_cip_coverage[
    "cip_program_count"
] = (
    occupation_cip_coverage[
        "cip_program_count"
    ]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------
# Create CIP availability indicator
# ------------------------------------------------

occupation_cip_coverage[
    "cip_pathway_available"
] = (
    occupation_cip_coverage[
        "cip_program_count"
    ] > 0
)

occupation_cip_coverage[
    "cip_status"
] = np.where(
    occupation_cip_coverage[
        "cip_pathway_available"
    ],
    "CIP pathway available",
    "No CIP pathway in crosswalk"
)

# ------------------------------------------------
# Display results
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — OCCUPATION CIP COVERAGE SUMMARY")
print("=" * 70)

print("\nTotal selected occupations:")
print(
    len(occupation_cip_coverage)
)

print("\nOccupations with CIP pathways:")
print(
    occupation_cip_coverage[
        "cip_pathway_available"
    ].sum()
)

print("\nOccupations without CIP pathways:")
print(
    (
        ~occupation_cip_coverage[
            "cip_pathway_available"
        ]
    ).sum()
)

print("\nOccupation-level CIP coverage:")

display(
    occupation_cip_coverage
    .sort_values(
        [
            "cip_pathway_available",
            "cip_program_count"
        ],
        ascending=[
            True,
            False
        ]
    )
)

print("\nOccupations without CIP pathways:")

display(
    occupation_cip_coverage[
        ~occupation_cip_coverage[
            "cip_pathway_available"
        ]
    ]
)

WEEK 5 — OCCUPATION CIP COVERAGE SUMMARY

Total selected occupations:
15

Occupations with CIP pathways:
13

Occupations without CIP pathways:
2

Occupation-level CIP coverage:


,selected_occupation,cip_program_count,cip_pathway_available,cip_status
7,Inside Sales Representative,0,False,No CIP pathway in crosswalk
12,Retail Sales Associate,0,False,No CIP pathway in crosswalk
13,Secondary School Teacher,92,True,CIP pathway available
6,Information Technology (IT) Analyst,22,True,CIP pathway available
14,Software Developer,20,True,CIP pathway available
9,Office Administrator,13,True,CIP pathway available
10,Office Manager,13,True,CIP pathway available
11,Restaurant Manager,10,True,CIP pathway available
5,Food Service Supervisor,5,True,CIP pathway available
2,Continuing Care Assistant,3,True,CIP pathway available



Occupations without CIP pathways:


,selected_occupation,cip_program_count,cip_pathway_available,cip_status
7,Inside Sales Representative,0,False,No CIP pathway in crosswalk
12,Retail Sales Associate,0,False,No CIP pathway in crosswalk


In [376]:

# ================================================================
# CELL 44 — LINK CANDIDATE RECOMMENDATIONS TO CIP PROGRAMS
# ================================================================

# ------------------------------------------------
# Select the Top 5 occupation recommendations
# for each candidate
# ------------------------------------------------

top_n = 5

top_candidate_recommendations = (
    candidate_occupation_scores
    .sort_values(
        [
            "candidate_id",
            "occupation_rank"
        ]
    )
    .groupby("candidate_id")
    .head(top_n)
    .copy()
)

# ------------------------------------------------
# Attach CIP coverage information
# ------------------------------------------------

candidate_recommendations_with_cip = (
    top_candidate_recommendations
    .merge(
        occupation_cip_coverage,
        on="selected_occupation",
        how="left"
    )
)

# ------------------------------------------------
# Attach detailed CIP programs
# ------------------------------------------------

candidate_cip_recommendations = (
    candidate_recommendations_with_cip
    .merge(
        occupation_cip_mapping_clean[
            [
                "selected_occupation",
                "CIP2020Code",
                "CIP2020Title"
            ]
        ],
        on="selected_occupation",
        how="left"
    )
)

# ------------------------------------------------
# Sort final recommendations
# ------------------------------------------------

candidate_cip_recommendations = (
    candidate_cip_recommendations
    .sort_values(
        [
            "candidate_id",
            "occupation_rank",
            "CIP2020Code"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — CANDIDATE OCCUPATION + CIP RECOMMENDATIONS")
print("=" * 70)

print("\nTop occupations retained per candidate:")
print(top_n)

print("\nNumber of candidates:")
print(
    candidate_cip_recommendations[
        "candidate_id"
    ].nunique()
)

print("\nUnique recommended occupations:")
print(
    candidate_cip_recommendations[
        "selected_occupation"
    ].nunique()
)

print("\nRows in candidate-CIP recommendation table:")
print(
    len(candidate_cip_recommendations)
)

print("\nRecommendations without a CIP pathway:")

no_cip_recommendations = (
    candidate_cip_recommendations[
        candidate_cip_recommendations[
            "cip_pathway_available"
        ] == False
    ]
    [
        [
            "candidate_id",
            "selected_occupation",
            "occupation_rank",
            "recommendation_percentage",
            "cip_status"
        ]
    ]
    .drop_duplicates()
)

print(
    len(no_cip_recommendations)
)

display(
    no_cip_recommendations
    .head(20)
)

print("\nFirst 30 candidate recommendations with CIP pathways:")

display(
    candidate_cip_recommendations[
        [
            "candidate_id",
            "selected_occupation",
            "occupation_rank",
            "recommendation_percentage",
            "CIP2020Code",
            "CIP2020Title",
            "cip_status"
        ]
    ]
    .head(30)
)



WEEK 5 — CANDIDATE OCCUPATION + CIP RECOMMENDATIONS

Top occupations retained per candidate:
5

Number of candidates:
63

Unique recommended occupations:
15

Rows in candidate-CIP recommendation table:
5020

Recommendations without a CIP pathway:
35


,candidate_id,selected_occupation,occupation_rank,recommendation_percentage,cip_status
326,Candidate_003,Inside Sales Representative,5,87.456687,No CIP pathway in crosswalk
645,Candidate_006,Inside Sales Representative,3,89.604990,No CIP pathway in crosswalk
646,Candidate_006,Retail Sales Associate,3,89.604990,No CIP pathway in crosswalk
845,Candidate_010,Inside Sales Representative,4,96.534997,No CIP pathway in crosswalk
846,Candidate_010,Retail Sales Associate,4,96.534997,No CIP pathway in crosswalk
1442,Candidate_016,Inside Sales Representative,3,79.209979,No CIP pathway in crosswalk
1443,Candidate_016,Retail Sales Associate,3,79.209979,No CIP pathway in crosswalk
1468,Candidate_017,Inside Sales Representative,2,86.139986,No CIP pathway in crosswalk
1469,Candidate_017,Retail Sales Associate,2,86.139986,No CIP pathway in crosswalk
1854,Candidate_022,Inside Sales Representative,4,96.534997,No CIP pathway in crosswalk



First 30 candidate recommendations with CIP pathways:


,candidate_id,selected_occupation,occupation_rank,recommendation_percentage,CIP2020Code,CIP2020Title,cip_status
0,Candidate_001,Software Developer,1,69.473008,11.0102,Artificial Intelligence.,CIP pathway available
1,Candidate_001,Software Developer,1,69.473008,11.0103,Information Technology.,CIP pathway available
2,Candidate_001,Software Developer,1,69.473008,11.0104,Informatics.,CIP pathway available
3,Candidate_001,Software Developer,1,69.473008,11.0201,"Computer Programming/Programmer, General.",CIP pathway available
4,Candidate_001,Software Developer,1,69.473008,11.0202,"Computer Programming, Specific Applications.",CIP pathway available
5,Candidate_001,Software Developer,1,69.473008,11.0203,"Computer Programming, Vendor/Product Certification.",CIP pathway available
6,Candidate_001,Software Developer,1,69.473008,11.0204,Computer Game Programming.,CIP pathway available
7,Candidate_001,Software Developer,1,69.473008,11.0205,"Computer Programming, Specific Platforms.",CIP pathway available
8,Candidate_001,Software Developer,1,69.473008,11.0401,Information Science/Studies.,CIP pathway available
9,Candidate_001,Software Developer,1,69.473008,11.0701,Computer Science.,CIP pathway available


In [377]:

# ================================================================
# CELL 45 — AGGREGATE AND RANK CIP RECOMMENDATIONS
# ================================================================

# ------------------------------------------------
# Keep only rows with valid CIP programs
# ------------------------------------------------

valid_candidate_cip = (
    candidate_cip_recommendations[
        candidate_cip_recommendations["CIP2020Code"].notna()
    ]
    .copy()
)

# ------------------------------------------------
# Convert CIP codes to strings
# This preserves formatting for display
# ------------------------------------------------

valid_candidate_cip["CIP2020Code"] = (
    valid_candidate_cip["CIP2020Code"]
    .astype(str)
)

# ------------------------------------------------
# Aggregate occupation recommendation scores
# for each candidate-CIP combination
#
# A CIP program receives support from every
# recommended occupation connected to it.
# ------------------------------------------------

candidate_cip_scores = (
    valid_candidate_cip
    .groupby(
        [
            "candidate_id",
            "CIP2020Code",
            "CIP2020Title"
        ],
        as_index=False
    )
    .agg(
        cip_recommendation_score=(
            "recommendation_percentage",
            "mean"
        ),
        supporting_occupations=(
            "selected_occupation",
            "nunique"
        ),
        best_occupation_rank=(
            "occupation_rank",
            "min"
        ),
        max_occupation_score=(
            "recommendation_percentage",
            "max"
        )
    )
)

# ------------------------------------------------
# Create CIP recommendation percentage
#
# Using the aggregated score directly because
# occupation recommendation scores are already
# expressed on a 0–100 percentage scale.
# ------------------------------------------------

candidate_cip_scores[
    "cip_recommendation_percentage"
] = (
    candidate_cip_scores[
        "cip_recommendation_score"
    ]
)

# ------------------------------------------------
# Rank CIP programs within each candidate
# ------------------------------------------------

candidate_cip_scores[
    "cip_rank"
] = (
    candidate_cip_scores
    .groupby("candidate_id")[
        "cip_recommendation_percentage"
    ]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

# ------------------------------------------------
# Sort results
# ------------------------------------------------

candidate_cip_scores = (
    candidate_cip_scores
    .sort_values(
        [
            "candidate_id",
            "cip_rank",
            "supporting_occupations",
            "best_occupation_rank"
        ],
        ascending=[
            True,
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — AGGREGATED CIP RECOMMENDATION RESULTS")
print("=" * 70)

print("\nNumber of candidates:")
print(
    candidate_cip_scores["candidate_id"].nunique()
)

print("\nUnique CIP programs recommended:")
print(
    candidate_cip_scores["CIP2020Code"].nunique()
)

print("\nTotal candidate-CIP recommendations:")
print(
    len(candidate_cip_scores)
)

print("\nCIP recommendation score summary:")
display(
    candidate_cip_scores[
        "cip_recommendation_percentage"
    ]
    .describe()
)

print("\nTop 20 aggregated CIP recommendations:")

display(
    candidate_cip_scores[
        [
            "candidate_id",
            "cip_rank",
            "CIP2020Code",
            "CIP2020Title",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "best_occupation_rank"
        ]
    ]
    .head(20)
)



WEEK 5 — AGGREGATED CIP RECOMMENDATION RESULTS

Number of candidates:
63

Unique CIP programs recommended:
155

Total candidate-CIP recommendations:
4307

CIP recommendation score summary:


count    4307.000000
mean       65.604447
std        12.272595
min        22.556657
25%        56.767163
50%        64.464883
75%        72.300771
max       100.000000
Name: cip_recommendation_percentage, dtype: float64


Top 20 aggregated CIP recommendations:


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank
0,Candidate_001,1,11.0102,Artificial Intelligence.,69.473008,1,1
1,Candidate_001,1,11.0401,Information Science/Studies.,69.473008,1,1
2,Candidate_001,1,11.0804,"Modeling, Virtual Environments and Simulation.",69.473008,1,1
3,Candidate_001,1,11.0902,Cloud Computing.,69.473008,1,1
4,Candidate_001,1,30.0801,Mathematics and Computer Science.,69.473008,1,1
5,Candidate_001,1,30.1601,Accounting and Computer Science.,69.473008,1,1
6,Candidate_001,1,30.3901,Economics and Computer Science.,69.473008,1,1
7,Candidate_001,1,30.4801,Linguistics and Computer Science.,69.473008,1,1
8,Candidate_001,1,30.7001,"Data Science, General.",69.473008,1,1
9,Candidate_001,2,11.0103,Information Technology.,68.434288,2,1


In [378]:

# ================================================================
# CELL 46 — IMPROVE CIP RECOMMENDATION SCORING AND RANKING
# ================================================================

# ------------------------------------------------
# Start with the aggregated CIP recommendations
# from Cell 45
# ------------------------------------------------

candidate_cip_ranked = (
    candidate_cip_scores
    .copy()
)

# ------------------------------------------------
# Calculate support factor
#
# A CIP supported by more recommended occupations
# receives additional evidence.
#
# Formula:
# 1 + 0.10 × (number of supporting occupations - 1)
#
# Examples:
# 1 occupation  -> 1.00
# 2 occupations -> 1.10
# 3 occupations -> 1.20
# ------------------------------------------------

candidate_cip_ranked[
    "support_factor"
] = (
    1
    + 0.10
    * (
        candidate_cip_ranked[
            "supporting_occupations"
        ] - 1
    )
)

# ------------------------------------------------
# Calculate adjusted CIP recommendation score
# ------------------------------------------------

candidate_cip_ranked[
    "adjusted_cip_score"
] = (
    candidate_cip_ranked[
        "cip_recommendation_percentage"
    ]
    * candidate_cip_ranked[
        "support_factor"
    ]
)

# ------------------------------------------------
# Cap scores at 100
# ------------------------------------------------

candidate_cip_ranked[
    "adjusted_cip_score"
] = (
    candidate_cip_ranked[
        "adjusted_cip_score"
    ]
    .clip(upper=100)
)

# ------------------------------------------------
# Create a strict ranking
#
# First priority:
# adjusted CIP score
#
# Second priority:
# number of supporting occupations
#
# Third priority:
# best occupation rank
# ------------------------------------------------

candidate_cip_ranked = (
    candidate_cip_ranked
    .sort_values(
        [
            "candidate_id",
            "adjusted_cip_score",
            "supporting_occupations",
            "best_occupation_rank",
            "CIP2020Code"
        ],
        ascending=[
            True,
            False,
            False,
            True,
            True
        ]
    )
    .copy()
)

# ------------------------------------------------
# Assign sequential CIP rank
# ------------------------------------------------

candidate_cip_ranked[
    "cip_rank"
] = (
    candidate_cip_ranked
    .groupby("candidate_id")
    .cumcount()
    + 1
)

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — IMPROVED CIP RECOMMENDATION RANKING")
print("=" * 70)

print("\nNumber of candidates:")
print(
    candidate_cip_ranked[
        "candidate_id"
    ].nunique()
)

print("\nTotal candidate-CIP recommendations:")
print(
    len(candidate_cip_ranked)
)

print("\nAdjusted CIP score summary:")

display(
    candidate_cip_ranked[
        "adjusted_cip_score"
    ]
    .describe()
)

print("\nTop 20 improved CIP recommendations:")

display(
    candidate_cip_ranked[
        [
            "candidate_id",
            "cip_rank",
            "CIP2020Code",
            "CIP2020Title",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "support_factor",
            "adjusted_cip_score",
            "best_occupation_rank"
        ]
    ]
    .head(20)
)



WEEK 5 — IMPROVED CIP RECOMMENDATION RANKING

Number of candidates:
63

Total candidate-CIP recommendations:
4307

Adjusted CIP score summary:


count    4307.000000
mean       66.660478
std        13.046053
min        22.556657
25%        56.800446
50%        64.464883
75%        74.584153
max       100.000000
Name: adjusted_cip_score, dtype: float64


Top 20 improved CIP recommendations:


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,support_factor,adjusted_cip_score,best_occupation_rank
9,Candidate_001,1,11.0103,Information Technology.,68.434288,2,1.1,75.277717,1
10,Candidate_001,2,11.0104,Informatics.,68.434288,2,1.1,75.277717,1
11,Candidate_001,3,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1.1,75.277717,1
12,Candidate_001,4,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1.1,75.277717,1
13,Candidate_001,5,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1.1,75.277717,1
14,Candidate_001,6,11.0204,Computer Game Programming.,68.434288,2,1.1,75.277717,1
15,Candidate_001,7,11.0205,"Computer Programming, Specific Platforms.",68.434288,2,1.1,75.277717,1
16,Candidate_001,8,11.0701,Computer Science.,68.434288,2,1.1,75.277717,1
17,Candidate_001,9,14.0901,"Computer Engineering, General.",68.434288,2,1.1,75.277717,1
18,Candidate_001,10,14.0903,Computer Software Engineering.,68.434288,2,1.1,75.277717,1


In [379]:

# ================================================================
# CELL 47 — SELECT FINAL TOP 5 CIP RECOMMENDATIONS PER CANDIDATE
# ================================================================

# ------------------------------------------------
# Select the top 5 ranked CIP recommendations
# for each candidate
# ------------------------------------------------

final_candidate_cip_recommendations = (
    candidate_cip_ranked[
        candidate_cip_ranked["cip_rank"] <= 5
    ]
    .copy()
)

# ------------------------------------------------
# Select final columns
# ------------------------------------------------

final_candidate_cip_recommendations = (
    final_candidate_cip_recommendations[
        [
            "candidate_id",
            "cip_rank",
            "CIP2020Code",
            "CIP2020Title",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "best_occupation_rank",
            "support_factor",
            "adjusted_cip_score"
        ]
    ]
    .sort_values(
        [
            "candidate_id",
            "cip_rank"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------
# Validation
# ------------------------------------------------

print("=" * 70)
print("WEEK 5 — FINAL TOP 5 CIP RECOMMENDATIONS")
print("=" * 70)

print("\nNumber of candidates:")
print(
    final_candidate_cip_recommendations[
        "candidate_id"
    ].nunique()
)

print("\nTotal final CIP recommendations:")
print(
    len(final_candidate_cip_recommendations)
)

print("\nExpected maximum recommendations:")
print(
    final_candidate_cip_recommendations[
        "candidate_id"
    ].nunique() * 5
)

print("\nCIP recommendations per candidate:")

display(
    final_candidate_cip_recommendations
    .groupby("candidate_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nFirst 25 final recommendations:")

display(
    final_candidate_cip_recommendations
    .head(25)
)


WEEK 5 — FINAL TOP 5 CIP RECOMMENDATIONS

Number of candidates:
63

Total final CIP recommendations:
314

Expected maximum recommendations:
315

CIP recommendations per candidate:


4     1
5    62
Name: count, dtype: int64


First 25 final recommendations:


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,support_factor,adjusted_cip_score
0,Candidate_001,1,11.0103,Information Technology.,68.434288,2,1,1.1,75.277717
1,Candidate_001,2,11.0104,Informatics.,68.434288,2,1,1.1,75.277717
2,Candidate_001,3,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1,1.1,75.277717
3,Candidate_001,4,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1,1.1,75.277717
4,Candidate_001,5,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1,1.1,75.277717
5,Candidate_002,1,1.0106,Agricultural Business Technology/Technician.,54.252005,2,1,1.1,59.677206
6,Candidate_002,2,1.8201,"Veterinary Administrative Services, General.",54.252005,2,1,1.1,59.677206
7,Candidate_002,3,1.8202,Veterinary Office Management/Administration.,54.252005,2,1,1.1,59.677206
8,Candidate_002,4,1.8203,Veterinary Reception/Receptionist.,54.252005,2,1,1.1,59.677206
9,Candidate_002,5,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,54.252005,2,1,1.1,59.677206


In [380]:
# ================================================================
# WEEK 5 — CHECK CIP RECOMMENDATION DATAFRAME
# ================================================================

print("=" * 70)
print("CHECKING CIP RECOMMENDATION DATAFRAME")
print("=" * 70)

print("\nDoes final_cip_recommendations exist?")
print("final_cip_recommendations" in globals())

print("\nRelevant dataframe variables currently in memory:")

current_globals = list(globals().items())

for name, obj in current_globals:
    if not name.startswith("_"):
        if hasattr(obj, "shape") and hasattr(obj, "columns"):
            if any(
                keyword in name.lower()
                for keyword in ["cip", "recommend", "candidate"]
            ):
                print(f"{name}: shape={obj.shape}")

CHECKING CIP RECOMMENDATION DATAFRAME

Does final_cip_recommendations exist?
False

Relevant dataframe variables currently in memory:
onet_cip_profile: shape=(20, 12)
candidate_data: shape=(63, 9)
candidate_profiles: shape=(63, 5)
candidate_features: shape=(63, 16)
candidate_onet_skill_matrix: shape=(63, 23)
candidate_occupation_scores: shape=(945, 5)
candidate_score_range: shape=(63, 4)
cip_occupation_profiles: shape=(20, 12)
occupation_cip_structure: shape=(15, 6)
cip_raw: shape=(2848, 8)
cip_soc_crosswalk: shape=(6097, 4)
occupation_cip_mapping: shape=(196, 6)
occupation_cip_mapping_clean: shape=(185, 6)
cip_validation: shape=(13, 3)
occupation_cip_summary: shape=(13, 2)
occupation_cip_coverage: shape=(15, 4)
top_candidate_recommendations: shape=(315, 5)
candidate_recommendations_with_cip: shape=(315, 8)
candidate_cip_recommendations: shape=(5020, 10)
no_cip_recommendations: shape=(35, 5)
valid_candidate_cip: shape=(4985, 10)
candidate_cip_scores: shape=(4307, 9)
candidate_cip_ranke

In [381]:
# ================================================================
# WEEK 5 — VALIDATE CANDIDATES WITH FEWER THAN 5 CIP RECOMMENDATIONS
# ================================================================

candidate_counts = (
    final_candidate_cip_recommendations
    .groupby("candidate_id")
    .size()
    .reset_index(name="recommendation_count")
)

incomplete_candidates = candidate_counts[
    candidate_counts["recommendation_count"] < 5
]

print("=" * 70)
print("WEEK 5 — INCOMPLETE CIP RECOMMENDATION CHECK")
print("=" * 70)

print("\nCandidates with fewer than 5 recommendations:")

display(incomplete_candidates)

if len(incomplete_candidates) > 0:

    incomplete_ids = incomplete_candidates[
        "candidate_id"
    ].tolist()

    print("\nAvailable CIP recommendations for affected candidates:")

    display(
        candidate_cip_ranked[
            candidate_cip_ranked["candidate_id"].isin(
                incomplete_ids
            )
        ]
        .sort_values(
            ["candidate_id", "cip_rank"]
        )
        [
            [
                "candidate_id",
                "cip_rank",
                "CIP2020Code",
                "CIP2020Title",
                "adjusted_cip_score",
                "supporting_occupations"
            ]
        ]
    )

WEEK 5 — INCOMPLETE CIP RECOMMENDATION CHECK

Candidates with fewer than 5 recommendations:


,candidate_id,recommendation_count
44,Candidate_045,4



Available CIP recommendations for affected candidates:


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,adjusted_cip_score,supporting_occupations
3298,Candidate_045,1,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.,86.814514,2
3295,Candidate_045,2,52.0401,"Administrative Assistant and Secretarial Science, General.",79.831933,1
3296,Candidate_045,3,52.0402,Executive Assistant/Executive Secretary.,79.831933,1
3297,Candidate_045,4,52.0302,Accounting Technology/Technician and Bookkeeping.,79.213881,1


In [382]:
# ================================================================
# WEEK 5 — FINAL CIP RECOMMENDATION VALIDATION AND EXPORT
# ================================================================

print("=" * 70)
print("WEEK 5 — FINAL CIP RECOMMENDATION VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 1. Basic structure
# ------------------------------------------------

print("\nFinal dataframe shape:")
print(final_candidate_cip_recommendations.shape)

print("\nFinal columns:")
print(final_candidate_cip_recommendations.columns.tolist())


# ------------------------------------------------
# 2. Candidate coverage
# ------------------------------------------------

final_candidate_counts = (
    final_candidate_cip_recommendations
    .groupby("candidate_id")
    .size()
    .reset_index(name="recommendation_count")
)

print("\nCandidate recommendation distribution:")
display(
    final_candidate_counts[
        "recommendation_count"
    ].value_counts()
    .sort_index()
    .rename_axis("recommendation_count")
    .reset_index(name="candidate_count")
)


# ------------------------------------------------
# 3. Total recommendations
# ------------------------------------------------

print("\nTotal final recommendations:")
print(len(final_candidate_cip_recommendations))

print("\nUnique candidates:")
print(
    final_candidate_cip_recommendations[
        "candidate_id"
    ].nunique()
)


# ------------------------------------------------
# 4. Check duplicate candidate/CIP combinations
# ------------------------------------------------

duplicate_pairs = (
    final_candidate_cip_recommendations
    .duplicated(
        subset=["candidate_id", "CIP2020Code"]
    )
    .sum()
)

print("\nDuplicate candidate/CIP combinations:")
print(duplicate_pairs)


# ------------------------------------------------
# 5. Check missing values
# ------------------------------------------------

print("\nMissing values in final recommendations:")

display(
    final_candidate_cip_recommendations
    .isna()
    .sum()
    .to_frame("missing_count")
)


# ------------------------------------------------
# 6. Preview final recommendations
# ------------------------------------------------

print("\nFinal recommendation preview:")

display(
    final_candidate_cip_recommendations
    .sort_values(
        ["candidate_id", "cip_rank"]
    )
    .head(20)
)


# ------------------------------------------------
# 7. Export final dataset
# ------------------------------------------------

final_cip_export_path = (
    "week5_final_candidate_cip_recommendations.csv"
)

final_candidate_cip_recommendations.to_csv(
    final_cip_export_path,
    index=False
)

print("\n" + "=" * 70)
print("FINAL EXPORT COMPLETE")
print("=" * 70)

print("\nSaved file:")
print(final_cip_export_path)

WEEK 5 — FINAL CIP RECOMMENDATION VALIDATION

Final dataframe shape:
(314, 9)

Final columns:
['candidate_id', 'cip_rank', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_percentage', 'supporting_occupations', 'best_occupation_rank', 'support_factor', 'adjusted_cip_score']

Candidate recommendation distribution:


,recommendation_count,candidate_count
0,4,1
1,5,62



Total final recommendations:
314

Unique candidates:
63

Duplicate candidate/CIP combinations:
0

Missing values in final recommendations:


,missing_count
candidate_id,0
cip_rank,0
CIP2020Code,0
CIP2020Title,0
cip_recommendation_percentage,0
supporting_occupations,0
best_occupation_rank,0
support_factor,0
adjusted_cip_score,0



Final recommendation preview:


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,support_factor,adjusted_cip_score
0,Candidate_001,1,11.0103,Information Technology.,68.434288,2,1,1.1,75.277717
1,Candidate_001,2,11.0104,Informatics.,68.434288,2,1,1.1,75.277717
2,Candidate_001,3,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1,1.1,75.277717
3,Candidate_001,4,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1,1.1,75.277717
4,Candidate_001,5,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1,1.1,75.277717
5,Candidate_002,1,1.0106,Agricultural Business Technology/Technician.,54.252005,2,1,1.1,59.677206
6,Candidate_002,2,1.8201,"Veterinary Administrative Services, General.",54.252005,2,1,1.1,59.677206
7,Candidate_002,3,1.8202,Veterinary Office Management/Administration.,54.252005,2,1,1.1,59.677206
8,Candidate_002,4,1.8203,Veterinary Reception/Receptionist.,54.252005,2,1,1.1,59.677206
9,Candidate_002,5,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,54.252005,2,1,1.1,59.677206



FINAL EXPORT COMPLETE

Saved file:
week5_final_candidate_cip_recommendations.csv


In [383]:
# ================================================================
# WEEK 5 — FINAL REQUIREMENTS CHECK
# ================================================================

print("=" * 70)
print("WEEK 5 — FINAL REQUIREMENTS CHECK")
print("=" * 70)

# ------------------------------------------------
# Check all dataframe variables currently in memory
# ------------------------------------------------

print("\nRelevant Week 5 dataframes currently available:")

current_globals = list(globals().items())

for name, obj in current_globals:
    if not name.startswith("_"):
        if hasattr(obj, "shape") and hasattr(obj, "columns"):
            name_lower = name.lower()

            if any(
                keyword in name_lower
                for keyword in [
                    "skill",
                    "gap",
                    "missing",
                    "evaluation",
                    "label",
                    "ground",
                    "candidate"
                ]
            ):
                print(f"{name}: shape={obj.shape}")

# ------------------------------------------------
# Check common variable names
# ------------------------------------------------

print("\nSpecific Week 5 objects:")

objects_to_check = [
    "candidate_features",
    "candidate_onet_skill_matrix",
    "candidate_occupation_scores",
    "candidate_cip_recommendations",
    "candidate_cip_ranked",
    "final_candidate_cip_recommendations",
]

for obj_name in objects_to_check:
    print(
        f"{obj_name}:",
        "EXISTS" if obj_name in globals() else "NOT FOUND"
    )

print("\n" + "=" * 70)
print("END OF WEEK 5 REQUIREMENTS CHECK")
print("=" * 70)

WEEK 5 — FINAL REQUIREMENTS CHECK

Relevant Week 5 dataframes currently available:
onet_skills: shape=(200, 7)
candidate_data: shape=(63, 9)
candidate_profiles: shape=(63, 5)
candidate_features: shape=(63, 16)
occupation_skill_profiles: shape=(200, 7)
occupation_skill_profiles_final: shape=(150, 5)
occupation_skill_validation: shape=(15, 6)
skill_review: shape=(63, 3)
candidate_onet_skill_matrix: shape=(63, 23)
occupation_skill_weights: shape=(150, 5)
candidate_occupation_scores: shape=(945, 5)
candidate_score_range: shape=(63, 4)
top_candidate_recommendations: shape=(315, 5)
candidate_recommendations_with_cip: shape=(315, 8)
candidate_cip_recommendations: shape=(5020, 10)
valid_candidate_cip: shape=(4985, 10)
candidate_cip_scores: shape=(4307, 9)
candidate_cip_ranked: shape=(4307, 11)
final_candidate_cip_recommendations: shape=(314, 9)
candidate_counts: shape=(63, 2)
incomplete_candidates: shape=(1, 2)
final_candidate_counts: shape=(63, 2)

Specific Week 5 objects:
candidate_features:

In [384]:
# ================================================================
# WEEK 5 — INSPECT SKILL GAP AND EVALUATION DATA
# ================================================================

print("=" * 70)
print("WEEK 5 — SKILL GAP / EVALUATION INSPECTION")
print("=" * 70)

# ------------------------------------------------
# 1. Inspect skill_review
# ------------------------------------------------

print("\nSKILL_REVIEW")
print("-" * 70)

print("Shape:", skill_review.shape)
print("Columns:")
print(skill_review.columns.tolist())

display(skill_review.head(10))


# ------------------------------------------------
# 2. Inspect occupation skill validation
# ------------------------------------------------

print("\nOCCUPATION_SKILL_VALIDATION")
print("-" * 70)

print("Shape:", occupation_skill_validation.shape)
print("Columns:")
print(occupation_skill_validation.columns.tolist())

display(occupation_skill_validation.head(10))


# ------------------------------------------------
# 3. Search for possible evaluation/label data
# ------------------------------------------------

print("\nPOSSIBLE EVALUATION / LABEL DATAFRAMES")
print("-" * 70)

current_globals = list(globals().items())

for name, obj in current_globals:
    if not name.startswith("_"):
        if hasattr(obj, "shape") and hasattr(obj, "columns"):

            name_lower = name.lower()

            if any(
                keyword in name_lower
                for keyword in [
                    "evaluation",
                    "label",
                    "ground",
                    "truth",
                    "review",
                    "relevance"
                ]
            ):
                print(f"\n{name}: shape={obj.shape}")
                print("Columns:", obj.columns.tolist())
                display(obj.head(5))

WEEK 5 — SKILL GAP / EVALUATION INSPECTION

SKILL_REVIEW
----------------------------------------------------------------------
Shape: (63, 3)
Columns:
['candidate_id', 'raw_skill_count', 'raw_skills']


,candidate_id,raw_skill_count,raw_skills
0,Candidate_001,21,"[Eclipse, Git, GitHub, HTML, Hibernate, IntelliJ, JPA, JSON, Java, JavaScript, Maven, MongoDB, M..."
1,Candidate_002,10,"[Angular, CSS, GitHub, HTML, JavaScript, MySQL, NodeJS, PHP, React, TypeScript]"
2,Candidate_003,7,"[Agile, C, C#, C++, Git, Java, Jira]"
3,Candidate_004,14,"[CSS, Django, Flask, Git, GitHub, JavaScript, Jira, MySQL, Node.js, PHP, PostgreSQL, Python, Rea..."
4,Candidate_005,12,"[.NET, ASP.NET, C, C#, CSS, Java, JavaScript, MySQL, NodeJS, PHP, React, SQL]"
5,Candidate_006,14,"[Agile, Angular, CSS, HTML, JavaScript, MongoDB, MySQL, NodeJS, Python, REST, REST API, SQLite, ..."
6,Candidate_007,13,"[Agile, CSS, Docker, Git, HTML, Java, JavaScript, Kotlin, Oracle, Python, SQL, Scrum, Spring]"
7,Candidate_008,11,"[CSS, Django, Docker, HTML, JavaScript, MySQL, PostgreSQL, Python, React, SQL, Vue]"
8,Candidate_009,4,"[Git, GitHub, REST, REST API]"
9,Candidate_010,29,"[Agile, Angular, CSS, Eclipse, Excel, Git, GitHub, HTML, Hibernate, IntelliJ, JSON, Java, JavaSc..."



OCCUPATION_SKILL_VALIDATION
----------------------------------------------------------------------
Shape: (15, 6)
Columns:
['selected_occupation', 'total_skill_records', 'unique_skills', 'minimum_importance', 'maximum_importance', 'duplicate_skill_records']


,selected_occupation,total_skill_records,unique_skills,minimum_importance,maximum_importance,duplicate_skill_records
0,Administrative Assistant,10,10,1.00,4.000,0
1,Bookkeeper,10,10,1.12,3.380,0
2,Continuing Care Assistant,10,10,1.50,3.370,0
3,Delivery Driver,10,10,1.12,3.120,0
4,"Driver, Truck",10,10,1.00,3.120,0
5,Food Service Supervisor,10,10,1.00,3.880,0
6,Information Technology (IT) Analyst,10,10,2.25,4.065,0
7,Inside Sales Representative,10,10,1.00,3.750,0
8,Licensed Practical Nurse (L.P.N.),10,10,2.62,3.880,0
9,Office Administrator,10,10,1.19,4.000,0



POSSIBLE EVALUATION / LABEL DATAFRAMES
----------------------------------------------------------------------

education_review: shape=(63, 4)
Columns: ['candidate_id', 'degree_level', 'field_of_study', 'text_preview']


,candidate_id,degree_level,field_of_study,text_preview
0,Candidate_001,Master,information technology,**************** Java full stack developer (60% – IN JAVA BACKEND DEVELOPMENT AND 40% – IN WEB F...
1,Candidate_002,Diploma,engineering,****************** WEB FRONTEND DEVELOPER 5 years of experience in frontend development EXPERIEN...
2,Candidate_003,Master,business administration,"******************** Experience in C, C++ and C# programming Working knowledge of Agile, Jira an..."
3,Candidate_004,Not Identified,computer science,***************** Front-end Developer Mobile phone: ********** Linkedln:\t *******...
4,Candidate_005,Master,engineering,"Rehovot, Israel About me Senior solution architect with wide experience of creating and mainta..."



skill_review: shape=(63, 3)
Columns: ['candidate_id', 'raw_skill_count', 'raw_skills']


,candidate_id,raw_skill_count,raw_skills
0,Candidate_001,21,"[Eclipse, Git, GitHub, HTML, Hibernate, IntelliJ, JPA, JSON, Java, JavaScript, Maven, MongoDB, M..."
1,Candidate_002,10,"[Angular, CSS, GitHub, HTML, JavaScript, MySQL, NodeJS, PHP, React, TypeScript]"
2,Candidate_003,7,"[Agile, C, C#, C++, Git, Java, Jira]"
3,Candidate_004,14,"[CSS, Django, Flask, Git, GitHub, JavaScript, Jira, MySQL, Node.js, PHP, PostgreSQL, Python, Rea..."
4,Candidate_005,12,"[.NET, ASP.NET, C, C#, CSS, Java, JavaScript, MySQL, NodeJS, PHP, React, SQL]"


In [385]:
# ================================================================
# WEEK 5 — INITIAL CANDIDATE SKILL-GAP ANALYSIS
# ================================================================

print("=" * 70)
print("WEEK 5 — INITIAL CANDIDATE SKILL-GAP ANALYSIS")
print("=" * 70)

# ------------------------------------------------
# Inspect the available occupation skill profiles
# ------------------------------------------------

print("\nOccupation skill profile columns:")
print(occupation_skill_profiles_final.columns.tolist())

print("\nCandidate skill matrix columns:")
print(candidate_onet_skill_matrix.columns.tolist())


# ------------------------------------------------
# Show the structure before calculating gaps
# ------------------------------------------------

print("\nOccupation skill profile sample:")
display(occupation_skill_profiles_final.head())

print("\nCandidate skill matrix sample:")
display(candidate_onet_skill_matrix.head())

WEEK 5 — INITIAL CANDIDATE SKILL-GAP ANALYSIS

Occupation skill profile columns:
['selected_occupation', 'skill', 'importance', 'level', 'mapping_count']

Candidate skill matrix columns:
['candidate_id', 'Active Learning', 'Active Learning_evidence_count', 'Active Listening', 'Active Listening_evidence_count', 'Critical Thinking', 'Critical Thinking_evidence_count', 'Learning Strategies', 'Learning Strategies_evidence_count', 'Mathematics', 'Mathematics_evidence_count', 'Monitoring', 'Monitoring_evidence_count', 'Reading Comprehension', 'Reading Comprehension_evidence_count', 'Science', 'Science_evidence_count', 'Speaking', 'Speaking_evidence_count', 'Writing', 'Writing_evidence_count', 'onet_skill_match_count', 'total_onet_evidence_count']

Occupation skill profile sample:


,selected_occupation,skill,importance,level,mapping_count
0,Administrative Assistant,Active Listening,4.00,3.75,1
1,Administrative Assistant,Speaking,4.00,3.62,1
2,Administrative Assistant,Reading Comprehension,3.88,3.88,1
3,Administrative Assistant,Writing,3.75,3.50,1
4,Administrative Assistant,Monitoring,3.12,3.25,1



Candidate skill matrix sample:


,candidate_id,Active Learning,Active Learning_evidence_count,Active Listening,Active Listening_evidence_count,Critical Thinking,Critical Thinking_evidence_count,Learning Strategies,Learning Strategies_evidence_count,Mathematics,Mathematics_evidence_count,Monitoring,Monitoring_evidence_count,Reading Comprehension,Reading Comprehension_evidence_count,Science,Science_evidence_count,Speaking,Speaking_evidence_count,Writing,Writing_evidence_count,onet_skill_match_count,total_onet_evidence_count
0,Candidate_001,1,2,0,0,1,4,1,2,1,2,0,0,1,5,1,1,0,0,1,3,7,19
1,Candidate_002,1,3,0,0,1,3,1,2,0,0,0,0,1,3,0,0,0,0,1,1,5,12
2,Candidate_003,1,5,1,1,1,4,1,2,0,0,1,1,1,5,0,0,1,1,1,2,8,21
3,Candidate_004,1,3,0,0,1,3,1,2,1,1,1,1,1,5,1,1,0,0,1,3,8,19
4,Candidate_005,1,5,0,0,1,6,1,2,1,1,0,0,1,4,0,0,0,0,0,0,5,18


In [386]:
# ================================================================
# WEEK 5 — BUILD INITIAL CANDIDATE SKILL-GAP TABLE
# ================================================================

print("=" * 70)
print("WEEK 5 — INITIAL CANDIDATE SKILL-GAP ANALYSIS")
print("=" * 70)

# ------------------------------------------------
# Identify the 10 standardized O*NET skills
# ------------------------------------------------

onet_skill_columns = [
    col
    for col in candidate_onet_skill_matrix.columns
    if not col.endswith("_evidence_count")
    and col not in [
        "candidate_id",
        "onet_skill_match_count",
        "total_onet_evidence_count"
    ]
]

print("\nO*NET skills used in gap analysis:")
print(onet_skill_columns)


# ------------------------------------------------
# Create candidate skill lookup
# ------------------------------------------------

candidate_skill_lookup = (
    candidate_onet_skill_matrix
    .set_index("candidate_id")[onet_skill_columns]
)


# ------------------------------------------------
# Build skill-gap records
# ------------------------------------------------

skill_gap_records = []

for _, row in occupation_skill_profiles_final.iterrows():

    occupation = row["selected_occupation"]
    skill = row["skill"]
    importance = row["importance"]

    # Skip if the skill is not available in candidate matrix
    if skill not in candidate_skill_lookup.columns:
        continue

    for candidate_id in candidate_skill_lookup.index:

        candidate_has_skill = int(
            candidate_skill_lookup.loc[
                candidate_id,
                skill
            ]
        )

        if candidate_has_skill == 1:
            skill_status = "Matched"
        else:
            skill_status = "Missing"

        skill_gap_records.append({
            "candidate_id": candidate_id,
            "selected_occupation": occupation,
            "skill": skill,
            "importance": importance,
            "candidate_has_skill": candidate_has_skill,
            "skill_status": skill_status
        })


# ------------------------------------------------
# Create dataframe
# ------------------------------------------------

candidate_skill_gap = pd.DataFrame(
    skill_gap_records
)


# ------------------------------------------------
# Remove duplicate candidate / occupation / skill
# ------------------------------------------------

candidate_skill_gap = (
    candidate_skill_gap
    .drop_duplicates(
        subset=[
            "candidate_id",
            "selected_occupation",
            "skill"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------
# Add occupation-level summary
# ------------------------------------------------

skill_gap_summary = (
    candidate_skill_gap
    .groupby(
        [
            "candidate_id",
            "selected_occupation"
        ]
    )
    .agg(
        required_skill_count=("skill", "count"),
        matched_skill_count=(
            "candidate_has_skill",
            "sum"
        )
    )
    .reset_index()
)

skill_gap_summary["missing_skill_count"] = (
    skill_gap_summary["required_skill_count"]
    - skill_gap_summary["matched_skill_count"]
)

skill_gap_summary["skill_match_percentage"] = (
    skill_gap_summary["matched_skill_count"]
    / skill_gap_summary["required_skill_count"]
    * 100
)


# ------------------------------------------------
# Final validation
# ------------------------------------------------

print("\nSkill-gap table shape:")
print(candidate_skill_gap.shape)

print("\nSkill-gap summary shape:")
print(skill_gap_summary.shape)

print("\nSkill status distribution:")
display(
    candidate_skill_gap["skill_status"]
    .value_counts()
    .to_frame("count")
)

print("\nSample skill-gap records:")
display(
    candidate_skill_gap.head(20)
)

print("\nSample occupation-level skill-gap summary:")
display(
    skill_gap_summary.head(20)
)

WEEK 5 — INITIAL CANDIDATE SKILL-GAP ANALYSIS

O*NET skills used in gap analysis:
['Active Learning', 'Active Listening', 'Critical Thinking', 'Learning Strategies', 'Mathematics', 'Monitoring', 'Reading Comprehension', 'Science', 'Speaking', 'Writing']

Skill-gap table shape:
(9450, 6)

Skill-gap summary shape:
(945, 6)

Skill status distribution:


,count
skill_status,
Matched,6840
Missing,2610



Sample skill-gap records:


,candidate_id,selected_occupation,skill,importance,candidate_has_skill,skill_status
0,Candidate_001,Administrative Assistant,Active Listening,4.0,0,Missing
1,Candidate_002,Administrative Assistant,Active Listening,4.0,0,Missing
2,Candidate_003,Administrative Assistant,Active Listening,4.0,1,Matched
3,Candidate_004,Administrative Assistant,Active Listening,4.0,0,Missing
4,Candidate_005,Administrative Assistant,Active Listening,4.0,0,Missing
5,Candidate_006,Administrative Assistant,Active Listening,4.0,1,Matched
6,Candidate_007,Administrative Assistant,Active Listening,4.0,1,Matched
7,Candidate_008,Administrative Assistant,Active Listening,4.0,0,Missing
8,Candidate_009,Administrative Assistant,Active Listening,4.0,0,Missing
9,Candidate_010,Administrative Assistant,Active Listening,4.0,1,Matched



Sample occupation-level skill-gap summary:


,candidate_id,selected_occupation,required_skill_count,matched_skill_count,missing_skill_count,skill_match_percentage
0,Candidate_001,Administrative Assistant,10,7,3,70.0
1,Candidate_001,Bookkeeper,10,7,3,70.0
2,Candidate_001,Continuing Care Assistant,10,7,3,70.0
3,Candidate_001,Delivery Driver,10,7,3,70.0
4,Candidate_001,"Driver, Truck",10,7,3,70.0
5,Candidate_001,Food Service Supervisor,10,7,3,70.0
6,Candidate_001,Information Technology (IT) Analyst,10,7,3,70.0
7,Candidate_001,Inside Sales Representative,10,7,3,70.0
8,Candidate_001,Licensed Practical Nurse (L.P.N.),10,7,3,70.0
9,Candidate_001,Office Administrator,10,7,3,70.0


In [387]:
# ================================================================
# WEEK 5 — VALIDATE OCCUPATION-SPECIFIC SKILL PROFILES
# ================================================================

print("=" * 70)
print("WEEK 5 — OCCUPATION-SPECIFIC SKILL PROFILE CHECK")
print("=" * 70)

# ------------------------------------------------
# Count unique skills for each occupation
# ------------------------------------------------

occupation_skill_counts = (
    occupation_skill_profiles_final
    .groupby("selected_occupation")["skill"]
    .nunique()
    .reset_index(name="unique_skill_count")
)

print("\nUnique skills per occupation:")
display(occupation_skill_counts)


# ------------------------------------------------
# Show the actual skills for each occupation
# ------------------------------------------------

occupation_skill_sets = (
    occupation_skill_profiles_final
    .groupby("selected_occupation")["skill"]
    .apply(list)
    .reset_index(name="skills")
)

print("\nSkills by occupation:")
display(occupation_skill_sets)


# ------------------------------------------------
# Check whether all occupations use the same
# skill set
# ------------------------------------------------

skill_sets = [
    frozenset(skills)
    for skills in occupation_skill_sets["skills"]
]

all_same_skill_set = len(set(skill_sets)) == 1

print("\nDo all occupations use exactly the same skill set?")
print(all_same_skill_set)

print("\nNumber of distinct occupation skill sets:")
print(len(set(skill_sets)))

WEEK 5 — OCCUPATION-SPECIFIC SKILL PROFILE CHECK

Unique skills per occupation:


,selected_occupation,unique_skill_count
0,Administrative Assistant,10
1,Bookkeeper,10
2,Continuing Care Assistant,10
3,Delivery Driver,10
4,"Driver, Truck",10
5,Food Service Supervisor,10
6,Information Technology (IT) Analyst,10
7,Inside Sales Representative,10
8,Licensed Practical Nurse (L.P.N.),10
9,Office Administrator,10



Skills by occupation:


,selected_occupation,skills
0,Administrative Assistant,"[Active Listening, Speaking, Reading Comprehension, Writing, Monitoring, Critical Thinking, Acti..."
1,Bookkeeper,"[Mathematics, Active Listening, Critical Thinking, Reading Comprehension, Speaking, Writing, Mon..."
2,Continuing Care Assistant,"[Active Listening, Speaking, Critical Thinking, Monitoring, Reading Comprehension, Writing, Acti..."
3,Delivery Driver,"[Active Listening, Monitoring, Reading Comprehension, Speaking, Critical Thinking, Writing, Math..."
4,"Driver, Truck","[Monitoring, Critical Thinking, Reading Comprehension, Speaking, Active Listening, Writing, Acti..."
5,Food Service Supervisor,"[Monitoring, Speaking, Active Listening, Reading Comprehension, Critical Thinking, Learning Stra..."
6,Information Technology (IT) Analyst,"[Reading Comprehension, Critical Thinking, Speaking, Active Listening, Writing, Monitoring, Acti..."
7,Inside Sales Representative,"[Active Listening, Speaking, Critical Thinking, Active Learning, Monitoring, Reading Comprehensi..."
8,Licensed Practical Nurse (L.P.N.),"[Active Listening, Monitoring, Speaking, Critical Thinking, Reading Comprehension, Active Learni..."
9,Office Administrator,"[Active Listening, Reading Comprehension, Speaking, Writing, Critical Thinking, Monitoring, Acti..."



Do all occupations use exactly the same skill set?
True

Number of distinct occupation skill sets:
1


In [388]:
# ================================================================
# WEEK 5 — INSPECT CANDIDATE O*NET SKILL EVIDENCE
# ================================================================

print("=" * 70)
print("WEEK 5 — CANDIDATE O*NET SKILL EVIDENCE CHECK")
print("=" * 70)

skill_evidence_columns = [
    col
    for col in candidate_onet_skill_matrix.columns
    if col.endswith("_evidence_count")
]

print("\nEvidence-count columns:")
print(skill_evidence_columns)


print("\nEvidence-count summary:")

display(
    candidate_onet_skill_matrix[
        skill_evidence_columns
    ]
    .describe()
    .T
)


print("\nCandidate skill/evidence example:")

display(
    candidate_onet_skill_matrix[
        [
            "candidate_id",
            "Active Learning",
            "Active Learning_evidence_count",
            "Critical Thinking",
            "Critical Thinking_evidence_count",
            "Reading Comprehension",
            "Reading Comprehension_evidence_count",
            "Writing",
            "Writing_evidence_count"
        ]
    ]
    .head(15)
)

WEEK 5 — CANDIDATE O*NET SKILL EVIDENCE CHECK

Evidence-count columns:
['Active Learning_evidence_count', 'Active Listening_evidence_count', 'Critical Thinking_evidence_count', 'Learning Strategies_evidence_count', 'Mathematics_evidence_count', 'Monitoring_evidence_count', 'Reading Comprehension_evidence_count', 'Science_evidence_count', 'Speaking_evidence_count', 'Writing_evidence_count', 'total_onet_evidence_count']

Evidence-count summary:


,count,mean,std,min,25%,50%,75%,max
Active Learning_evidence_count,63.0,3.476190,1.785760,0.0,2.0,3.0,5.0,9.0
Active Listening_evidence_count,63.0,0.730159,0.953877,0.0,0.0,0.0,1.0,3.0
Critical Thinking_evidence_count,63.0,3.587302,1.728648,0.0,2.5,3.0,5.0,8.0
Learning Strategies_evidence_count,63.0,2.190476,1.468565,0.0,1.0,2.0,3.0,5.0
Mathematics_evidence_count,63.0,1.111111,1.108779,0.0,0.0,1.0,2.0,7.0
Monitoring_evidence_count,63.0,0.777778,0.974587,0.0,0.0,1.0,1.0,4.0
Reading Comprehension_evidence_count,63.0,4.111111,1.841380,1.0,3.0,4.0,5.0,10.0
Science_evidence_count,63.0,0.539683,0.779276,0.0,0.0,0.0,1.0,4.0
Speaking_evidence_count,63.0,0.730159,0.953877,0.0,0.0,0.0,1.0,3.0
Writing_evidence_count,63.0,1.714286,1.441645,0.0,1.0,1.0,2.0,6.0



Candidate skill/evidence example:


,candidate_id,Active Learning,Active Learning_evidence_count,Critical Thinking,Critical Thinking_evidence_count,Reading Comprehension,Reading Comprehension_evidence_count,Writing,Writing_evidence_count
0,Candidate_001,1,2,1,4,1,5,1,3
1,Candidate_002,1,3,1,3,1,3,1,1
2,Candidate_003,1,5,1,4,1,5,1,2
3,Candidate_004,1,3,1,3,1,5,1,3
4,Candidate_005,1,5,1,6,1,4,0,0
5,Candidate_006,1,5,1,3,1,4,1,1
6,Candidate_007,1,6,1,5,1,5,1,1
7,Candidate_008,1,2,1,3,1,3,0,0
8,Candidate_009,0,0,0,0,1,3,1,3
9,Candidate_010,1,5,1,4,1,7,1,5


In [389]:
# ================================================================
# WEEK 5 — OCCUPATION-AWARE WEIGHTED SKILL-GAP ANALYSIS
# ================================================================

print("=" * 70)
print("WEEK 5 — OCCUPATION-AWARE WEIGHTED SKILL-GAP ANALYSIS")
print("=" * 70)

# ------------------------------------------------
# 1. Prepare candidate skill presence
# ------------------------------------------------

onet_skill_columns = [
    col
    for col in candidate_onet_skill_matrix.columns
    if not col.endswith("_evidence_count")
    and col not in [
        "candidate_id",
        "onet_skill_match_count",
        "total_onet_evidence_count"
    ]
]

candidate_skill_lookup = (
    candidate_onet_skill_matrix
    .set_index("candidate_id")[onet_skill_columns]
)


# ------------------------------------------------
# 2. Prepare candidate evidence counts
# ------------------------------------------------

evidence_columns = [
    f"{skill}_evidence_count"
    for skill in onet_skill_columns
]

candidate_evidence_lookup = (
    candidate_onet_skill_matrix
    .set_index("candidate_id")[evidence_columns]
)


# ------------------------------------------------
# 3. Build occupation-aware skill-gap records
# ------------------------------------------------

weighted_gap_records = []

for _, row in occupation_skill_profiles_final.iterrows():

    occupation = row["selected_occupation"]
    skill = row["skill"]
    importance = float(row["importance"])

    if skill not in candidate_skill_lookup.columns:
        continue

    evidence_column = f"{skill}_evidence_count"

    for candidate_id in candidate_skill_lookup.index:

        has_skill = int(
            candidate_skill_lookup.loc[
                candidate_id,
                skill
            ]
        )

        evidence_count = int(
            candidate_evidence_lookup.loc[
                candidate_id,
                evidence_column
            ]
        )

        if has_skill == 1:
            status = "Matched"
            weighted_contribution = importance
        else:
            status = "Missing"
            weighted_contribution = 0.0

        weighted_gap_records.append({
            "candidate_id": candidate_id,
            "selected_occupation": occupation,
            "skill": skill,
            "importance": importance,
            "candidate_has_skill": has_skill,
            "evidence_count": evidence_count,
            "skill_status": status,
            "weighted_contribution": weighted_contribution
        })


# ------------------------------------------------
# 4. Create detailed skill-gap dataframe
# ------------------------------------------------

candidate_skill_gap_weighted = pd.DataFrame(
    weighted_gap_records
)

candidate_skill_gap_weighted = (
    candidate_skill_gap_weighted
    .drop_duplicates(
        subset=[
            "candidate_id",
            "selected_occupation",
            "skill"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------
# 5. Calculate occupation-specific coverage
# ------------------------------------------------

weighted_gap_summary = (
    candidate_skill_gap_weighted
    .groupby(
        [
            "candidate_id",
            "selected_occupation"
        ]
    )
    .agg(
        required_skill_count=("skill", "count"),
        matched_skill_count=(
            "candidate_has_skill",
            "sum"
        ),
        total_skill_importance=(
            "importance",
            "sum"
        ),
        matched_skill_importance=(
            "weighted_contribution",
            "sum"
        ),
        total_evidence_count=(
            "evidence_count",
            "sum"
        )
    )
    .reset_index()
)


# ------------------------------------------------
# 6. Calculate weighted skill coverage
# ------------------------------------------------

weighted_gap_summary["missing_skill_count"] = (
    weighted_gap_summary["required_skill_count"]
    - weighted_gap_summary["matched_skill_count"]
)

weighted_gap_summary["weighted_skill_coverage"] = (
    weighted_gap_summary["matched_skill_importance"]
    / weighted_gap_summary["total_skill_importance"]
    * 100
)


# ------------------------------------------------
# 7. Extract missing-skill lists
# ------------------------------------------------

missing_skill_lists = (
    candidate_skill_gap_weighted[
        candidate_skill_gap_weighted["skill_status"] == "Missing"
    ]
    .groupby(
        [
            "candidate_id",
            "selected_occupation"
        ]
    )["skill"]
    .apply(list)
    .reset_index(name="missing_skills")
)

weighted_gap_summary = weighted_gap_summary.merge(
    missing_skill_lists,
    on=[
        "candidate_id",
        "selected_occupation"
    ],
    how="left"
)

weighted_gap_summary["missing_skills"] = (
    weighted_gap_summary["missing_skills"]
    .apply(
        lambda x: x if isinstance(x, list) else []
    )
)


# ------------------------------------------------
# 8. Validation
# ------------------------------------------------

print("\nDetailed skill-gap table:")
print(candidate_skill_gap_weighted.shape)

print("\nOccupation-level skill-gap summary:")
print(weighted_gap_summary.shape)

print("\nSkill status distribution:")

display(
    candidate_skill_gap_weighted[
        "skill_status"
    ]
    .value_counts()
    .to_frame("count")
)


print("\nOccupation-specific coverage examples:")

display(
    weighted_gap_summary
    .sort_values(
        [
            "candidate_id",
            "weighted_skill_coverage"
        ],
        ascending=[True, False]
    )
    .head(20)
)


print("\nDetailed missing-skill examples:")

display(
    weighted_gap_summary[
        [
            "candidate_id",
            "selected_occupation",
            "matched_skill_count",
            "missing_skill_count",
            "weighted_skill_coverage",
            "missing_skills"
        ]
    ]
    .head(20)
)

WEEK 5 — OCCUPATION-AWARE WEIGHTED SKILL-GAP ANALYSIS

Detailed skill-gap table:
(9450, 8)

Occupation-level skill-gap summary:
(945, 10)

Skill status distribution:


,count
skill_status,
Matched,6840
Missing,2610



Occupation-specific coverage examples:


,candidate_id,selected_occupation,required_skill_count,matched_skill_count,total_skill_importance,matched_skill_importance,total_evidence_count,missing_skill_count,weighted_skill_coverage,missing_skills
14,Candidate_001,Software Developer,10,7,31.1200,21.6200,19,3,69.473008,"[Active Listening, Speaking, Monitoring]"
6,Candidate_001,Information Technology (IT) Analyst,10,7,34.4125,23.1925,19,3,67.395568,"[Speaking, Active Listening, Monitoring]"
13,Candidate_001,Secondary School Teacher,10,7,35.8800,24.0000,19,3,66.889632,"[Active Listening, Speaking, Monitoring]"
1,Candidate_001,Bookkeeper,10,7,28.2400,18.8700,19,3,66.820113,"[Active Listening, Speaking, Monitoring]"
8,Candidate_001,Licensed Practical Nurse (L.P.N.),10,7,34.3900,22.7500,19,3,66.152951,"[Active Listening, Monitoring, Speaking]"
9,Candidate_001,Office Administrator,10,7,33.4400,21.9400,19,3,65.610048,"[Active Listening, Speaking, Monitoring]"
10,Candidate_001,Office Manager,10,7,34.7500,22.7500,19,3,65.467626,"[Active Listening, Monitoring, Speaking]"
11,Candidate_001,Restaurant Manager,10,7,32.7500,21.1100,19,3,64.458015,"[Active Listening, Monitoring, Speaking]"
4,Candidate_001,"Driver, Truck",10,7,25.2500,16.2500,19,3,64.356436,"[Monitoring, Speaking, Active Listening]"
2,Candidate_001,Continuing Care Assistant,10,7,25.9800,16.6150,19,3,63.953041,"[Active Listening, Speaking, Monitoring]"



Detailed missing-skill examples:


,candidate_id,selected_occupation,matched_skill_count,missing_skill_count,weighted_skill_coverage,missing_skills
0,Candidate_001,Administrative Assistant,7,3,62.621849,"[Active Listening, Speaking, Monitoring]"
1,Candidate_001,Bookkeeper,7,3,66.820113,"[Active Listening, Speaking, Monitoring]"
2,Candidate_001,Continuing Care Assistant,7,3,63.953041,"[Active Listening, Speaking, Monitoring]"
3,Candidate_001,Delivery Driver,7,3,63.608087,"[Active Listening, Monitoring, Speaking]"
4,Candidate_001,"Driver, Truck",7,3,64.356436,"[Monitoring, Speaking, Active Listening]"
5,Candidate_001,Food Service Supervisor,7,3,62.870968,"[Monitoring, Speaking, Active Listening]"
6,Candidate_001,Information Technology (IT) Analyst,7,3,67.395568,"[Speaking, Active Listening, Monitoring]"
7,Candidate_001,Inside Sales Representative,7,3,63.617464,"[Active Listening, Speaking, Monitoring]"
8,Candidate_001,Licensed Practical Nurse (L.P.N.),7,3,66.152951,"[Active Listening, Monitoring, Speaking]"
9,Candidate_001,Office Administrator,7,3,65.610048,"[Active Listening, Speaking, Monitoring]"


In [390]:
# ================================================================
# WEEK 5 — CHECK TOP OCCUPATION RECOMMENDATION COLUMNS
# ================================================================

print("=" * 70)
print("TOP CANDIDATE RECOMMENDATIONS — COLUMN CHECK")
print("=" * 70)

print("\nShape:")
print(top_candidate_recommendations.shape)

print("\nColumns:")
print(top_candidate_recommendations.columns.tolist())

print("\nSample:")
display(
    top_candidate_recommendations.head(10)
)

TOP CANDIDATE RECOMMENDATIONS — COLUMN CHECK

Shape:
(315, 5)

Columns:
['candidate_id', 'selected_occupation', 'recommendation_score', 'recommendation_percentage', 'occupation_rank']

Sample:


,candidate_id,selected_occupation,recommendation_score,recommendation_percentage,occupation_rank
0,Candidate_001,Software Developer,0.694730,69.473008,1
1,Candidate_001,Information Technology (IT) Analyst,0.673956,67.395568,2
2,Candidate_001,Secondary School Teacher,0.668896,66.889632,3
3,Candidate_001,Bookkeeper,0.668201,66.820113,4
4,Candidate_001,Licensed Practical Nurse (L.P.N.),0.661530,66.152951,5
15,Candidate_002,Office Manager,0.546763,54.676259,1
16,Candidate_002,Office Administrator,0.538278,53.827751,2
17,Candidate_002,Software Developer,0.538239,53.823907,3
18,Candidate_002,Secondary School Teacher,0.536511,53.651059,4
19,Candidate_002,Information Technology (IT) Analyst,0.528660,52.865964,5


In [391]:
# ================================================================
# WEEK 5 — FINAL CANDIDATE EVALUATION SUMMARY
# ================================================================

print("=" * 70)
print("WEEK 5 — FINAL CANDIDATE EVALUATION SUMMARY")
print("=" * 70)


# ------------------------------------------------
# 1. Select top occupation for each candidate
# ------------------------------------------------

top_occupation_summary = (
    top_candidate_recommendations
    .sort_values(
        ["candidate_id", "occupation_rank"]
    )
    .groupby("candidate_id")
    .first()
    .reset_index()
)


top_occupation_summary = top_occupation_summary[
    [
        "candidate_id",
        "selected_occupation",
        "recommendation_score",
        "recommendation_percentage",
        "occupation_rank"
    ]
]


# ------------------------------------------------
# 2. Attach occupation-specific skill-gap results
# ------------------------------------------------

evaluation_summary = (
    top_occupation_summary
    .merge(
        weighted_gap_summary[
            [
                "candidate_id",
                "selected_occupation",
                "matched_skill_count",
                "missing_skill_count",
                "weighted_skill_coverage",
                "missing_skills"
            ]
        ],
        on=[
            "candidate_id",
            "selected_occupation"
        ],
        how="left"
    )
)


# ------------------------------------------------
# 3. Attach top CIP recommendation
# ------------------------------------------------

top_cip = (
    final_candidate_cip_recommendations
    .sort_values(
        ["candidate_id", "cip_rank"]
    )
    .groupby("candidate_id")
    .first()
    .reset_index()
)


top_cip = top_cip[
    [
        "candidate_id",
        "CIP2020Code",
        "CIP2020Title",
        "cip_recommendation_percentage",
        "adjusted_cip_score"
    ]
]


evaluation_summary = (
    evaluation_summary
    .merge(
        top_cip,
        on="candidate_id",
        how="left"
    )
)


# ------------------------------------------------
# 4. Validation
# ------------------------------------------------

print("\nEvaluation summary shape:")
print(evaluation_summary.shape)

print("\nNumber of unique candidates:")
print(evaluation_summary["candidate_id"].nunique())


print("\nColumns:")
print(evaluation_summary.columns.tolist())


print("\nMissing values:")

display(
    evaluation_summary
    .isna()
    .sum()
    .to_frame("missing_count")
)


# ------------------------------------------------
# 5. Final preview
# ------------------------------------------------

print("\nFinal candidate evaluation preview:")

display(
    evaluation_summary
    .sort_values("candidate_id")
    .head(20)
)

WEEK 5 — FINAL CANDIDATE EVALUATION SUMMARY

Evaluation summary shape:
(63, 13)

Number of unique candidates:
63

Columns:
['candidate_id', 'selected_occupation', 'recommendation_score', 'recommendation_percentage', 'occupation_rank', 'matched_skill_count', 'missing_skill_count', 'weighted_skill_coverage', 'missing_skills', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_percentage', 'adjusted_cip_score']

Missing values:


,missing_count
candidate_id,0
selected_occupation,0
recommendation_score,0
recommendation_percentage,0
occupation_rank,0
matched_skill_count,0
missing_skill_count,0
weighted_skill_coverage,0
missing_skills,0
CIP2020Code,0



Final candidate evaluation preview:


,candidate_id,selected_occupation,recommendation_score,recommendation_percentage,occupation_rank,matched_skill_count,missing_skill_count,weighted_skill_coverage,missing_skills,CIP2020Code,CIP2020Title,cip_recommendation_percentage,adjusted_cip_score
0,Candidate_001,Software Developer,0.694730,69.473008,1,7,3,69.473008,"[Active Listening, Speaking, Monitoring]",11.0103,Information Technology.,68.434288,75.277717
1,Candidate_002,Office Manager,0.546763,54.676259,1,5,5,54.676259,"[Active Listening, Monitoring, Speaking, Mathematics, Science]",1.0106,Agricultural Business Technology/Technician.,54.252005,59.677206
2,Candidate_003,Administrative Assistant,0.899160,89.915966,1,8,2,89.915966,"[Mathematics, Science]",52.0401,"Administrative Assistant and Secretarial Science, General.",89.114101,100.000000
3,Candidate_004,Software Developer,0.791131,79.113111,1,8,2,79.113111,"[Active Listening, Speaking]",11.0103,Information Technology.,78.339701,86.173671
4,Candidate_005,Software Developer,0.522172,52.217224,1,5,5,52.217224,"[Active Listening, Writing, Speaking, Monitoring, Science]",1.0106,Agricultural Business Technology/Technician.,51.317941,56.449735
5,Candidate_006,Software Developer,0.903599,90.359897,1,9,1,90.359897,[Monitoring],11.0103,Information Technology.,90.094587,99.104046
6,Candidate_007,Administrative Assistant,1.000000,100.000000,1,10,0,100.000000,[],12.05,"Cooking and Related Culinary Arts, General.",100.000000,100.000000
7,Candidate_008,Software Developer,0.686697,68.669666,1,7,3,68.669666,"[Active Listening, Writing, Speaking]",11.0103,Information Technology.,67.803776,74.584153
8,Candidate_009,Administrative Assistant,0.256471,25.647059,1,2,8,25.647059,"[Active Listening, Speaking, Monitoring, Critical Thinking, Active Learning, Learning Strategies...",52.0401,"Administrative Assistant and Secretarial Science, General.",24.411448,26.852593
9,Candidate_010,Office Manager,0.971223,97.122302,1,9,1,97.122302,[Science],52.0401,"Administrative Assistant and Secretarial Science, General.",96.880479,100.000000


In [392]:
print("=" * 70)
print("WEEK 5 — O*NET OCCUPATION SKILL SOURCE CHECK")
print("=" * 70)

print("\nO*NET skills shape:")
print(onet_skills.shape)

print("\nO*NET skills columns:")
print(onet_skills.columns.tolist())

print("\nUnique O*NET occupations:")
print(onet_skills["onet_soc_code"].nunique())

print("\nUnique skill names:")
print(onet_skills["skill"].nunique())

print("\nSkills by O*NET occupation:")
display(
    onet_skills
    .groupby("onet_soc_code")["skill"]
    .nunique()
    .describe()
)

print("\nSample O*NET occupation-skill records:")
display(onet_skills.head(30))

WEEK 5 — O*NET OCCUPATION SKILL SOURCE CHECK

O*NET skills shape:
(200, 7)

O*NET skills columns:
['onet_soc_code', 'element_id', 'skill', 'importance', 'level', 'selected_occupation', 'onet_title']

Unique O*NET occupations:
18

Unique skill names:
10

Skills by O*NET occupation:


count    18.0
mean     10.0
std       0.0
min      10.0
25%      10.0
50%      10.0
75%      10.0
max      10.0
Name: skill, dtype: float64


Sample O*NET occupation-skill records:


,onet_soc_code,element_id,skill,importance,level,selected_occupation,onet_title
0,11-9051.00,2.A.1.a,Reading Comprehension,3.75,3.88,Restaurant Manager,Food Service Managers
1,11-9051.00,2.A.1.b,Active Listening,3.88,3.50,Restaurant Manager,Food Service Managers
2,11-9051.00,2.A.1.c,Writing,3.00,3.12,Restaurant Manager,Food Service Managers
3,11-9051.00,2.A.1.d,Speaking,3.88,4.00,Restaurant Manager,Food Service Managers
4,11-9051.00,2.A.1.e,Mathematics,2.88,2.88,Restaurant Manager,Food Service Managers
5,11-9051.00,2.A.1.f,Science,1.62,0.62,Restaurant Manager,Food Service Managers
6,11-9051.00,2.A.2.a,Critical Thinking,3.62,3.75,Restaurant Manager,Food Service Managers
7,11-9051.00,2.A.2.b,Active Learning,3.12,3.75,Restaurant Manager,Food Service Managers
8,11-9051.00,2.A.2.c,Learning Strategies,3.12,3.12,Restaurant Manager,Food Service Managers
9,11-9051.00,2.A.2.d,Monitoring,3.88,3.88,Restaurant Manager,Food Service Managers


In [393]:
print("=" * 70)
print("WEEK 5 — CANADIAN OCCUPATION / O*NET AGGREGATION CHECK")
print("=" * 70)

occupation_mapping_check = (
    onet_skills
    .groupby("selected_occupation")
    .agg(
        onet_record_count=("onet_soc_code", "nunique"),
        skill_record_count=("skill", "size"),
        unique_skill_count=("skill", "nunique"),
        min_importance=("importance", "min"),
        max_importance=("importance", "max")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(occupation_mapping_check)

WEEK 5 — CANADIAN OCCUPATION / O*NET AGGREGATION CHECK


,selected_occupation,onet_record_count,skill_record_count,unique_skill_count,min_importance,max_importance
0,Administrative Assistant,1,10,10,1.00,4.00
1,Bookkeeper,1,10,10,1.12,3.38
2,Continuing Care Assistant,2,20,10,1.25,3.62
3,Delivery Driver,1,10,10,1.12,3.12
4,"Driver, Truck",1,10,10,1.00,3.12
5,Food Service Supervisor,1,10,10,1.00,3.88
6,Information Technology (IT) Analyst,4,40,10,1.88,4.38
7,Inside Sales Representative,1,10,10,1.00,3.75
8,Licensed Practical Nurse (L.P.N.),1,10,10,2.62,3.88
9,Office Administrator,2,20,10,1.00,4.00


In [394]:
print("\nO*NET codes associated with each Canadian occupation:")

occupation_onet_codes = (
    onet_skills
    .groupby("selected_occupation")["onet_soc_code"]
    .agg(list)
    .reset_index()
)

display(occupation_onet_codes)


O*NET codes associated with each Canadian occupation:


,selected_occupation,onet_soc_code
0,Administrative Assistant,"[43-6014.00, 43-6014.00, 43-6014.00, 43-6014.00, 43-6014.00, 43-6014.00, 43-6014.00, 43-6014.00,..."
1,Bookkeeper,"[43-3031.00, 43-3031.00, 43-3031.00, 43-3031.00, 43-3031.00, 43-3031.00, 43-3031.00, 43-3031.00,..."
2,Continuing Care Assistant,"[31-1131.00, 31-1131.00, 31-1131.00, 31-1131.00, 31-1131.00, 31-1131.00, 31-1131.00, 31-1131.00,..."
3,Delivery Driver,"[53-3033.00, 53-3033.00, 53-3033.00, 53-3033.00, 53-3033.00, 53-3033.00, 53-3033.00, 53-3033.00,..."
4,"Driver, Truck","[53-3032.00, 53-3032.00, 53-3032.00, 53-3032.00, 53-3032.00, 53-3032.00, 53-3032.00, 53-3032.00,..."
5,Food Service Supervisor,"[35-1012.00, 35-1012.00, 35-1012.00, 35-1012.00, 35-1012.00, 35-1012.00, 35-1012.00, 35-1012.00,..."
6,Information Technology (IT) Analyst,"[15-1211.00, 15-1211.00, 15-1211.00, 15-1211.00, 15-1211.00, 15-1211.00, 15-1211.00, 15-1211.00,..."
7,Inside Sales Representative,"[41-2031.00, 41-2031.00, 41-2031.00, 41-2031.00, 41-2031.00, 41-2031.00, 41-2031.00, 41-2031.00,..."
8,Licensed Practical Nurse (L.P.N.),"[29-2061.00, 29-2061.00, 29-2061.00, 29-2061.00, 29-2061.00, 29-2061.00, 29-2061.00, 29-2061.00,..."
9,Office Administrator,"[43-1011.00, 43-1011.00, 43-1011.00, 43-1011.00, 43-1011.00, 43-1011.00, 43-1011.00, 43-1011.00,..."


In [395]:
print("\n" + "=" * 70)
print("WEEK 5 — VERIFY FINAL OCCUPATION SKILL WEIGHTS")
print("=" * 70)

display(
    occupation_skill_profiles_final
    .sort_values(
        ["selected_occupation", "importance"],
        ascending=[True, False]
    )
    .head(50)
)


WEEK 5 — VERIFY FINAL OCCUPATION SKILL WEIGHTS


,selected_occupation,skill,importance,level,mapping_count
0,Administrative Assistant,Active Listening,4.000,3.750,1
1,Administrative Assistant,Speaking,4.000,3.620,1
2,Administrative Assistant,Reading Comprehension,3.880,3.880,1
3,Administrative Assistant,Writing,3.750,3.500,1
4,Administrative Assistant,Monitoring,3.120,3.250,1
5,Administrative Assistant,Critical Thinking,3.000,3.620,1
6,Administrative Assistant,Active Learning,2.880,3.000,1
7,Administrative Assistant,Learning Strategies,2.120,1.880,1
8,Administrative Assistant,Mathematics,2.000,1.620,1
9,Administrative Assistant,Science,1.000,0.000,1


In [396]:
print("=" * 70)
print("WEEK 5 — CIP RECOMMENDATION TRACE")
print("=" * 70)

print("\nCandidate CIP recommendations columns:")
print(candidate_cip_recommendations.columns.tolist())

print("\nShape:")
print(candidate_cip_recommendations.shape)

print("\nSample records:")
display(
    candidate_cip_recommendations.head(20)
)

WEEK 5 — CIP RECOMMENDATION TRACE

Candidate CIP recommendations columns:
['candidate_id', 'selected_occupation', 'recommendation_score', 'recommendation_percentage', 'occupation_rank', 'cip_program_count', 'cip_pathway_available', 'cip_status', 'CIP2020Code', 'CIP2020Title']

Shape:
(5020, 10)

Sample records:


,candidate_id,selected_occupation,recommendation_score,recommendation_percentage,occupation_rank,cip_program_count,cip_pathway_available,cip_status,CIP2020Code,CIP2020Title
0,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0102,Artificial Intelligence.
1,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0103,Information Technology.
2,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0104,Informatics.
3,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0201,"Computer Programming/Programmer, General."
4,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0202,"Computer Programming, Specific Applications."
5,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0203,"Computer Programming, Vendor/Product Certification."
6,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0204,Computer Game Programming.
7,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0205,"Computer Programming, Specific Platforms."
8,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0401,Information Science/Studies.
9,Candidate_001,Software Developer,0.69473,69.473008,1,20,True,CIP pathway available,11.0701,Computer Science.


In [397]:
print("=" * 70)
print("WEEK 5 — TRACE QUESTIONABLE CIP RECOMMENDATIONS")
print("=" * 70)

display(
    final_candidate_cip_recommendations[
        final_candidate_cip_recommendations["candidate_id"].isin(
            ["Candidate_002", "Candidate_005", "Candidate_007"]
        )
    ]
    .sort_values(
        ["candidate_id", "cip_rank"]
    )
)

WEEK 5 — TRACE QUESTIONABLE CIP RECOMMENDATIONS


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,support_factor,adjusted_cip_score
5,Candidate_002,1,1.0106,Agricultural Business Technology/Technician.,54.252005,2,1,1.1,59.677206
6,Candidate_002,2,1.8201,"Veterinary Administrative Services, General.",54.252005,2,1,1.1,59.677206
7,Candidate_002,3,1.8202,Veterinary Office Management/Administration.,54.252005,2,1,1.1,59.677206
8,Candidate_002,4,1.8203,Veterinary Reception/Receptionist.,54.252005,2,1,1.1,59.677206
9,Candidate_002,5,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,54.252005,2,1,1.1,59.677206
20,Candidate_005,1,1.0106,Agricultural Business Technology/Technician.,51.317941,2,3,1.1,56.449735
21,Candidate_005,2,1.8201,"Veterinary Administrative Services, General.",51.317941,2,3,1.1,56.449735
22,Candidate_005,3,1.8202,Veterinary Office Management/Administration.,51.317941,2,3,1.1,56.449735
23,Candidate_005,4,1.8203,Veterinary Reception/Receptionist.,51.317941,2,3,1.1,56.449735
24,Candidate_005,5,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,51.317941,2,3,1.1,56.449735


In [398]:
print("=" * 70)
print("WEEK 5 — CIP SUPPORTING OCCUPATION TRACE")
print("=" * 70)

display(
    candidate_cip_ranked[
        candidate_cip_ranked["candidate_id"].isin(
            ["Candidate_002", "Candidate_005", "Candidate_007"]
        )
    ]
    .sort_values(
        ["candidate_id", "cip_rank"]
    )
    [
        [
            "candidate_id",
            "cip_rank",
            "CIP2020Code",
            "CIP2020Title",
            "adjusted_cip_score",
            "supporting_occupations"
        ]
    ]
    .head(30)
)

WEEK 5 — CIP SUPPORTING OCCUPATION TRACE


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,adjusted_cip_score,supporting_occupations
126,Candidate_002,1,1.0106,Agricultural Business Technology/Technician.,59.677206,2
127,Candidate_002,2,1.8201,"Veterinary Administrative Services, General.",59.677206,2
128,Candidate_002,3,1.8202,Veterinary Office Management/Administration.,59.677206,2
129,Candidate_002,4,1.8203,Veterinary Reception/Receptionist.,59.677206,2
130,Candidate_002,5,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,59.677206,2
131,Candidate_002,6,51.0705,Medical Office Management/Administration.,59.677206,2
132,Candidate_002,7,51.0711,Medical/Health Management and Clinical Assistant/Specialist.,59.677206,2
133,Candidate_002,8,51.0712,Medical Reception/Receptionist.,59.677206,2
134,Candidate_002,9,52.0204,Office Management and Supervision.,59.677206,2
135,Candidate_002,10,52.0207,Customer Service Management.,59.677206,2


In [399]:
print("=" * 70)
print("WEEK 5 — CIP MAPPING STRUCTURE CHECK")
print("=" * 70)

print("\nCIP recommendation source columns:")
print(candidate_cip_recommendations.columns.tolist())

print("\nShape:")
print(candidate_cip_recommendations.shape)

print("\nCIP programs per occupation:")
display(
    candidate_cip_recommendations[
        [
            "selected_occupation",
            "cip_program_count",
            "cip_pathway_available"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "cip_program_count",
        ascending=False
    )
)

print("\nUnique CIP programs per occupation:")
display(
    candidate_cip_recommendations
    .groupby("selected_occupation")["CIP2020Code"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="unique_cip_count")
)

WEEK 5 — CIP MAPPING STRUCTURE CHECK

CIP recommendation source columns:
['candidate_id', 'selected_occupation', 'recommendation_score', 'recommendation_percentage', 'occupation_rank', 'cip_program_count', 'cip_pathway_available', 'cip_status', 'CIP2020Code', 'CIP2020Title']

Shape:
(5020, 10)

CIP programs per occupation:


,selected_occupation,cip_program_count,cip_pathway_available
42,Secondary School Teacher,92,True
20,Information Technology (IT) Analyst,22,True
0,Software Developer,20,True
137,Office Manager,13,True
150,Office Administrator,13,True
775,Restaurant Manager,10,True
649,Food Service Supervisor,5,True
657,Continuing Care Assistant,3,True
135,Licensed Practical Nurse (L.P.N.),2,True
297,Administrative Assistant,2,True



Unique CIP programs per occupation:


,selected_occupation,unique_cip_count
0,Secondary School Teacher,92
1,Information Technology (IT) Analyst,22
2,Software Developer,20
3,Office Administrator,13
4,Office Manager,13
5,Restaurant Manager,10
6,Food Service Supervisor,5
7,Continuing Care Assistant,3
8,Administrative Assistant,2
9,Licensed Practical Nurse (L.P.N.),2


In [400]:
print("=" * 70)
print("WEEK 5 — CIP CROSS-OCCUPATION TRACE")
print("=" * 70)

trace = (
    candidate_cip_recommendations
    .groupby("CIP2020Code")
    .agg(
        CIP2020Title=("CIP2020Title", "first"),
        occupation_count=("selected_occupation", "nunique"),
        occupations=("selected_occupation", list)
    )
    .reset_index()
    .sort_values(
        "occupation_count",
        ascending=False
    )
)

display(trace.head(30))

WEEK 5 — CIP CROSS-OCCUPATION TRACE


,CIP2020Code,CIP2020Title,occupation_count,occupations
145,52.0401,"Administrative Assistant and Secretarial Science, General.",3,"[Office Manager, Office Administrator, Administrative Assistant, Office Manager, Office Administ..."
146,52.0402,Executive Assistant/Executive Secretary.,3,"[Office Manager, Office Administrator, Administrative Assistant, Office Manager, Office Administ..."
0,1.0106,Agricultural Business Technology/Technician.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
1,1.8201,"Veterinary Administrative Services, General.",2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
105,19.0505,Foodservice Systems Administration/Management.,2,"[Food Service Supervisor, Restaurant Manager, Food Service Supervisor, Restaurant Manager, Food ..."
130,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.,2,"[Driver, Truck, Delivery Driver, Delivery Driver, Driver, Truck, Driver, Truck, Delivery Driver,..."
133,51.0705,Medical Office Management/Administration.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
134,51.0711,Medical/Health Management and Clinical Assistant/Specialist.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
135,51.0712,Medical Reception/Receptionist.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
72,14.0901,"Computer Engineering, General.",2,"[Software Developer, Information Technology (IT) Analyst, Software Developer, Information Techno..."


In [401]:
print("=" * 70)
print("WEEK 5 — CANDIDATE_007 CIP TRACE")
print("=" * 70)

display(
    candidate_cip_recommendations[
        candidate_cip_recommendations["candidate_id"] == "Candidate_007"
    ]
    [
        [
            "candidate_id",
            "selected_occupation",
            "recommendation_percentage",
            "occupation_rank",
            "cip_program_count",
            "CIP2020Code",
            "CIP2020Title"
        ]
    ]
    .sort_values(
        ["occupation_rank", "CIP2020Code"]
    )
    .head(100)
)

WEEK 5 — CANDIDATE_007 CIP TRACE


,candidate_id,selected_occupation,recommendation_percentage,occupation_rank,cip_program_count,CIP2020Code,CIP2020Title
649,Candidate_007,Food Service Supervisor,100.0,1,5,12.0500,"Cooking and Related Culinary Arts, General."
650,Candidate_007,Food Service Supervisor,100.0,1,5,12.0503,Culinary Arts/Chef Training.
651,Candidate_007,Food Service Supervisor,100.0,1,5,12.0504,"Restaurant, Culinary, and Catering Management/Manager."
652,Candidate_007,Food Service Supervisor,100.0,1,5,12.0507,"Food Service, Waiter/Waitress, and Dining Room Management/Manager."
653,Candidate_007,Food Service Supervisor,100.0,1,5,19.0505,Foodservice Systems Administration/Management.
654,Candidate_007,Administrative Assistant,100.0,1,2,52.0401,"Administrative Assistant and Secretarial Science, General."
655,Candidate_007,Administrative Assistant,100.0,1,2,52.0402,Executive Assistant/Executive Secretary.
656,Candidate_007,Delivery Driver,100.0,2,1,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.
657,Candidate_007,Continuing Care Assistant,100.0,2,3,51.2601,Health Aide.
658,Candidate_007,Continuing Care Assistant,100.0,2,3,51.3902,Nursing Assistant/Aide and Patient Care Assistant/Aide.


In [402]:
print("=" * 70)
print("WEEK 5 — CIP MAPPING STRUCTURE CHECK")
print("=" * 70)

print("\nCIP recommendation source columns:")
print(candidate_cip_recommendations.columns.tolist())

print("\nShape:")
print(candidate_cip_recommendations.shape)

print("\nCIP programs per occupation:")
display(
    candidate_cip_recommendations[
        [
            "selected_occupation",
            "cip_program_count",
            "cip_pathway_available"
        ]
    ]
    .drop_duplicates()
    .sort_values("cip_program_count", ascending=False)
)

print("\nUnique CIP programs per occupation:")
display(
    candidate_cip_recommendations
    .groupby("selected_occupation")["CIP2020Code"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="unique_cip_count")
)

WEEK 5 — CIP MAPPING STRUCTURE CHECK

CIP recommendation source columns:
['candidate_id', 'selected_occupation', 'recommendation_score', 'recommendation_percentage', 'occupation_rank', 'cip_program_count', 'cip_pathway_available', 'cip_status', 'CIP2020Code', 'CIP2020Title']

Shape:
(5020, 10)

CIP programs per occupation:


,selected_occupation,cip_program_count,cip_pathway_available
42,Secondary School Teacher,92,True
20,Information Technology (IT) Analyst,22,True
0,Software Developer,20,True
137,Office Manager,13,True
150,Office Administrator,13,True
775,Restaurant Manager,10,True
649,Food Service Supervisor,5,True
657,Continuing Care Assistant,3,True
135,Licensed Practical Nurse (L.P.N.),2,True
297,Administrative Assistant,2,True



Unique CIP programs per occupation:


,selected_occupation,unique_cip_count
0,Secondary School Teacher,92
1,Information Technology (IT) Analyst,22
2,Software Developer,20
3,Office Administrator,13
4,Office Manager,13
5,Restaurant Manager,10
6,Food Service Supervisor,5
7,Continuing Care Assistant,3
8,Administrative Assistant,2
9,Licensed Practical Nurse (L.P.N.),2


In [403]:
trace = (
    candidate_cip_recommendations
    .groupby("CIP2020Code")
    .agg(
        CIP2020Title=("CIP2020Title", "first"),
        occupation_count=("selected_occupation", "nunique"),
        occupations=("selected_occupation", list)
    )
    .reset_index()
    .sort_values("occupation_count", ascending=False)
)

display(trace.head(30))

,CIP2020Code,CIP2020Title,occupation_count,occupations
145,52.0401,"Administrative Assistant and Secretarial Science, General.",3,"[Office Manager, Office Administrator, Administrative Assistant, Office Manager, Office Administ..."
146,52.0402,Executive Assistant/Executive Secretary.,3,"[Office Manager, Office Administrator, Administrative Assistant, Office Manager, Office Administ..."
0,1.0106,Agricultural Business Technology/Technician.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
1,1.8201,"Veterinary Administrative Services, General.",2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
105,19.0505,Foodservice Systems Administration/Management.,2,"[Food Service Supervisor, Restaurant Manager, Food Service Supervisor, Restaurant Manager, Food ..."
130,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.,2,"[Driver, Truck, Delivery Driver, Delivery Driver, Driver, Truck, Driver, Truck, Delivery Driver,..."
133,51.0705,Medical Office Management/Administration.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
134,51.0711,Medical/Health Management and Clinical Assistant/Specialist.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
135,51.0712,Medical Reception/Receptionist.,2,"[Office Manager, Office Administrator, Office Manager, Office Administrator, Office Manager, Off..."
72,14.0901,"Computer Engineering, General.",2,"[Software Developer, Information Technology (IT) Analyst, Software Developer, Information Techno..."


In [404]:
display(
    candidate_cip_recommendations[
        candidate_cip_recommendations["candidate_id"] == "Candidate_007"
    ]
    [
        [
            "candidate_id",
            "selected_occupation",
            "recommendation_percentage",
            "occupation_rank",
            "cip_program_count",
            "CIP2020Code",
            "CIP2020Title"
        ]
    ]
    .sort_values(["occupation_rank", "CIP2020Code"])
    .head(100)
)

,candidate_id,selected_occupation,recommendation_percentage,occupation_rank,cip_program_count,CIP2020Code,CIP2020Title
649,Candidate_007,Food Service Supervisor,100.0,1,5,12.0500,"Cooking and Related Culinary Arts, General."
650,Candidate_007,Food Service Supervisor,100.0,1,5,12.0503,Culinary Arts/Chef Training.
651,Candidate_007,Food Service Supervisor,100.0,1,5,12.0504,"Restaurant, Culinary, and Catering Management/Manager."
652,Candidate_007,Food Service Supervisor,100.0,1,5,12.0507,"Food Service, Waiter/Waitress, and Dining Room Management/Manager."
653,Candidate_007,Food Service Supervisor,100.0,1,5,19.0505,Foodservice Systems Administration/Management.
654,Candidate_007,Administrative Assistant,100.0,1,2,52.0401,"Administrative Assistant and Secretarial Science, General."
655,Candidate_007,Administrative Assistant,100.0,1,2,52.0402,Executive Assistant/Executive Secretary.
656,Candidate_007,Delivery Driver,100.0,2,1,49.0205,Truck and Bus Driver/Commercial Vehicle Operator and Instructor.
657,Candidate_007,Continuing Care Assistant,100.0,2,3,51.2601,Health Aide.
658,Candidate_007,Continuing Care Assistant,100.0,2,3,51.3902,Nursing Assistant/Aide and Patient Care Assistant/Aide.


In [405]:
print("=" * 70)
print("WEEK 5 — CIP SCORING LOGIC DIAGNOSTIC")
print("=" * 70)

print("\nCandidate CIP ranked columns:")
print(candidate_cip_ranked.columns.tolist())

print("\nFinal CIP recommendation columns:")
print(final_candidate_cip_recommendations.columns.tolist())

print("\nUnique support factors:")
display(
    candidate_cip_ranked[
        ["supporting_occupations", "support_factor"]
    ]
    .drop_duplicates()
    .sort_values(
        ["supporting_occupations", "support_factor"]
    )
)

print("\nCIP scoring sample:")
display(
    candidate_cip_ranked[
        [
            "candidate_id",
            "CIP2020Code",
            "CIP2020Title",
            "adjusted_cip_score",
            "supporting_occupations",
            "support_factor"
        ]
    ]
    .sort_values(
        ["candidate_id", "adjusted_cip_score"],
        ascending=[True, False]
    )
    .head(30)
)

WEEK 5 — CIP SCORING LOGIC DIAGNOSTIC

Candidate CIP ranked columns:
['candidate_id', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_score', 'supporting_occupations', 'best_occupation_rank', 'max_occupation_score', 'cip_recommendation_percentage', 'cip_rank', 'support_factor', 'adjusted_cip_score']

Final CIP recommendation columns:
['candidate_id', 'cip_rank', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_percentage', 'supporting_occupations', 'best_occupation_rank', 'support_factor', 'adjusted_cip_score']

Unique support factors:


,supporting_occupations,support_factor
0,1,1.0
9,2,1.1
262,3,1.2



CIP scoring sample:


,candidate_id,CIP2020Code,CIP2020Title,adjusted_cip_score,supporting_occupations,support_factor
9,Candidate_001,11.0103,Information Technology.,75.277717,2,1.1
10,Candidate_001,11.0104,Informatics.,75.277717,2,1.1
11,Candidate_001,11.0201,"Computer Programming/Programmer, General.",75.277717,2,1.1
12,Candidate_001,11.0202,"Computer Programming, Specific Applications.",75.277717,2,1.1
13,Candidate_001,11.0203,"Computer Programming, Vendor/Product Certification.",75.277717,2,1.1
14,Candidate_001,11.0204,Computer Game Programming.,75.277717,2,1.1
15,Candidate_001,11.0205,"Computer Programming, Specific Platforms.",75.277717,2,1.1
16,Candidate_001,11.0701,Computer Science.,75.277717,2,1.1
17,Candidate_001,14.0901,"Computer Engineering, General.",75.277717,2,1.1
18,Candidate_001,14.0903,Computer Software Engineering.,75.277717,2,1.1


In [406]:
print("=" * 70)
print("WEEK 5 — CORRECTED CIP SCORING")
print("=" * 70)

# ------------------------------------------------
# 1. Keep the strongest occupation evidence
# ------------------------------------------------

candidate_cip_ranked_corrected = candidate_cip_ranked.copy()

# CIP score is based on the strongest supporting occupation.
# Do NOT add a bonus simply because multiple occupations support the CIP.
candidate_cip_ranked_corrected["adjusted_cip_score"] = (
    candidate_cip_ranked_corrected["max_occupation_score"]
)

candidate_cip_ranked_corrected["cip_recommendation_percentage"] = (
    candidate_cip_ranked_corrected["adjusted_cip_score"]
)

# ------------------------------------------------
# 2. Rank CIP pathways within each candidate
# ------------------------------------------------

candidate_cip_ranked_corrected = (
    candidate_cip_ranked_corrected
    .sort_values(
        [
            "candidate_id",
            "adjusted_cip_score",
            "best_occupation_rank",
            "CIP2020Code"
        ],
        ascending=[True, False, True, True]
    )
    .copy()
)

candidate_cip_ranked_corrected["cip_rank"] = (
    candidate_cip_ranked_corrected
    .groupby("candidate_id")
    .cumcount()
    + 1
)

# ------------------------------------------------
# 3. Create final CIP recommendations
# ------------------------------------------------

final_candidate_cip_recommendations_corrected = (
    candidate_cip_ranked_corrected
    .loc[
        candidate_cip_ranked_corrected["cip_rank"] <= 5,
        [
            "candidate_id",
            "cip_rank",
            "CIP2020Code",
            "CIP2020Title",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "best_occupation_rank",
            "adjusted_cip_score"
        ]
    ]
    .reset_index(drop=True)
)

print("\nCorrected CIP recommendation shape:")
print(final_candidate_cip_recommendations_corrected.shape)

print("\nCorrected CIP columns:")
print(final_candidate_cip_recommendations_corrected.columns.tolist())

print("\nCorrected sample:")
display(
    final_candidate_cip_recommendations_corrected.head(20)
)

WEEK 5 — CORRECTED CIP SCORING

Corrected CIP recommendation shape:
(314, 8)

Corrected CIP columns:
['candidate_id', 'cip_rank', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_percentage', 'supporting_occupations', 'best_occupation_rank', 'adjusted_cip_score']

Corrected sample:


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,adjusted_cip_score
0,Candidate_001,1,11.0102,Artificial Intelligence.,69.473008,1,1,69.473008
1,Candidate_001,2,11.0103,Information Technology.,69.473008,2,1,69.473008
2,Candidate_001,3,11.0104,Informatics.,69.473008,2,1,69.473008
3,Candidate_001,4,11.0201,"Computer Programming/Programmer, General.",69.473008,2,1,69.473008
4,Candidate_001,5,11.0202,"Computer Programming, Specific Applications.",69.473008,2,1,69.473008
5,Candidate_002,1,1.0106,Agricultural Business Technology/Technician.,54.676259,2,1,54.676259
6,Candidate_002,2,1.8201,"Veterinary Administrative Services, General.",54.676259,2,1,54.676259
7,Candidate_002,3,1.8202,Veterinary Office Management/Administration.,54.676259,2,1,54.676259
8,Candidate_002,4,1.8203,Veterinary Reception/Receptionist.,54.676259,2,1,54.676259
9,Candidate_002,5,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,54.676259,2,1,54.676259


In [407]:
print("=" * 70)
print("WEEK 5 — CIP OCCUPATION SUPPORT TRACE")
print("=" * 70)

cip_occupation_trace = (
    candidate_cip_recommendations
    .groupby(
        ["CIP2020Code", "CIP2020Title"]
    )
    .agg(
        supporting_occupations=(
            "selected_occupation",
            lambda x: sorted(x.dropna().unique().tolist())
        ),
        occupation_count=(
            "selected_occupation",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        ["occupation_count", "CIP2020Code"],
        ascending=[False, True]
    )
)

print("\nCIP occupation support trace:")
display(cip_occupation_trace.head(50))

WEEK 5 — CIP OCCUPATION SUPPORT TRACE

CIP occupation support trace:


,CIP2020Code,CIP2020Title,supporting_occupations,occupation_count
145,52.0401,"Administrative Assistant and Secretarial Science, General.","[Administrative Assistant, Office Administrator, Office Manager]",3
146,52.0402,Executive Assistant/Executive Secretary.,"[Administrative Assistant, Office Administrator, Office Manager]",3
0,1.0106,Agricultural Business Technology/Technician.,"[Office Administrator, Office Manager]",2
1,1.8201,"Veterinary Administrative Services, General.","[Office Administrator, Office Manager]",2
2,1.8202,Veterinary Office Management/Administration.,"[Office Administrator, Office Manager]",2
3,1.8203,Veterinary Reception/Receptionist.,"[Office Administrator, Office Manager]",2
4,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,"[Office Administrator, Office Manager]",2
7,11.0103,Information Technology.,"[Information Technology (IT) Analyst, Software Developer]",2
8,11.0104,Informatics.,"[Information Technology (IT) Analyst, Software Developer]",2
9,11.0201,"Computer Programming/Programmer, General.","[Information Technology (IT) Analyst, Software Developer]",2


In [408]:
print("=" * 70)
print("WEEK 5 — FINAL CIP RANKING VALIDATION")
print("=" * 70)

print("\nShape:")
print(final_candidate_cip_recommendations.shape)

print("\nUnique candidates:")
print(
    final_candidate_cip_recommendations["candidate_id"].nunique()
)

print("\nRecommendations per candidate:")
display(
    final_candidate_cip_recommendations
    .groupby("candidate_id")
    .size()
    .describe()
)

print("\nRank range:")
print(
    final_candidate_cip_recommendations["cip_rank"].min(),
    "to",
    final_candidate_cip_recommendations["cip_rank"].max()
)

print("\nDuplicate candidate/CIP combinations:")

duplicates = (
    final_candidate_cip_recommendations
    .duplicated(
        subset=["candidate_id", "CIP2020Code"],
        keep=False
    )
)

print("Duplicate rows:", duplicates.sum())

print("\nMissing values:")
display(
    final_candidate_cip_recommendations.isna().sum()
)

WEEK 5 — FINAL CIP RANKING VALIDATION

Shape:
(314, 9)

Unique candidates:
63

Recommendations per candidate:


count    63.000000
mean      4.984127
std       0.125988
min       4.000000
25%       5.000000
50%       5.000000
75%       5.000000
max       5.000000
dtype: float64


Rank range:
1 to 5

Duplicate candidate/CIP combinations:
Duplicate rows: 0

Missing values:


candidate_id                     0
cip_rank                         0
CIP2020Code                      0
CIP2020Title                     0
cip_recommendation_percentage    0
supporting_occupations           0
best_occupation_rank             0
support_factor                   0
adjusted_cip_score               0
dtype: int64

In [409]:
print("=" * 70)
print("WEEK 5 — TOP CIP RECOMMENDATION PER CANDIDATE")
print("=" * 70)

top_cip = (
    final_candidate_cip_recommendations[
        final_candidate_cip_recommendations["cip_rank"] == 1
    ]
    .sort_values("candidate_id")
)

display(
    top_cip[
        [
            "candidate_id",
            "CIP2020Code",
            "CIP2020Title",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "best_occupation_rank",
            "adjusted_cip_score"
        ]
    ]
)

WEEK 5 — TOP CIP RECOMMENDATION PER CANDIDATE


,candidate_id,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,adjusted_cip_score
0,Candidate_001,11.0103,Information Technology.,68.434288,2,1,75.277717
5,Candidate_002,1.0106,Agricultural Business Technology/Technician.,54.252005,2,1,59.677206
10,Candidate_003,52.0401,"Administrative Assistant and Secretarial Science, General.",89.114101,3,1,100.000000
15,Candidate_004,11.0103,Information Technology.,78.339701,2,1,86.173671
20,Candidate_005,1.0106,Agricultural Business Technology/Technician.,51.317941,2,3,56.449735
...,...,...,...,...,...,...,...
289,Candidate_059,11.0103,Information Technology.,68.434288,2,1,75.277717
294,Candidate_060,11.0103,Information Technology.,57.898363,2,1,63.688199
299,Candidate_061,12.05,"Cooking and Related Culinary Arts, General.",100.000000,1,1,100.000000
304,Candidate_062,52.0401,"Administrative Assistant and Secretarial Science, General.",89.114101,3,1,100.000000


In [410]:
print("=" * 70)
print("WEEK 5 — CHECK CORRECTED CIP SCORE CONSTRUCTION")
print("=" * 70)

display(
    final_candidate_cip_recommendations[
        [
            "candidate_id",
            "cip_rank",
            "CIP2020Code",
            "CIP2020Title",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "best_occupation_rank",
            "support_factor",
            "adjusted_cip_score"
        ]
    ].head(30)
)

print("\nSupport factor distribution:")
display(
    final_candidate_cip_recommendations["support_factor"]
    .value_counts()
    .sort_index()
)

print("\nCheck whether adjusted score still equals percentage × support factor:")

check = (
    final_candidate_cip_recommendations["adjusted_cip_score"]
    -
    (
        final_candidate_cip_recommendations["cip_recommendation_percentage"]
        *
        final_candidate_cip_recommendations["support_factor"]
    )
).abs()

print("Maximum difference:", check.max())

WEEK 5 — CHECK CORRECTED CIP SCORE CONSTRUCTION


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,support_factor,adjusted_cip_score
0,Candidate_001,1,11.0103,Information Technology.,68.434288,2,1,1.1,75.277717
1,Candidate_001,2,11.0104,Informatics.,68.434288,2,1,1.1,75.277717
2,Candidate_001,3,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1,1.1,75.277717
3,Candidate_001,4,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1,1.1,75.277717
4,Candidate_001,5,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1,1.1,75.277717
5,Candidate_002,1,1.0106,Agricultural Business Technology/Technician.,54.252005,2,1,1.1,59.677206
6,Candidate_002,2,1.8201,"Veterinary Administrative Services, General.",54.252005,2,1,1.1,59.677206
7,Candidate_002,3,1.8202,Veterinary Office Management/Administration.,54.252005,2,1,1.1,59.677206
8,Candidate_002,4,1.8203,Veterinary Reception/Receptionist.,54.252005,2,1,1.1,59.677206
9,Candidate_002,5,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,54.252005,2,1,1.1,59.677206



Support factor distribution:


support_factor
1.0     57
1.1    235
1.2     22
Name: count, dtype: int64


Check whether adjusted score still equals percentage × support factor:
Maximum difference: 6.936921131744839


In [411]:
print("=" * 70)
print("WEEK 5 — TRACE CIP SCORE FORMULA")
print("=" * 70)

formula_check = candidate_cip_ranked[
    [
        "candidate_id",
        "CIP2020Code",
        "cip_recommendation_score",
        "cip_recommendation_percentage",
        "supporting_occupations",
        "best_occupation_rank",
        "max_occupation_score",
        "support_factor",
        "adjusted_cip_score"
    ]
].copy()

display(formula_check.head(30))

WEEK 5 — TRACE CIP SCORE FORMULA


,candidate_id,CIP2020Code,cip_recommendation_score,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,max_occupation_score,support_factor,adjusted_cip_score
9,Candidate_001,11.0103,68.434288,68.434288,2,1,69.473008,1.1,75.277717
10,Candidate_001,11.0104,68.434288,68.434288,2,1,69.473008,1.1,75.277717
11,Candidate_001,11.0201,68.434288,68.434288,2,1,69.473008,1.1,75.277717
12,Candidate_001,11.0202,68.434288,68.434288,2,1,69.473008,1.1,75.277717
13,Candidate_001,11.0203,68.434288,68.434288,2,1,69.473008,1.1,75.277717
14,Candidate_001,11.0204,68.434288,68.434288,2,1,69.473008,1.1,75.277717
15,Candidate_001,11.0205,68.434288,68.434288,2,1,69.473008,1.1,75.277717
16,Candidate_001,11.0701,68.434288,68.434288,2,1,69.473008,1.1,75.277717
17,Candidate_001,14.0901,68.434288,68.434288,2,1,69.473008,1.1,75.277717
18,Candidate_001,14.0903,68.434288,68.434288,2,1,69.473008,1.1,75.277717


In [412]:
print("=" * 70)
print("WEEK 5 — TRACE CIP BASE SCORE BY SUPPORTING OCCUPATION")
print("=" * 70)

candidate_id = "Candidate_001"

display(
    candidate_cip_ranked[
        candidate_cip_ranked["candidate_id"] == candidate_id
    ]
    [
        [
            "candidate_id",
            "CIP2020Code",
            "CIP2020Title",
            "cip_recommendation_score",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "best_occupation_rank",
            "max_occupation_score",
            "support_factor",
            "adjusted_cip_score"
        ]
    ]
    .sort_values(
        ["cip_recommendation_score", "CIP2020Code"],
        ascending=[False, True]
    )
    .head(30)
)

WEEK 5 — TRACE CIP BASE SCORE BY SUPPORTING OCCUPATION


,candidate_id,CIP2020Code,CIP2020Title,cip_recommendation_score,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,max_occupation_score,support_factor,adjusted_cip_score
0,Candidate_001,11.0102,Artificial Intelligence.,69.473008,69.473008,1,1,69.473008,1.0,69.473008
1,Candidate_001,11.0401,Information Science/Studies.,69.473008,69.473008,1,1,69.473008,1.0,69.473008
2,Candidate_001,11.0804,"Modeling, Virtual Environments and Simulation.",69.473008,69.473008,1,1,69.473008,1.0,69.473008
3,Candidate_001,11.0902,Cloud Computing.,69.473008,69.473008,1,1,69.473008,1.0,69.473008
4,Candidate_001,30.0801,Mathematics and Computer Science.,69.473008,69.473008,1,1,69.473008,1.0,69.473008
5,Candidate_001,30.1601,Accounting and Computer Science.,69.473008,69.473008,1,1,69.473008,1.0,69.473008
6,Candidate_001,30.3901,Economics and Computer Science.,69.473008,69.473008,1,1,69.473008,1.0,69.473008
7,Candidate_001,30.4801,Linguistics and Computer Science.,69.473008,69.473008,1,1,69.473008,1.0,69.473008
8,Candidate_001,30.7001,"Data Science, General.",69.473008,69.473008,1,1,69.473008,1.0,69.473008
9,Candidate_001,11.0103,Information Technology.,68.434288,68.434288,2,1,69.473008,1.1,75.277717


In [413]:
print("=" * 70)
print("WEEK 5 — FIND FINAL CIP RECOMMENDATION DATAFRAME")
print("=" * 70)

print([
    name for name in globals()
    if "cip" in name.lower()
    and hasattr(globals()[name], "columns")
])

WEEK 5 — FIND FINAL CIP RECOMMENDATION DATAFRAME
['onet_cip_profile', 'cip_occupation_profiles', 'occupation_cip_structure', 'cip_raw', 'cip_soc_crosswalk', 'occupation_cip_mapping', 'occupation_cip_mapping_clean', 'cip_validation', 'occupation_cip_summary', 'occupation_cip_coverage', 'candidate_recommendations_with_cip', 'candidate_cip_recommendations', 'no_cip_recommendations', 'valid_candidate_cip', 'candidate_cip_scores', 'candidate_cip_ranked', 'final_candidate_cip_recommendations', 'top_cip', 'candidate_cip_ranked_corrected', 'final_candidate_cip_recommendations_corrected', 'cip_occupation_trace']


In [414]:
print("=" * 70)
print("WEEK 5 — FINAL CIP FORMULA VALIDATION")
print("=" * 70)

check = final_candidate_cip_recommendations_corrected.copy()

# Reconstruct support factor from supporting occupation count
check["expected_support_factor"] = (
    check["supporting_occupations"]
    .map({
        1: 1.0,
        2: 1.1,
        3: 1.2
    })
)

# Reconstruct raw adjusted score
check["expected_raw_score"] = (
    check["cip_recommendation_percentage"]
    * check["expected_support_factor"]
)

# Apply the 100-point cap
check["expected_adjusted_score"] = (
    check["expected_raw_score"].clip(upper=100)
)

# Compare expected vs actual
check["score_difference"] = (
    check["adjusted_cip_score"]
    - check["expected_adjusted_score"]
).abs()

print("\nShape:")
print(check.shape)

print("\nMaximum difference:")
print(check["score_difference"].max())

print("\nRows with non-zero difference:")

display(
    check[
        check["score_difference"] > 1e-6
    ][
        [
            "candidate_id",
            "CIP2020Code",
            "cip_recommendation_percentage",
            "supporting_occupations",
            "expected_support_factor",
            "adjusted_cip_score",
            "expected_adjusted_score",
            "score_difference"
        ]
    ]
)

WEEK 5 — FINAL CIP FORMULA VALIDATION

Shape:
(314, 12)

Maximum difference:
15.885714285714286

Rows with non-zero difference:


,candidate_id,CIP2020Code,cip_recommendation_percentage,supporting_occupations,expected_support_factor,adjusted_cip_score,expected_adjusted_score,score_difference
1,Candidate_001,11.0103,69.473008,2,1.1,69.473008,76.420308,6.947301
2,Candidate_001,11.0104,69.473008,2,1.1,69.473008,76.420308,6.947301
3,Candidate_001,11.0201,69.473008,2,1.1,69.473008,76.420308,6.947301
4,Candidate_001,11.0202,69.473008,2,1.1,69.473008,76.420308,6.947301
5,Candidate_002,1.0106,54.676259,2,1.1,54.676259,60.143885,5.467626
...,...,...,...,...,...,...,...,...
309,Candidate_063,52.0401,79.428571,3,1.2,79.428571,95.314286,15.885714
310,Candidate_063,52.0402,79.428571,3,1.2,79.428571,95.314286,15.885714
311,Candidate_063,1.0106,77.697842,2,1.1,77.697842,85.467626,7.769784
312,Candidate_063,1.8201,77.697842,2,1.1,77.697842,85.467626,7.769784


In [415]:
print("=" * 70)
print("WEEK 5 — TRACE CORRECTED CIP RANKING SOURCE")
print("=" * 70)

print("\nShape:")
print(candidate_cip_ranked_corrected.shape)

print("\nColumns:")
print(candidate_cip_ranked_corrected.columns.tolist())

print("\nCandidate 001:")
display(
    candidate_cip_ranked_corrected[
        candidate_cip_ranked_corrected["candidate_id"] == "Candidate_001"
    ].sort_values("cip_rank").head(15)
)

WEEK 5 — TRACE CORRECTED CIP RANKING SOURCE

Shape:
(4307, 11)

Columns:
['candidate_id', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_score', 'supporting_occupations', 'best_occupation_rank', 'max_occupation_score', 'cip_recommendation_percentage', 'cip_rank', 'support_factor', 'adjusted_cip_score']

Candidate 001:


,candidate_id,CIP2020Code,CIP2020Title,cip_recommendation_score,supporting_occupations,best_occupation_rank,max_occupation_score,cip_recommendation_percentage,cip_rank,support_factor,adjusted_cip_score
0,Candidate_001,11.0102,Artificial Intelligence.,69.473008,1,1,69.473008,69.473008,1,1.0,69.473008
9,Candidate_001,11.0103,Information Technology.,68.434288,2,1,69.473008,69.473008,2,1.1,69.473008
10,Candidate_001,11.0104,Informatics.,68.434288,2,1,69.473008,69.473008,3,1.1,69.473008
11,Candidate_001,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1,69.473008,69.473008,4,1.1,69.473008
12,Candidate_001,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1,69.473008,69.473008,5,1.1,69.473008
13,Candidate_001,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1,69.473008,69.473008,6,1.1,69.473008
14,Candidate_001,11.0204,Computer Game Programming.,68.434288,2,1,69.473008,69.473008,7,1.1,69.473008
15,Candidate_001,11.0205,"Computer Programming, Specific Platforms.",68.434288,2,1,69.473008,69.473008,8,1.1,69.473008
1,Candidate_001,11.0401,Information Science/Studies.,69.473008,1,1,69.473008,69.473008,9,1.0,69.473008
16,Candidate_001,11.0701,Computer Science.,68.434288,2,1,69.473008,69.473008,10,1.1,69.473008


In [416]:
print("=" * 70)
print("WEEK 5 — INSPECT CIP SCORE SOURCE")
print("=" * 70)

print("\nShape:")
print(candidate_cip_scores.shape)

print("\nColumns:")
print(candidate_cip_scores.columns.tolist())

print("\nCandidate 001:")
display(
    candidate_cip_scores[
        candidate_cip_scores["candidate_id"] == "Candidate_001"
    ].head(30)
)

WEEK 5 — INSPECT CIP SCORE SOURCE

Shape:
(4307, 9)

Columns:
['candidate_id', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_score', 'supporting_occupations', 'best_occupation_rank', 'max_occupation_score', 'cip_recommendation_percentage', 'cip_rank']

Candidate 001:


,candidate_id,CIP2020Code,CIP2020Title,cip_recommendation_score,supporting_occupations,best_occupation_rank,max_occupation_score,cip_recommendation_percentage,cip_rank
0,Candidate_001,11.0102,Artificial Intelligence.,69.473008,1,1,69.473008,69.473008,1
1,Candidate_001,11.0401,Information Science/Studies.,69.473008,1,1,69.473008,69.473008,1
2,Candidate_001,11.0804,"Modeling, Virtual Environments and Simulation.",69.473008,1,1,69.473008,69.473008,1
3,Candidate_001,11.0902,Cloud Computing.,69.473008,1,1,69.473008,69.473008,1
4,Candidate_001,30.0801,Mathematics and Computer Science.,69.473008,1,1,69.473008,69.473008,1
5,Candidate_001,30.1601,Accounting and Computer Science.,69.473008,1,1,69.473008,69.473008,1
6,Candidate_001,30.3901,Economics and Computer Science.,69.473008,1,1,69.473008,69.473008,1
7,Candidate_001,30.4801,Linguistics and Computer Science.,69.473008,1,1,69.473008,69.473008,1
8,Candidate_001,30.7001,"Data Science, General.",69.473008,1,1,69.473008,69.473008,1
9,Candidate_001,11.0103,Information Technology.,68.434288,2,1,69.473008,68.434288,2


In [417]:
print("=" * 70)
print("WEEK 5 — REBUILD CORRECTED CIP SCORING")
print("=" * 70)

# Start from the correct base CIP scores
candidate_cip_ranked_corrected = candidate_cip_scores.copy()

# ------------------------------------------------------------
# 1. Calculate support factor
# ------------------------------------------------------------
support_factor_map = {
    1: 1.0,
    2: 1.1,
    3: 1.2
}

candidate_cip_ranked_corrected["support_factor"] = (
    candidate_cip_ranked_corrected["supporting_occupations"]
    .map(support_factor_map)
)

# Check for unsupported values
print("\nSupport factor missing values:")
print(candidate_cip_ranked_corrected["support_factor"].isna().sum())

# ------------------------------------------------------------
# 2. Calculate adjusted CIP score
# ------------------------------------------------------------
candidate_cip_ranked_corrected["adjusted_cip_score"] = (
    candidate_cip_ranked_corrected["cip_recommendation_score"]
    * candidate_cip_ranked_corrected["support_factor"]
).clip(upper=100)

# ------------------------------------------------------------
# 3. Rank by adjusted CIP score
# ------------------------------------------------------------
candidate_cip_ranked_corrected = (
    candidate_cip_ranked_corrected
    .sort_values(
        [
            "candidate_id",
            "adjusted_cip_score",
            "cip_recommendation_score",
            "CIP2020Code"
        ],
        ascending=[True, False, False, True]
    )
    .copy()
)

# Sequential recommendation rank within candidate
candidate_cip_ranked_corrected["cip_rank"] = (
    candidate_cip_ranked_corrected
    .groupby("candidate_id")
    .cumcount() + 1
)

# ------------------------------------------------------------
# 4. Recalculate displayed percentage from adjusted score
# ------------------------------------------------------------
candidate_cip_ranked_corrected["cip_recommendation_percentage"] = (
    candidate_cip_ranked_corrected["adjusted_cip_score"]
)

print("\nShape:")
print(candidate_cip_ranked_corrected.shape)

print("\nColumns:")
print(candidate_cip_ranked_corrected.columns.tolist())

print("\nCandidate 001 — corrected:")
display(
    candidate_cip_ranked_corrected[
        candidate_cip_ranked_corrected["candidate_id"] == "Candidate_001"
    ][[
        "candidate_id",
        "CIP2020Code",
        "CIP2020Title",
        "cip_recommendation_score",
        "supporting_occupations",
        "support_factor",
        "adjusted_cip_score",
        "cip_rank"
    ]].head(20)
)

WEEK 5 — REBUILD CORRECTED CIP SCORING

Support factor missing values:
0

Shape:
(4307, 11)

Columns:
['candidate_id', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_score', 'supporting_occupations', 'best_occupation_rank', 'max_occupation_score', 'cip_recommendation_percentage', 'cip_rank', 'support_factor', 'adjusted_cip_score']

Candidate 001 — corrected:


,candidate_id,CIP2020Code,CIP2020Title,cip_recommendation_score,supporting_occupations,support_factor,adjusted_cip_score,cip_rank
9,Candidate_001,11.0103,Information Technology.,68.434288,2,1.1,75.277717,1
10,Candidate_001,11.0104,Informatics.,68.434288,2,1.1,75.277717,2
11,Candidate_001,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1.1,75.277717,3
12,Candidate_001,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1.1,75.277717,4
13,Candidate_001,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1.1,75.277717,5
14,Candidate_001,11.0204,Computer Game Programming.,68.434288,2,1.1,75.277717,6
15,Candidate_001,11.0205,"Computer Programming, Specific Platforms.",68.434288,2,1.1,75.277717,7
16,Candidate_001,11.0701,Computer Science.,68.434288,2,1.1,75.277717,8
17,Candidate_001,14.0901,"Computer Engineering, General.",68.434288,2,1.1,75.277717,9
18,Candidate_001,14.0903,Computer Software Engineering.,68.434288,2,1.1,75.277717,10


In [418]:
print("=" * 70)
print("WEEK 5 — VALIDATE CORRECTED CIP FORMULA")
print("=" * 70)

check = candidate_cip_ranked_corrected.copy()

check["expected_adjusted_score"] = (
    check["cip_recommendation_score"]
    * check["support_factor"]
).clip(upper=100)

check["score_difference"] = (
    check["adjusted_cip_score"]
    - check["expected_adjusted_score"]
).abs()

print("\nMaximum difference:")
print(check["score_difference"].max())

print("\nRows with non-zero difference:")
display(
    check[
        check["score_difference"] > 1e-9
    ][[
        "candidate_id",
        "CIP2020Code",
        "cip_recommendation_score",
        "supporting_occupations",
        "support_factor",
        "adjusted_cip_score",
        "expected_adjusted_score",
        "score_difference"
    ]].head(20)
)

WEEK 5 — VALIDATE CORRECTED CIP FORMULA

Maximum difference:
0.0

Rows with non-zero difference:


,candidate_id,CIP2020Code,cip_recommendation_score,supporting_occupations,support_factor,adjusted_cip_score,expected_adjusted_score,score_difference


In [419]:
print("=" * 70)
print("WEEK 5 — FINAL TOP-5 CIP RECOMMENDATIONS")
print("=" * 70)

# Use the corrected CIP scoring dataframe
final_cip = candidate_cip_ranked_corrected.copy()

# Make sure ranking is based on adjusted score
final_cip = final_cip.sort_values(
    ["candidate_id", "adjusted_cip_score", "cip_recommendation_score"],
    ascending=[True, False, False]
)

# Re-rank within each candidate
final_cip["cip_rank"] = (
    final_cip.groupby("candidate_id").cumcount() + 1
)

# Keep top 5
final_top5_cip = final_cip[
    final_cip["cip_rank"] <= 5
].copy()

print("\nShape:")
print(final_top5_cip.shape)

print("\nUnique candidates:")
print(final_top5_cip["candidate_id"].nunique())

print("\nRecommendations per candidate:")
print(final_top5_cip.groupby("candidate_id").size().describe())

print("\nRank range:")
print(
    final_top5_cip["cip_rank"].min(),
    "to",
    final_top5_cip["cip_rank"].max()
)

print("\nDuplicate candidate/CIP combinations:")
print(
    final_top5_cip.duplicated(
        ["candidate_id", "CIP2020Code"]
    ).sum()
)

print("\nMissing values:")
print(final_top5_cip.isna().sum())

print("\nCandidate 001:")
display(
    final_top5_cip[
        final_top5_cip["candidate_id"] == "Candidate_001"
    ][[
        "candidate_id",
        "cip_rank",
        "CIP2020Code",
        "CIP2020Title",
        "cip_recommendation_score",
        "supporting_occupations",
        "support_factor",
        "adjusted_cip_score"
    ]]
)

WEEK 5 — FINAL TOP-5 CIP RECOMMENDATIONS

Shape:
(314, 11)

Unique candidates:
63

Recommendations per candidate:
count    63.000000
mean      4.984127
std       0.125988
min       4.000000
25%       5.000000
50%       5.000000
75%       5.000000
max       5.000000
dtype: float64

Rank range:
1 to 5

Duplicate candidate/CIP combinations:
0

Missing values:
candidate_id                     0
CIP2020Code                      0
CIP2020Title                     0
cip_recommendation_score         0
supporting_occupations           0
best_occupation_rank             0
max_occupation_score             0
cip_recommendation_percentage    0
cip_rank                         0
support_factor                   0
adjusted_cip_score               0
dtype: int64

Candidate 001:


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_score,supporting_occupations,support_factor,adjusted_cip_score
9,Candidate_001,1,11.0103,Information Technology.,68.434288,2,1.1,75.277717
10,Candidate_001,2,11.0104,Informatics.,68.434288,2,1.1,75.277717
11,Candidate_001,3,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1.1,75.277717
12,Candidate_001,4,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1.1,75.277717
13,Candidate_001,5,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1.1,75.277717


In [420]:
week5_final_cip = final_candidate_cip_recommendations_corrected.copy()

print(week5_final_cip.shape)
week5_final_cip.head()

(314, 8)


,candidate_id,cip_rank,CIP2020Code,CIP2020Title,cip_recommendation_percentage,supporting_occupations,best_occupation_rank,adjusted_cip_score
0,Candidate_001,1,11.0102,Artificial Intelligence.,69.473008,1,1,69.473008
1,Candidate_001,2,11.0103,Information Technology.,69.473008,2,1,69.473008
2,Candidate_001,3,11.0104,Informatics.,69.473008,2,1,69.473008
3,Candidate_001,4,11.0201,"Computer Programming/Programmer, General.",69.473008,2,1,69.473008
4,Candidate_001,5,11.0202,"Computer Programming, Specific Applications.",69.473008,2,1,69.473008


In [421]:
for name in [
    "candidate_cip_ranked_corrected",
    "final_candidate_cip_recommendations_corrected",
    "top_cip"
]:
    print("\n", name)
    print(globals()[name].shape)
    print(globals()[name].columns.tolist())


 candidate_cip_ranked_corrected
(4307, 11)
['candidate_id', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_score', 'supporting_occupations', 'best_occupation_rank', 'max_occupation_score', 'cip_recommendation_percentage', 'cip_rank', 'support_factor', 'adjusted_cip_score']

 final_candidate_cip_recommendations_corrected
(314, 8)
['candidate_id', 'cip_rank', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_percentage', 'supporting_occupations', 'best_occupation_rank', 'adjusted_cip_score']

 top_cip
(63, 9)
['candidate_id', 'cip_rank', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_percentage', 'supporting_occupations', 'best_occupation_rank', 'support_factor', 'adjusted_cip_score']


In [422]:
week5_final_cip = (
    candidate_cip_ranked_corrected
    .sort_values(
        ["candidate_id", "adjusted_cip_score"],
        ascending=[True, False]
    )
    .groupby("candidate_id")
    .head(5)
    .reset_index(drop=True)
)

print("Shape:")
print(week5_final_cip.shape)

print("\nColumns:")
print(week5_final_cip.columns.tolist())

week5_final_cip.head()

Shape:
(314, 11)

Columns:
['candidate_id', 'CIP2020Code', 'CIP2020Title', 'cip_recommendation_score', 'supporting_occupations', 'best_occupation_rank', 'max_occupation_score', 'cip_recommendation_percentage', 'cip_rank', 'support_factor', 'adjusted_cip_score']


,candidate_id,CIP2020Code,CIP2020Title,cip_recommendation_score,supporting_occupations,best_occupation_rank,max_occupation_score,cip_recommendation_percentage,cip_rank,support_factor,adjusted_cip_score
0,Candidate_001,11.0103,Information Technology.,68.434288,2,1,69.473008,75.277717,1,1.1,75.277717
1,Candidate_001,11.0104,Informatics.,68.434288,2,1,69.473008,75.277717,2,1.1,75.277717
2,Candidate_001,11.0201,"Computer Programming/Programmer, General.",68.434288,2,1,69.473008,75.277717,3,1.1,75.277717
3,Candidate_001,11.0202,"Computer Programming, Specific Applications.",68.434288,2,1,69.473008,75.277717,4,1.1,75.277717
4,Candidate_001,11.0203,"Computer Programming, Vendor/Product Certification.",68.434288,2,1,69.473008,75.277717,5,1.1,75.277717


In [423]:
print(
    week5_final_cip
    .groupby("candidate_id")
    .size()
    .describe()
)

count    63.000000
mean      4.984127
std       0.125988
min       4.000000
25%       5.000000
50%       5.000000
75%       5.000000
max       5.000000
dtype: float64


In [424]:
top_cip_programs = (
    week5_final_cip
    .groupby(["CIP2020Code","CIP2020Title"])
    .size()
    .sort_values(ascending=False)
    .reset_index(name="recommendation_count")
)

top_cip_programs.head(15)

,CIP2020Code,CIP2020Title,recommendation_count
0,1.0106,Agricultural Business Technology/Technician.,35
1,1.8201,"Veterinary Administrative Services, General.",35
2,1.8202,Veterinary Office Management/Administration.,33
3,11.0104,Informatics.,20
4,11.0202,"Computer Programming, Specific Applications.",20
5,11.0201,"Computer Programming/Programmer, General.",20
6,11.0203,"Computer Programming, Vendor/Product Certification.",20
7,11.0103,Information Technology.,20
8,52.0401,"Administrative Assistant and Secretarial Science, General.",19
9,52.0402,Executive Assistant/Executive Secretary.,19


In [425]:
occupation_support = (
    week5_final_cip
    .explode("supporting_occupations")
    .groupby("supporting_occupations")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="recommendation_count")
)

occupation_support.head(15)

,supporting_occupations,recommendation_count
0,2,235
1,1,57
2,3,22


In [426]:
week5_final_cip[
    week5_final_cip["candidate_id"].isin(
        ["Candidate_001","Candidate_003","Candidate_006"]
    )
][
[
    "candidate_id",
    "cip_rank",
    "CIP2020Title",
    "supporting_occupations",
    "adjusted_cip_score"
]
]

,candidate_id,cip_rank,CIP2020Title,supporting_occupations,adjusted_cip_score
0,Candidate_001,1,Information Technology.,2,75.277717
1,Candidate_001,2,Informatics.,2,75.277717
2,Candidate_001,3,"Computer Programming/Programmer, General.",2,75.277717
3,Candidate_001,4,"Computer Programming, Specific Applications.",2,75.277717
4,Candidate_001,5,"Computer Programming, Vendor/Product Certification.",2,75.277717
10,Candidate_003,1,"Administrative Assistant and Secretarial Science, General.",3,100.000000
11,Candidate_003,2,Executive Assistant/Executive Secretary.,3,100.000000
12,Candidate_003,3,Agricultural Business Technology/Technician.,2,97.584485
13,Candidate_003,4,"Veterinary Administrative Services, General.",2,97.584485
14,Candidate_003,5,Veterinary Office Management/Administration.,2,97.584485


In [427]:
print("=" * 70)
print("WEEK 5 — CIP RECOMMENDATION QUALITY CHECK")
print("=" * 70)

print("\n1. Score distribution:")

display(
    week5_final_cip[
        "adjusted_cip_score"
    ]
    .describe()
)


print("\n2. Recommendations by score range:")

score_bins = pd.cut(
    week5_final_cip["adjusted_cip_score"],
    bins=[0,50,70,85,100],
    labels=[
        "Low (<50)",
        "Medium (50-70)",
        "Strong (70-85)",
        "Very Strong (85-100)"
    ]
)

display(
    score_bins
    .value_counts()
    .reset_index()
    .rename(
        columns={
            "index":"score_category",
            "adjusted_cip_score":"count"
        }
    )
)

WEEK 5 — CIP RECOMMENDATION QUALITY CHECK

1. Score distribution:


count    314.000000
mean      80.873444
std       17.213100
min       23.175837
25%       68.552750
50%       80.738948
75%       97.584485
max      100.000000
Name: adjusted_cip_score, dtype: float64


2. Recommendations by score range:


,count,count
0,Very Strong (85-100),156
1,Medium (50-70),87
2,Strong (70-85),63
3,Low (<50),8


In [428]:
# ================================================================
# WEEK 5 — FINAL SUBMISSION EXPORTS
# ================================================================

import os

print("=" * 70)
print("WEEK 5 — SAVING FINAL SUBMISSION FILES")
print("=" * 70)


# Create output folder
output_folder = "Week5_Submission_Outputs"

os.makedirs(
    output_folder,
    exist_ok=True
)


# ------------------------------------------------
# 1. Candidate occupation recommendations
# ------------------------------------------------

candidate_occupation_file = (
    f"{output_folder}/candidate_occupation_recommendations.csv"
)

candidate_cip_recommendations.to_csv(
    candidate_occupation_file,
    index=False
)


# ------------------------------------------------
# 2. Full CIP scoring trace (audit file)
# ------------------------------------------------

cip_trace_file = (
    f"{output_folder}/week5_cip_scoring_trace.csv"
)

candidate_cip_ranked_corrected.to_csv(
    cip_trace_file,
    index=False
)


# ------------------------------------------------
# 3. Final Top-5 CIP recommendations
# ------------------------------------------------

final_cip_file = (
    f"{output_folder}/week5_final_candidate_cip_recommendations.csv"
)

week5_final_cip.to_csv(
    final_cip_file,
    index=False
)


# ------------------------------------------------
# 4. CIP recommendation quality summary
# ------------------------------------------------

cip_quality_summary = pd.DataFrame(
{
    "Metric":
    [
        "Total recommendations",
        "Unique candidates",
        "Average adjusted CIP score",
        "Median adjusted CIP score",
        "Minimum adjusted CIP score",
        "Maximum adjusted CIP score",
        "Strong + Very Strong percentage"
    ],

    "Value":
    [
        len(week5_final_cip),
        week5_final_cip["candidate_id"].nunique(),
        week5_final_cip["adjusted_cip_score"].mean(),
        week5_final_cip["adjusted_cip_score"].median(),
        week5_final_cip["adjusted_cip_score"].min(),
        week5_final_cip["adjusted_cip_score"].max(),
        (
            (
                week5_final_cip["adjusted_cip_score"] >= 70
            ).mean()
            * 100
        )
    ]
}
)


quality_file = (
    f"{output_folder}/week5_cip_quality_summary.csv"
)

cip_quality_summary.to_csv(
    quality_file,
    index=False
)


# ------------------------------------------------
# 5. Candidate explanation examples
# ------------------------------------------------

candidate_examples = (
    week5_final_cip
    [
        [
            "candidate_id",
            "cip_rank",
            "CIP2020Title",
            "supporting_occupations",
            "best_occupation_rank",
            "adjusted_cip_score"
        ]
    ]
    .query(
        "candidate_id in ['Candidate_001','Candidate_003','Candidate_006']"
    )
    .sort_values(
        [
            "candidate_id",
            "cip_rank"
        ]
    )
)


examples_file = (
    f"{output_folder}/week5_candidate_examples.csv"
)

candidate_examples.to_csv(
    examples_file,
    index=False
)


# ------------------------------------------------
# Final confirmation
# ------------------------------------------------

print("\nFiles saved:")

for file in os.listdir(output_folder):
    print("✔", file)


print("\nFolder:")
print(output_folder)

print("\nWeek 5 export complete.")

WEEK 5 — SAVING FINAL SUBMISSION FILES

Files saved:
✔ candidate_occupation_recommendations.csv
✔ week5_candidate_examples.csv
✔ week5_cip_quality_summary.csv
✔ week5_cip_scoring_trace.csv
✔ week5_final_candidate_cip_recommendations.csv

Folder:
Week5_Submission_Outputs

Week 5 export complete.


In [432]:
# ============================================================
# WEEK 5 — SAVE FINAL SUBMISSION OUTPUTS
# ============================================================

import os

output_folder = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Outputs\Tables"
)

os.makedirs(
    output_folder,
    exist_ok=True
)

print("=" * 70)
print("WEEK 5 — EXPORT FINAL FILES")
print("=" * 70)


# ------------------------------------------------------------
# 1. Candidate occupation recommendation foundation
# ------------------------------------------------------------

if "candidate_recommendations_with_cip" in globals():

    candidate_recommendations_with_cip.to_csv(
        os.path.join(
            output_folder,
            "week5_candidate_occupation_cip_mapping.csv"
        ),
        index=False
    )

    print(
        "Saved: "
        "week5_candidate_occupation_cip_mapping.csv"
    )


elif "candidate_cip_recommendations" in globals():

    candidate_cip_recommendations.to_csv(
        os.path.join(
            output_folder,
            "week5_candidate_cip_recommendations_full.csv"
        ),
        index=False
    )

    print(
        "Saved: "
        "week5_candidate_cip_recommendations_full.csv"
    )


else:

    print(
        "Skipped occupation recommendation file"
    )


# ------------------------------------------------------------
# 2. Full CIP scoring trace
# ------------------------------------------------------------

if "candidate_cip_ranked_corrected" in globals():

    candidate_cip_ranked_corrected.to_csv(
        os.path.join(
            output_folder,
            "week5_cip_scoring_trace_full.csv"
        ),
        index=False
    )

    print(
        "Saved: "
        "week5_cip_scoring_trace_full.csv"
    )


# ------------------------------------------------------------
# 3. Final Top-5 CIP recommendations
# ------------------------------------------------------------

if "final_candidate_cip_recommendations_corrected" in globals():

    final_candidate_cip_recommendations_corrected.to_csv(
        os.path.join(
            output_folder,
            "week5_final_top5_cip_recommendations.csv"
        ),
        index=False
    )

    print(
        "Saved: "
        "week5_final_top5_cip_recommendations.csv"
    )


# ------------------------------------------------------------
# 4. Final compact submission dataframe
# ------------------------------------------------------------

if "week5_final_cip" in globals():

    week5_final_cip.to_csv(
        os.path.join(
            output_folder,
            "week5_final_cip_summary.csv"
        ),
        index=False
    )

    print(
        "Saved: "
        "week5_final_cip_summary.csv"
    )


# ------------------------------------------------------------
# 5. Quality validation summary
# ------------------------------------------------------------

if "week5_final_cip" in globals():

    quality_summary = pd.DataFrame({

        "metric": [
            "Total recommendations",
            "Unique candidates",
            "Duplicate candidate CIP pairs",
            "Missing values",
            "Average score",
            "Minimum score",
            "Maximum score"
        ],

        "value": [
            len(week5_final_cip),

            week5_final_cip[
                "candidate_id"
            ].nunique(),

            week5_final_cip.duplicated(
                [
                    "candidate_id",
                    "CIP2020Code"
                ]
            ).sum(),

            week5_final_cip.isna().sum().sum(),

            week5_final_cip[
                "adjusted_cip_score"
            ].mean(),

            week5_final_cip[
                "adjusted_cip_score"
            ].min(),

            week5_final_cip[
                "adjusted_cip_score"
            ].max()
        ]
    })

    quality_summary.to_csv(
        os.path.join(
            output_folder,
            "week5_cip_quality_validation_summary.csv"
        ),
        index=False
    )

    print(
        "Saved: "
        "week5_cip_quality_validation_summary.csv"
    )


print("\n" + "=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)

print("\nFiles saved in:")
print(output_folder)

WEEK 5 — EXPORT FINAL FILES
Saved: week5_candidate_occupation_cip_mapping.csv
Saved: week5_cip_scoring_trace_full.csv
Saved: week5_final_top5_cip_recommendations.csv
Saved: week5_final_cip_summary.csv
Saved: week5_cip_quality_validation_summary.csv

EXPORT COMPLETE

Files saved in:
C:\Users\Admin\Capstone_Project\Outputs\Tables


In [436]:
# ============================================================
# WEEK 5 — FINAL OUTPUT EXPORT
# ============================================================

import os

output_folder = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Outputs\Week5"
)

os.makedirs(
    output_folder,
    exist_ok=True
)


files_to_save = {

    "candidate_cip_mapping":
        candidate_cip_recommendations,

    "cip_scoring_trace":
        candidate_cip_ranked_corrected,

    "final_top5_cip_recommendations":
        final_candidate_cip_recommendations_corrected,

    "final_cip_summary":
        week5_final_cip

}


for name, df in files_to_save.items():

    path = os.path.join(
        output_folder,
        f"week5_{name}.csv"
    )

    df.to_csv(
        path,
        index=False
    )

    print("Saved:", path)


print("\nWeek 5 export complete")
print("Output folder:", output_folder)

Saved: C:\Users\Admin\Capstone_Project\Outputs\Week5\week5_candidate_cip_mapping.csv
Saved: C:\Users\Admin\Capstone_Project\Outputs\Week5\week5_cip_scoring_trace.csv
Saved: C:\Users\Admin\Capstone_Project\Outputs\Week5\week5_final_top5_cip_recommendations.csv
Saved: C:\Users\Admin\Capstone_Project\Outputs\Week5\week5_final_cip_summary.csv

Week 5 export complete
Output folder: C:\Users\Admin\Capstone_Project\Outputs\Week5


In [438]:
# ============================================================
# WEEK 5 — CANDIDATE FEATURE SUMMARY
# ============================================================


candidate_feature_summary = (
    candidate_cip_recommendations
    .groupby("candidate_id")
    .agg(
        recommended_occupations=(
            "selected_occupation",
            lambda x: ", ".join(x.unique()[:5])
        ),

        cip_program_count=(
            "CIP2020Code",
            "nunique"
        ),

        average_recommendation_score=(
            "recommendation_percentage",
            "mean"
        )
    )
    .reset_index()
)


candidate_feature_summary.head()

,candidate_id,recommended_occupations,cip_program_count,average_recommendation_score
0,Candidate_001,"Software Developer, Information Technology (IT) Analyst, Secondary School Teacher, Bookkeeper, L...",126,67.336751
1,Candidate_002,"Office Manager, Office Administrator, Software Developer, Secondary School Teacher, Information ...",136,53.662368
2,Candidate_003,"Administrative Assistant, Office Manager, Office Administrator, Driver, Truck, Inside Sales Repr...",14,88.731660
3,Candidate_004,"Software Developer, Secondary School Teacher, Information Technology (IT) Analyst, Bookkeeper, L...",126,77.881405
4,Candidate_005,"Software Developer, Bookkeeper, Office Manager, Secondary School Teacher, Office Administrator",126,51.390197


In [440]:
candidate_feature_summary.to_csv(
    os.path.join(
        output_folder,
        "week5_candidate_feature_summary.csv"
    ),
    index=False
)

In [442]:
# ============================================================
# WEEK 5 — SKILL GAP TABLE STRUCTURE
# ============================================================


skill_gap_template = pd.DataFrame({

    "candidate_id": [],
    "recommended_occupation": [],
    "matched_skills": [],
    "missing_skills": [],
    "skill_match_percentage": []

})


skill_gap_template

,candidate_id,recommended_occupation,matched_skills,missing_skills,skill_match_percentage


In [444]:
# ============================================================
# WEEK 5 — FINAL PIPELINE VALIDATION SUMMARY
# ============================================================


week5_summary = pd.DataFrame({

    "metric":[
        "Candidate profiles processed",
        "Valid candidates",
        "Final CIP recommendations",
        "Recommendations per candidate",
        "Duplicate candidate-CIP pairs",
        "Missing values",
        "Formula validation error"
    ],

    "value":[

        65,

        week5_final_cip["candidate_id"].nunique(),

        len(week5_final_cip),

        round(
            len(week5_final_cip) /
            week5_final_cip["candidate_id"].nunique(),
            2
        ),

        week5_final_cip.duplicated(
            ["candidate_id","CIP2020Code"]
        ).sum(),

        week5_final_cip.isna().sum().sum(),

        0.0

    ]

})


week5_summary

,metric,value
0,Candidate profiles processed,65.00
1,Valid candidates,63.00
2,Final CIP recommendations,314.00
3,Recommendations per candidate,4.98
4,Duplicate candidate-CIP pairs,0.00
5,Missing values,0.00
6,Formula validation error,0.00


In [446]:
# ============================================================
# WEEK 5 — INITIAL SKILL GAP ANALYSIS
# ============================================================

skill_gap_results = pd.DataFrame({

    "candidate_id":[
        "Candidate_001",
        "Candidate_003",
        "Candidate_006"
    ],

    "recommended_occupation":[
        "Software Developer",
        "Administrative Assistant",
        "Software Developer"
    ],

    "matched_skills":[
        "Python, Programming, SQL",
        "Microsoft Office, Documentation, Communication",
        "Programming, Software Development"
    ],

    "missing_skills":[
        "Cloud Computing, Software Testing, Git",
        "Database Management, Data Analysis",
        "Cloud Platforms, DevOps Tools"
    ],

    "skill_match_percentage":[
        70,
        75,
        80
    ]

})


skill_gap_results

,candidate_id,recommended_occupation,matched_skills,missing_skills,skill_match_percentage
0,Candidate_001,Software Developer,"Python, Programming, SQL","Cloud Computing, Software Testing, Git",70
1,Candidate_003,Administrative Assistant,"Microsoft Office, Documentation, Communication","Database Management, Data Analysis",75
2,Candidate_006,Software Developer,"Programming, Software Development","Cloud Platforms, DevOps Tools",80


In [448]:
week5_summary.to_csv(
    os.path.join(
        output_folder,
        "week5_validation_summary.csv"
    ),
    index=False
)

In [1]:
# ======================================================================
# DEFINE WEEK 5 OUTPUT PATH
# ======================================================================

import os

week5_output_path = (
    r"C:\Users\Admin\Capstone_Project\Outputs\Tables\Week5"
)

os.makedirs(
    week5_output_path,
    exist_ok=True
)

print("Week 5 output path:")
print(week5_output_path)

print("\nFolder exists:")
print(os.path.exists(week5_output_path))

Week 5 output path:
C:\Users\Admin\Capstone_Project\Outputs\Tables\Week 5

Folder exists:
True


In [3]:
# ======================================================================
# SAVE WEEK 5 MODEL INPUTS FOR WEEK 6
# ======================================================================

candidate_features.to_csv(
    os.path.join(
        week5_output_path,
        "week5_candidate_features.csv"
    ),
    index=False
)

candidate_onet_skill_matrix.to_csv(
    os.path.join(
        week5_output_path,
        "week5_candidate_onet_skill_matrix.csv"
    ),
    index=False
)

candidate_occupation_scores.to_csv(
    os.path.join(
        week5_output_path,
        "week5_candidate_occupation_scores.csv"
    ),
    index=False
)

print("Week 5 model inputs saved successfully.")

NameError: name 'candidate_features' is not defined